In [1]:
%pip install requests beautifulsoup4 lxml pandas numpy tqdm sentence-transformers faiss-cpu

Note: you may need to restart the kernel to use updated packages.


In [2]:
import os
import json
import requests
import numpy as np
import pandas as pd

from pathlib import Path
from bs4 import BeautifulSoup
from tqdm.auto import tqdm

from sentence_transformers import SentenceTransformer
import faiss

print("RAG environment ready!")

c:\Users\yuvan\Desktop\ML_projects\Chest_Xray_AI\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


RAG environment ready!


In [3]:
import requests
import bs4
import lxml
import pandas
import numpy
import tqdm
import sentence_transformers
import faiss

print("requests:", requests.__version__)
print("beautifulsoup4:", bs4.__version__)
print("lxml:", lxml.__version__)
print("pandas:", pandas.__version__)
print("numpy:", numpy.__version__)
print("sentence-transformers:", sentence_transformers.__version__)
print("FAISS: imported successfully")

print("\n✅ All RAG dependencies are ready!")

requests: 2.34.2
beautifulsoup4: 4.15.0
lxml: 6.1.3
pandas: 3.0.6
numpy: 2.4.6
sentence-transformers: 6.1.0
FAISS: imported successfully

✅ All RAG dependencies are ready!


In [4]:
import json
import os

rag_sources = [
    {
        "id": "radiologyinfo_chest_xray_report",
        "title": "How to Read Your Chest X-ray Report",
        "organization": "RadiologyInfo",
        "url": "https://www.radiologyinfo.org/en/info/article-chest-xray-report",
        "type": "medical_reference"
    },
    {
        "id": "chestxray14_research",
        "title": "ChestX-ray14 Dataset and Multi-Label Chest X-ray Classification",
        "organization": "NIH / Research Literature",
        "url": "https://pmc.ncbi.nlm.nih.gov/articles/PMC9131331/",
        "type": "research"
    }
]

os.makedirs("rag/metadata", exist_ok=True)

with open(
    "rag/metadata/sources.json",
    "w",
    encoding="utf-8"
) as f:
    json.dump(rag_sources, f, indent=4)

print("RAG source registry created!")
print("Sources:", len(rag_sources))

for source in rag_sources:
    print(f"✓ {source['title']}")

RAG source registry created!
Sources: 2
✓ How to Read Your Chest X-ray Report
✓ ChestX-ray14 Dataset and Multi-Label Chest X-ray Classification


In [5]:
import requests

pmc_id = "PMC9131331"

url = (
    "https://eutils.ncbi.nlm.nih.gov/entrez/eutils/"
    f"efetch.fcgi?db=pmc&id={pmc_id}&retmode=xml"
)

response = requests.get(url, timeout=60)

print("HTTP status:", response.status_code)
print("Response size:", len(response.text), "characters")
print("Starts with:", response.text[:100])


HTTP status: 200
Response size: 11528 characters
Starts with: <?xml version="1.0"  ?><!DOCTYPE pmc-articleset PUBLIC "-//NLM//DTD ARTICLE SET 2.0//EN" "https://dt


In [6]:
from bs4 import BeautifulSoup

soup = BeautifulSoup(response.text, "xml")

# Extract the article text
text_parts = []

for element in soup.find_all(["article-title", "abstract", "p"]):
    text = element.get_text(" ", strip=True)
    if text:
        text_parts.append(text)

article_text = "\n\n".join(text_parts)

print("Extracted article text:", len(article_text), "characters")
print("\nFirst 2000 characters:\n")
print(article_text[:2000])

Extracted article text: 4637 characters

First 2000 characters:

Modeling long-range dependencies for weakly supervised disease classification and localization on chest X-ray

Contributions: (I) Conception and design: All authors; (II) Administrative support: All authors; (III) Provision of study materials or patients: C Chen, S Yang; (IV) Collection and assembly of data: F Li; (V) Data analysis and interpretation: F Li; (VI) Manuscript writing: All authors; (VII) Final approval of manuscript: All authors.

These authors contributed equally to this work and should be considered as co-first authors.

Conflicts of Interest: All authors have completed the ICMJE uniform disclosure form (available at https://qims.amegroups.com/article/view/10.21037/qims-21-1117/coif ). FL has a pending patent (No. 202110640995.8). The other authors have no conflicts of interest to declare.

Background Computer-aided diagnosis based on chest X-ray (CXR) is an exponentially growing field of research owing to 

In [8]:
import os

print("Current working directory:")
print(os.getcwd())

print("\nContents:")
print(os.listdir())

Current working directory:
c:\Users\yuvan\Desktop\ML_projects\Chest_Xray_AI\notebooks

Contents:
['Chest_Xray_AI.ipynb', 'rag', 'RAG_LLM.ipynb']


In [10]:
from pathlib import Path

PROJECT_ROOT = Path("..").resolve()

RAG_DIR = PROJECT_ROOT / "rag"
DOCUMENTS_DIR = RAG_DIR / "documents"
CHUNKS_DIR = RAG_DIR / "chunks"
VECTOR_STORE_DIR = RAG_DIR / "vector_store"
METADATA_DIR = RAG_DIR / "metadata"

for directory in [
    RAG_DIR,
    DOCUMENTS_DIR,
    CHUNKS_DIR,
    VECTOR_STORE_DIR,
    METADATA_DIR
]:
    directory.mkdir(parents=True, exist_ok=True)

print("Project root:", PROJECT_ROOT)
print("RAG directory:", RAG_DIR)
print("Documents directory:", DOCUMENTS_DIR)
print("✓ RAG paths ready!")

Project root: C:\Users\yuvan\Desktop\ML_projects\Chest_Xray_AI
RAG directory: C:\Users\yuvan\Desktop\ML_projects\Chest_Xray_AI\rag
Documents directory: C:\Users\yuvan\Desktop\ML_projects\Chest_Xray_AI\rag\documents
✓ RAG paths ready!


In [11]:
document_path = DOCUMENTS_DIR / "chestxray14_research.txt"

with open(
    document_path,
    "w",
    encoding="utf-8"
) as f:
    f.write(article_text)

print("Document saved successfully!")
print("Characters:", len(article_text))
print("Path:", document_path)

Document saved successfully!
Characters: 4637
Path: C:\Users\yuvan\Desktop\ML_projects\Chest_Xray_AI\rag\documents\chestxray14_research.txt


In [12]:
# Inspect the document structure

with open(
    document_path,
    "r",
    encoding="utf-8"
) as f:
    text = f.read()

paragraphs = [
    p.strip()
    for p in text.split("\n\n")
    if p.strip()
]

print("Total characters:", len(text))
print("Total paragraphs:", len(paragraphs))

print("\nFirst 10 paragraphs:\n")

for i, paragraph in enumerate(paragraphs[:10]):
    print(f"\n--- Paragraph {i + 1} ---")
    print(paragraph[:500])

Total characters: 4637
Total paragraphs: 9

First 10 paragraphs:


--- Paragraph 1 ---
Modeling long-range dependencies for weakly supervised disease classification and localization on chest X-ray

--- Paragraph 2 ---
Contributions: (I) Conception and design: All authors; (II) Administrative support: All authors; (III) Provision of study materials or patients: C Chen, S Yang; (IV) Collection and assembly of data: F Li; (V) Data analysis and interpretation: F Li; (VI) Manuscript writing: All authors; (VII) Final approval of manuscript: All authors.

--- Paragraph 3 ---
These authors contributed equally to this work and should be considered as co-first authors.

--- Paragraph 4 ---
Conflicts of Interest: All authors have completed the ICMJE uniform disclosure form (available at https://qims.amegroups.com/article/view/10.21037/qims-21-1117/coif ). FL has a pending patent (No. 202110640995.8). The other authors have no conflicts of interest to declare.

--- Paragraph 5 ---
Background Compu

In [13]:
from bs4 import BeautifulSoup

soup = BeautifulSoup(response.text, "xml")

title = soup.find("article-title")
title_text = title.get_text(" ", strip=True) if title else ""

abstract = soup.find("abstract")
abstract_text = abstract.get_text(" ", strip=True) if abstract else ""

# Extract paragraph-level content only
paragraphs = []

for p in soup.find_all("p"):
    text = p.get_text(" ", strip=True)
    if text:
        paragraphs.append(text)

# Build clean document
clean_article_text = "\n\n".join(
    [title_text, abstract_text] + paragraphs
)

print("Cleaned document characters:", len(clean_article_text))
print("Paragraphs:", len(paragraphs))

print("\nFirst 2000 characters:\n")
print(clean_article_text[:2000])

Cleaned document characters: 4637
Paragraphs: 7

First 2000 characters:

Modeling long-range dependencies for weakly supervised disease classification and localization on chest X-ray

Background Computer-aided diagnosis based on chest X-ray (CXR) is an exponentially growing field of research owing to the development of deep learning, especially convolutional neural networks (CNNs). However, due to the intrinsic locality of convolution operations, CNNs cannot model long-range dependencies. Although vision transformers (ViTs) have recently been proposed to alleviate this limitation, those trained on patches cannot learn any dependencies for inter-patch pixels and thus, are insufficient for medical image detection. To address this problem, in this paper, we propose a CXR detection method which integrates CNN with a ViT for modeling patch-wise and inter-patch dependencies. Methods We experimented on the ChestX-ray14 dataset and followed the official training-test set split. Because the tra

In [14]:
document_path = DOCUMENTS_DIR / "chestxray14_research.txt"

with open(
    document_path,
    "w",
    encoding="utf-8"
) as f:
    f.write(clean_article_text)

print("Clean document saved!")
print("Characters:", len(clean_article_text))
print("Path:", document_path)

Clean document saved!
Characters: 4637
Path: C:\Users\yuvan\Desktop\ML_projects\Chest_Xray_AI\rag\documents\chestxray14_research.txt


In [15]:
chunk_size = 800
chunk_overlap = 150

chunks = []

start = 0
text_length = len(clean_article_text)

while start < text_length:
    end = min(start + chunk_size, text_length)

    chunk = clean_article_text[start:end].strip()

    if chunk:
        chunks.append(chunk)

    start += chunk_size - chunk_overlap

print("Document characters:", text_length)
print("Number of chunks:", len(chunks))

print("\nChunk sizes:")
for i, chunk in enumerate(chunks):
    print(f"Chunk {i + 1}: {len(chunk)} characters")

print("\nFirst chunk:\n")
print(chunks[0])

Document characters: 4637
Number of chunks: 8

Chunk sizes:
Chunk 1: 800 characters
Chunk 2: 800 characters
Chunk 3: 799 characters
Chunk 4: 799 characters
Chunk 5: 800 characters
Chunk 6: 799 characters
Chunk 7: 737 characters
Chunk 8: 87 characters

First chunk:

Modeling long-range dependencies for weakly supervised disease classification and localization on chest X-ray

Background Computer-aided diagnosis based on chest X-ray (CXR) is an exponentially growing field of research owing to the development of deep learning, especially convolutional neural networks (CNNs). However, due to the intrinsic locality of convolution operations, CNNs cannot model long-range dependencies. Although vision transformers (ViTs) have recently been proposed to alleviate this limitation, those trained on patches cannot learn any dependencies for inter-patch pixels and thus, are insufficient for medical image detection. To address this problem, in this paper, we propose a CXR detection method which integ

In [16]:
MIN_CHUNK_SIZE = 200

clean_chunks = []

for chunk in chunks:
    chunk = chunk.strip()

    if not chunk:
        continue

    if len(chunk) < MIN_CHUNK_SIZE and clean_chunks:
        clean_chunks[-1] += " " + chunk
    else:
        clean_chunks.append(chunk)

chunks = clean_chunks

print("Final number of chunks:", len(chunks))

for i, chunk in enumerate(chunks):
    print(f"Chunk {i + 1}: {len(chunk)} characters")

Final number of chunks: 7
Chunk 1: 800 characters
Chunk 2: 800 characters
Chunk 3: 799 characters
Chunk 4: 799 characters
Chunk 5: 800 characters
Chunk 6: 799 characters
Chunk 7: 825 characters


In [17]:
chunk_records = []

for i, chunk in enumerate(chunks):
    chunk_records.append({
        "chunk_id": f"chestxray14_{i+1:03d}",
        "source_id": "chestxray14_research",
        "source_title": "ChestX-ray14 Dataset and Multi-Label Chest X-ray Classification",
        "source_url": "https://pmc.ncbi.nlm.nih.gov/articles/PMC9131331/",
        "chunk_index": i,
        "text": chunk
    })

chunks_path = CHUNKS_DIR / "chestxray14_chunks.json"

with open(chunks_path, "w", encoding="utf-8") as f:
    json.dump(chunk_records, f, indent=2, ensure_ascii=False)

print("Saved:", chunks_path)
print("Number of chunk records:", len(chunk_records))
print("\nFirst chunk metadata:")
print(json.dumps(chunk_records[0], indent=2, ensure_ascii=False))

Saved: C:\Users\yuvan\Desktop\ML_projects\Chest_Xray_AI\rag\chunks\chestxray14_chunks.json
Number of chunk records: 7

First chunk metadata:
{
  "chunk_id": "chestxray14_001",
  "source_id": "chestxray14_research",
  "source_title": "ChestX-ray14 Dataset and Multi-Label Chest X-ray Classification",
  "source_url": "https://pmc.ncbi.nlm.nih.gov/articles/PMC9131331/",
  "chunk_index": 0,
  "text": "Modeling long-range dependencies for weakly supervised disease classification and localization on chest X-ray\n\nBackground Computer-aided diagnosis based on chest X-ray (CXR) is an exponentially growing field of research owing to the development of deep learning, especially convolutional neural networks (CNNs). However, due to the intrinsic locality of convolution operations, CNNs cannot model long-range dependencies. Although vision transformers (ViTs) have recently been proposed to alleviate this limitation, those trained on patches cannot learn any dependencies for inter-patch pixels and t

In [18]:
embedding_model = SentenceTransformer("all-MiniLM-L6-v2")

print("Embedding model loaded successfully.")
print("Embedding dimension:", embedding_model.get_sentence_embedding_dimension())

c:\Users\yuvan\Desktop\ML_projects\Chest_Xray_AI\.venv\Lib\site-packages\huggingface_hub\file_download.py:149: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\yuvan\.cache\huggingface\hub\models--sentence-transformers--all-MiniLM-L6-v2. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)
c:\Users\yuvan\Desktop\ML_projects\Chest_Xray_AI\.venv\Lib\site-packages\hu

Embedding model loaded successfully.
Embedding dimension: 384


C:\Users\yuvan\AppData\Local\Temp\ipykernel_36920\3015113085.py:4: FutureWarning: The `get_sentence_embedding_dimension` method has been renamed to `get_embedding_dimension`.
  print("Embedding dimension:", embedding_model.get_sentence_embedding_dimension())


In [19]:
texts = [record["text"] for record in chunk_records]

embeddings = embedding_model.encode(
    texts,
    convert_to_numpy=True,
    normalize_embeddings=True,
    show_progress_bar=True
)

print("Embeddings shape:", embeddings.shape)
print("Embedding dtype:", embeddings.dtype)
print("First embedding (first 10 values):")
print(embeddings[0][:10])

Batches: 100%|██████████| 1/1 [00:00<00:00,  1.83it/s]

Embeddings shape: (7, 384)
Embedding dtype: float32
First embedding (first 10 values):
[-0.00103039 -0.01264188  0.07845823 -0.06923418  0.08762156 -0.02444632
 -0.04653818  0.0073574  -0.03253129 -0.1186422 ]


In [20]:
embedding_dim = embeddings.shape[1]

index = faiss.IndexFlatIP(embedding_dim)
index.add(embeddings)

print("FAISS index created successfully.")
print("Embedding dimension:", embedding_dim)
print("Vectors stored in index:", index.ntotal)

FAISS index created successfully.
Embedding dimension: 384
Vectors stored in index: 7


In [21]:
vector_store_path = VECTOR_STORE_DIR / "chestxray14.index"

faiss.write_index(index, str(vector_store_path))

print("FAISS index saved successfully.")
print("Path:", vector_store_path)
print("File size (KB):", round(vector_store_path.stat().st_size / 1024, 2))

FAISS index saved successfully.
Path: C:\Users\yuvan\Desktop\ML_projects\Chest_Xray_AI\rag\vector_store\chestxray14.index
File size (KB): 10.54


In [22]:
metadata_records = [
    {
        "faiss_index": record["chunk_index"],
        "chunk_id": record["chunk_id"],
        "source_id": record["source_id"],
        "source_title": record["source_title"],
        "source_url": record["source_url"]
    }
    for record in chunk_records
]

metadata_path = METADATA_DIR / "chestxray14_metadata.json"

with open(metadata_path, "w", encoding="utf-8") as f:
    json.dump(metadata_records, f, indent=2, ensure_ascii=False)

print("Metadata saved successfully.")
print("Path:", metadata_path)
print("Records:", len(metadata_records))
print("\nFirst record:")
print(json.dumps(metadata_records[0], indent=2, ensure_ascii=False))

Metadata saved successfully.
Path: C:\Users\yuvan\Desktop\ML_projects\Chest_Xray_AI\rag\metadata\chestxray14_metadata.json
Records: 7

First record:
{
  "faiss_index": 0,
  "chunk_id": "chestxray14_001",
  "source_id": "chestxray14_research",
  "source_title": "ChestX-ray14 Dataset and Multi-Label Chest X-ray Classification",
  "source_url": "https://pmc.ncbi.nlm.nih.gov/articles/PMC9131331/"
}


In [23]:
query = "What is the ChestX-ray14 dataset?"

query_embedding = embedding_model.encode(
    [query],
    convert_to_numpy=True,
    normalize_embeddings=True
)

top_k = 3
scores, indices = index.search(query_embedding, top_k)

print("Query:", query)
print("\nRetrieved chunks:\n")

for rank, (score, idx) in enumerate(zip(scores[0], indices[0]), start=1):
    record = chunk_records[idx]

    print(f"--- Rank {rank} | Score: {score:.4f} ---")
    print("Chunk ID:", record["chunk_id"])
    print(record["text"][:500])
    print()

Query: What is the ChestX-ray14 dataset?

Retrieved chunks:

--- Rank 1 | Score: 0.4504 ---
Chunk ID: chestxray14_002
o address this problem, in this paper, we propose a CXR detection method which integrates CNN with a ViT for modeling patch-wise and inter-patch dependencies. Methods We experimented on the ChestX-ray14 dataset and followed the official training-test set split. Because the training data only had global annotations, the detection network was weakly supervised. A DenseNet with a feature pyramid structure was designed and integrated with an adaptive ViT to model inter-patch and patch-wise long-rang

--- Rank 2 | Score: 0.4503 ---
Chunk ID: chestxray14_005
cle/view/10.21037/qims-21-1117/coif ). FL has a pending patent (No. 202110640995.8). The other authors have no conflicts of interest to declare.

Computer-aided diagnosis based on chest X-ray (CXR) is an exponentially growing field of research owing to the development of deep learning, especially convolutional neural netw

In [24]:
for i, record in enumerate(chunk_records):
    print("=" * 80)
    print(f"CHUNK {i + 1} | {record['chunk_id']} | {len(record['text'])} characters")
    print("=" * 80)
    print(record["text"])
    print()

CHUNK 1 | chestxray14_001 | 800 characters
Modeling long-range dependencies for weakly supervised disease classification and localization on chest X-ray

Background Computer-aided diagnosis based on chest X-ray (CXR) is an exponentially growing field of research owing to the development of deep learning, especially convolutional neural networks (CNNs). However, due to the intrinsic locality of convolution operations, CNNs cannot model long-range dependencies. Although vision transformers (ViTs) have recently been proposed to alleviate this limitation, those trained on patches cannot learn any dependencies for inter-patch pixels and thus, are insufficient for medical image detection. To address this problem, in this paper, we propose a CXR detection method which integrates CNN with a ViT for modeling patch-wise and inter-patch depen

CHUNK 2 | chestxray14_002 | 800 characters
o address this problem, in this paper, we propose a CXR detection method which integrates CNN with a ViT for mod

In [25]:
sections = soup.find_all("sec")

print("Number of sections:", len(sections))

for i, sec in enumerate(sections):
    title_tag = sec.find("title", recursive=False)
    title_text = title_tag.get_text(" ", strip=True) if title_tag else "[No title]"
    print(f"{i + 1}. {title_text}")

Number of sections: 4
1. Background
2. Methods
3. Results
4. Conclusions


In [26]:
clean_sections = []

for sec in soup.find_all("sec", recursive=False):
    title_tag = sec.find("title", recursive=False)
    
    if not title_tag:
        continue
    
    section_title = title_tag.get_text(" ", strip=True)
    
    paragraphs = []
    for p in sec.find_all("p"):
        text = p.get_text(" ", strip=True)
        if text:
            paragraphs.append(text)
    
    section_text = "\n\n".join(paragraphs).strip()
    
    if section_text:
        clean_sections.append({
            "section": section_title,
            "text": section_text
        })

print("Clean sections extracted:", len(clean_sections))

for section in clean_sections:
    print(f"\n{'=' * 70}")
    print(section["section"])
    print(f"{'=' * 70}")
    print("Characters:", len(section["text"]))
    print(section["text"][:500])

Clean sections extracted: 0


In [27]:
clean_sections = []

for sec in soup.find_all("sec"):
    title_tag = sec.find("title", recursive=False)

    if not title_tag:
        continue

    section_title = title_tag.get_text(" ", strip=True)

    # Only keep the four main article sections
    if section_title not in ["Background", "Methods", "Results", "Conclusions"]:
        continue

    paragraphs = []
    for p in sec.find_all("p"):
        text = p.get_text(" ", strip=True)
        if text:
            paragraphs.append(text)

    section_text = "\n\n".join(paragraphs).strip()

    if section_text:
        clean_sections.append({
            "section": section_title,
            "text": section_text
        })

print("Clean sections extracted:", len(clean_sections))

for section in clean_sections:
    print(f"\n{'=' * 70}")
    print(section["section"])
    print(f"{'=' * 70}")
    print("Characters:", len(section["text"]))
    print(section["text"][:500])

Clean sections extracted: 4

Background
Characters: 686
Computer-aided diagnosis based on chest X-ray (CXR) is an exponentially growing field of research owing to the development of deep learning, especially convolutional neural networks (CNNs). However, due to the intrinsic locality of convolution operations, CNNs cannot model long-range dependencies. Although vision transformers (ViTs) have recently been proposed to alleviate this limitation, those trained on patches cannot learn any dependencies for inter-patch pixels and thus, are insufficient fo

Methods
Characters: 476
We experimented on the ChestX-ray14 dataset and followed the official training-test set split. Because the training data only had global annotations, the detection network was weakly supervised. A DenseNet with a feature pyramid structure was designed and integrated with an adaptive ViT to model inter-patch and patch-wise long-range dependencies and obtain fine-grained feature maps. We compared the performance usin

In [28]:
semantic_chunks = []

for i, section in enumerate(clean_sections):
    semantic_chunks.append({
        "chunk_id": f"chestxray14_{i+1:03d}",
        "source_id": "chestxray14_research",
        "source_title": "ChestX-ray14 Dataset and Multi-Label Chest X-ray Classification",
        "source_url": "https://pmc.ncbi.nlm.nih.gov/articles/PMC9131331/",
        "section": section["section"],
        "chunk_index": i,
        "text": section["text"]
    })

print("Semantic chunks:", len(semantic_chunks))

for chunk in semantic_chunks:
    print(
        f"{chunk['chunk_id']} | "
        f"{chunk['section']} | "
        f"{len(chunk['text'])} characters"
    )

Semantic chunks: 4
chestxray14_001 | Background | 686 characters
chestxray14_002 | Methods | 476 characters
chestxray14_003 | Results | 542 characters
chestxray14_004 | Conclusions | 181 characters


In [29]:
chunk_records = semantic_chunks

chunks_path = CHUNKS_DIR / "chestxray14_chunks.json"

with open(chunks_path, "w", encoding="utf-8") as f:
    json.dump(chunk_records, f, indent=2, ensure_ascii=False)

print("Clean chunk file updated successfully.")
print("Path:", chunks_path)
print("Number of chunks:", len(chunk_records))

Clean chunk file updated successfully.
Path: C:\Users\yuvan\Desktop\ML_projects\Chest_Xray_AI\rag\chunks\chestxray14_chunks.json
Number of chunks: 4


In [30]:
texts = [record["text"] for record in chunk_records]

embeddings = embedding_model.encode(
    texts,
    convert_to_numpy=True,
    normalize_embeddings=True,
    show_progress_bar=True
)

print("New embeddings shape:", embeddings.shape)
print("Embedding dtype:", embeddings.dtype)

Batches: 100%|██████████| 1/1 [00:00<00:00, 18.39it/s]

New embeddings shape: (4, 384)
Embedding dtype: float32


In [31]:
embedding_dim = embeddings.shape[1]

index = faiss.IndexFlatIP(embedding_dim)
index.add(embeddings)

print("FAISS index rebuilt successfully.")
print("Embedding dimension:", embedding_dim)
print("Vectors stored in index:", index.ntotal)

FAISS index rebuilt successfully.
Embedding dimension: 384
Vectors stored in index: 4


In [32]:
vector_store_path = VECTOR_STORE_DIR / "chestxray14.index"

faiss.write_index(index, str(vector_store_path))

print("Updated FAISS index saved successfully.")
print("Path:", vector_store_path)
print("Vectors stored:", index.ntotal)
print("File size (KB):", round(vector_store_path.stat().st_size / 1024, 2))

Updated FAISS index saved successfully.
Path: C:\Users\yuvan\Desktop\ML_projects\Chest_Xray_AI\rag\vector_store\chestxray14.index
Vectors stored: 4
File size (KB): 6.04


In [33]:
metadata_records = [
    {
        "faiss_index": record["chunk_index"],
        "chunk_id": record["chunk_id"],
        "source_id": record["source_id"],
        "source_title": record["source_title"],
        "source_url": record["source_url"],
        "section": record["section"]
    }
    for record in chunk_records
]

metadata_path = METADATA_DIR / "chestxray14_metadata.json"

with open(metadata_path, "w", encoding="utf-8") as f:
    json.dump(metadata_records, f, indent=2, ensure_ascii=False)

print("Metadata updated successfully.")
print("Records:", len(metadata_records))

for record in metadata_records:
    print(
        f"{record['faiss_index']} → "
        f"{record['chunk_id']} → "
        f"{record['section']}"
    )

Metadata updated successfully.
Records: 4
0 → chestxray14_001 → Background
1 → chestxray14_002 → Methods
2 → chestxray14_003 → Results
3 → chestxray14_004 → Conclusions


In [34]:
query = "What is the ChestX-ray14 dataset?"

query_embedding = embedding_model.encode(
    [query],
    convert_to_numpy=True,
    normalize_embeddings=True
)

top_k = 3
scores, indices = index.search(query_embedding, top_k)

print("Query:", query)
print("\nRetrieved chunks:\n")

for rank, (score, idx) in enumerate(zip(scores[0], indices[0]), start=1):
    record = chunk_records[idx]

    print(f"--- Rank {rank} | Score: {score:.4f} ---")
    print("Chunk ID:", record["chunk_id"])
    print("Section:", record["section"])
    print(record["text"])
    print()

Query: What is the ChestX-ray14 dataset?

Retrieved chunks:

--- Rank 1 | Score: 0.5667 ---
Chunk ID: chestxray14_002
Section: Methods
We experimented on the ChestX-ray14 dataset and followed the official training-test set split. Because the training data only had global annotations, the detection network was weakly supervised. A DenseNet with a feature pyramid structure was designed and integrated with an adaptive ViT to model inter-patch and patch-wise long-range dependencies and obtain fine-grained feature maps. We compared the performance using our method with that of other disease detection methods.

--- Rank 2 | Score: 0.4160 ---
Chunk ID: chestxray14_001
Section: Background
Computer-aided diagnosis based on chest X-ray (CXR) is an exponentially growing field of research owing to the development of deep learning, especially convolutional neural networks (CNNs). However, due to the intrinsic locality of convolution operations, CNNs cannot model long-range dependencies. Although vi

In [35]:
rag_sources = [
    {
        "id": "chestxray14_dataset",
        "title": "NIH ChestX-ray14 Dataset",
        "organization": "NIH Clinical Center / Research Literature",
        "url": "https://pmc.ncbi.nlm.nih.gov/articles/PMC9131331/",
        "type": "dataset_research",
        "purpose": "Dataset characteristics, 14 findings, multi-label classification, and label generation"
    },
    {
        "id": "radiologyinfo_chest_xray",
        "title": "Chest X-ray (Radiography)",
        "organization": "RadiologyInfo",
        "url": "https://www.radiologyinfo.org/en/info/chestrad",
        "type": "medical_reference",
        "purpose": "Chest X-ray overview, uses, procedure, interpretation context, and limitations"
    },
    {
        "id": "radiologyinfo_chest_xray_report",
        "title": "How to Read Your Chest X-ray Report",
        "organization": "RadiologyInfo",
        "url": "https://www.radiologyinfo.org/en/info/article-chest-xray-report",
        "type": "medical_reference",
        "purpose": "Radiology report terminology, findings, impression, and common chest X-ray findings"
    }
]

sources_path = METADATA_DIR / "sources.json"

with open(sources_path, "w", encoding="utf-8") as f:
    json.dump(rag_sources, f, indent=2, ensure_ascii=False)

print("Source registry updated successfully.")
print("Number of sources:", len(rag_sources))

for source in rag_sources:
    print(f"{source['id']} → {source['title']}")

Source registry updated successfully.
Number of sources: 3
chestxray14_dataset → NIH ChestX-ray14 Dataset
radiologyinfo_chest_xray → Chest X-ray (Radiography)
radiologyinfo_chest_xray_report → How to Read Your Chest X-ray Report


In [36]:
pmc_id = "PMC9131331"

url = (
    "https://eutils.ncbi.nlm.nih.gov/entrez/eutils/"
    f"efetch.fcgi?db=pmc&id={pmc_id}&retmode=xml"
)

response = requests.get(url, timeout=60)

print("HTTP status:", response.status_code)
print("Downloaded characters:", len(response.text))
print("Content starts with:", response.text[:100])

HTTP status: 200
Downloaded characters: 11528
Content starts with: <?xml version="1.0"  ?><!DOCTYPE pmc-articleset PUBLIC "-//NLM//DTD ARTICLE SET 2.0//EN" "https://dt


In [37]:
source_soup = BeautifulSoup(response.text, "xml")

target_sections = ["Background", "Methods", "Results", "Conclusions"]

source_sections = []

for sec in source_soup.find_all("sec"):
    title_tag = sec.find("title", recursive=False)

    if not title_tag:
        continue

    section_title = title_tag.get_text(" ", strip=True)

    if section_title not in target_sections:
        continue

    paragraphs = []

    for p in sec.find_all("p"):
        text = p.get_text(" ", strip=True)
        if text:
            paragraphs.append(text)

    section_text = "\n\n".join(paragraphs).strip()

    if section_text:
        source_sections.append({
            "section": section_title,
            "text": section_text
        })

print("Sections extracted:", len(source_sections))

for section in source_sections:
    print(
        f"{section['section']}: "
        f"{len(section['text'])} characters"
    )

Sections extracted: 4
Background: 686 characters
Methods: 476 characters
Results: 542 characters
Conclusions: 181 characters


In [38]:
nih_document = "\n\n".join(
    f"{section['section']}\n\n{section['text']}"
    for section in source_sections
)

nih_path = DOCUMENTS_DIR / "chestxray14_nih_research.txt"

with open(nih_path, "w", encoding="utf-8") as f:
    f.write(nih_document)

print("NIH research document saved successfully.")
print("Path:", nih_path)
print("Characters:", len(nih_document))

NIH research document saved successfully.
Path: C:\Users\yuvan\Desktop\ML_projects\Chest_Xray_AI\rag\documents\chestxray14_nih_research.txt
Characters: 1934


In [39]:
radiology_url = "https://www.radiologyinfo.org/en/info/chestrad"

response_radiology = requests.get(
    radiology_url,
    timeout=60,
    headers={
        "User-Agent": "Mozilla/5.0"
    }
)

print("HTTP status:", response_radiology.status_code)
print("Downloaded characters:", len(response_radiology.text))
print("Content starts with:", response_radiology.text[:100])

HTTP status: 200
Downloaded characters: 55182
Content starts with: 


<!doctype html>
<html lang="en">

<head>
<meta charset="utf-8" />
<title>Chest X-ray (Rad


In [40]:
radiology_soup = BeautifulSoup(response_radiology.text, "lxml")

headings = radiology_soup.find_all(["h1", "h2", "h3"])

print("Number of headings:", len(headings))

for i, heading in enumerate(headings[:40]):
    text = heading.get_text(" ", strip=True)
    if text:
        print(f"{i + 1}. {heading.name}: {text}")

Number of headings: 18
1. h1: Chest X-ray
2. h2: What is a chest X-ray?
3. h2: What are some common uses of the procedure?
4. h2: How should I prepare?
5. h2: What does the equipment look like?
6. h2: How does the procedure work?
7. h2: How is the procedure performed?
8. h2: What will I experience during and after the procedure?
9. h2: Who interprets the results and how do I get them?
10. h2: What are the benefits vs. risks?
11. h3: Benefits
12. h3: Risks
13. h3: A Word About Minimizing Radiation Exposure
14. h2: What are the limitations of chest x-ray?
15. h2: Additional Information and Resources
16. h2: Send us your feedback
17. h2: Image Gallery
18. h1: Glossary


In [41]:
target_headings = [
    "What is a chest X-ray?",
    "What are some common uses of the procedure?",
    "How should I prepare?",
    "What does the equipment look like?",
    "How does the procedure work?",
    "How is the procedure performed?",
    "What will I experience during and after the procedure?",
    "Who interprets the results and how do I get them?",
    "What are the benefits vs. risks?",
    "What are the limitations of chest x-ray?"
]

radiology_sections = []

for heading in radiology_soup.find_all(["h2", "h3"]):
    heading_text = heading.get_text(" ", strip=True)

    if heading_text not in target_headings:
        continue

    content_parts = []
    
    for element in heading.find_all_next():
        if element.name in ["h2", "h1"] and element is not heading:
            break
        
        if element.name in ["p", "li"]:
            text = element.get_text(" ", strip=True)
            if text:
                content_parts.append(text)

    section_text = "\n\n".join(content_parts).strip()

    if section_text:
        radiology_sections.append({
            "section": heading_text,
            "text": section_text
        })

print("Sections extracted:", len(radiology_sections))

for section in radiology_sections:
    print(
        f"{section['section']} → "
        f"{len(section['text'])} characters"
    )

Sections extracted: 10
What is a chest X-ray? → 437 characters
What are some common uses of the procedure? → 526 characters
How should I prepare? → 627 characters
What does the equipment look like? → 700 characters
How does the procedure work? → 996 characters
How is the procedure performed? → 1157 characters
What will I experience during and after the procedure? → 461 characters
Who interprets the results and how do I get them? → 693 characters
What are the benefits vs. risks? → 1446 characters
What are the limitations of chest x-ray? → 552 characters


In [42]:
radiology_document = "\n\n".join(
    f"{section['section']}\n\n{section['text']}"
    for section in radiology_sections
)

radiology_path = DOCUMENTS_DIR / "radiologyinfo_chest_xray.txt"

with open(radiology_path, "w", encoding="utf-8") as f:
    f.write(radiology_document)

print("RadiologyInfo document saved successfully.")
print("Path:", radiology_path)
print("Characters:", len(radiology_document))

RadiologyInfo document saved successfully.
Path: C:\Users\yuvan\Desktop\ML_projects\Chest_Xray_AI\rag\documents\radiologyinfo_chest_xray.txt
Characters: 7987


In [43]:
report_url = "https://www.radiologyinfo.org/en/info/article-chest-xray-report"

response_report = requests.get(
    report_url,
    timeout=60,
    headers={
        "User-Agent": "Mozilla/5.0"
    }
)

print("HTTP status:", response_report.status_code)
print("Downloaded characters:", len(response_report.text))
print("Content starts with:", response_report.text[:100])

HTTP status: 200
Downloaded characters: 45585
Content starts with: 


<!doctype html>
<html lang="en">

<head>
<meta charset="utf-8" />
<title>How to Read Your


In [44]:
report_url = "https://www.radiologyinfo.org/en/info/article-chest-xray-report"

response_report = requests.get(
    report_url,
    timeout=60,
    headers={
        "User-Agent": "Mozilla/5.0"
    }
)

print("HTTP status:", response_report.status_code)
print("Downloaded characters:", len(response_report.text))
print("Content starts with:", response_report.text[:100])

HTTP status: 200
Downloaded characters: 45585
Content starts with: 


<!doctype html>
<html lang="en">

<head>
<meta charset="utf-8" />
<title>How to Read Your


In [45]:
report_soup = BeautifulSoup(response_report.text, "lxml")

headings = report_soup.find_all(["h1", "h2", "h3"])

print("Number of headings:", len(headings))

for i, heading in enumerate(headings[:40]):
    text = heading.get_text(" ", strip=True)
    if text:
        print(f"{i + 1}. {heading.name}: {text}")

Number of headings: 13
1. h1: How to Read Your Chest X-ray Report
2. h2: What is Chest X-ray commonly used for?
3. h2: Sections of the Radiology Report
4. h3: Type of exam
5. h3: History/Reason for exam
6. h3: Comparison/Priors
7. h3: Technique
8. h3: Findings
9. h3: Impression
10. h2: Additional Information
11. h2: Send us your feedback
12. h2: Image Gallery
13. h1: Glossary


In [46]:
target_report_headings = [
    "What is Chest X-ray commonly used for?",
    "Sections of the Radiology Report",
    "Type of exam",
    "History/Reason for exam",
    "Comparison/Priors",
    "Technique",
    "Findings",
    "Impression",
    "Additional Information"
]

report_sections = []

for heading in report_soup.find_all(["h2", "h3"]):
    heading_text = heading.get_text(" ", strip=True)

    if heading_text not in target_report_headings:
        continue

    content_parts = []

    for element in heading.find_all_next():
        if element.name in ["h2", "h1"] and element is not heading:
            break

        if element.name in ["p", "li"]:
            text = element.get_text(" ", strip=True)
            if text:
                content_parts.append(text)

    section_text = "\n\n".join(content_parts).strip()

    if section_text:
        report_sections.append({
            "section": heading_text,
            "text": section_text
        })

print("Sections extracted:", len(report_sections))

for section in report_sections:
    print(
        f"{section['section']} → "
        f"{len(section['text'])} characters"
    )

Sections extracted: 9
What is Chest X-ray commonly used for? → 434 characters
Sections of the Radiology Report → 4131 characters
Type of exam → 4131 characters
History/Reason for exam → 3963 characters
Comparison/Priors → 3345 characters
Technique → 2579 characters
Findings → 2237 characters
Impression → 1381 characters
Additional Information → 594 characters


In [47]:
report_sections = []

all_headings = report_soup.find_all(["h1", "h2", "h3"])

for heading in all_headings:
    heading_text = heading.get_text(" ", strip=True)

    if heading_text not in target_report_headings:
        continue

    current_level = int(heading.name[1])
    content_parts = []

    for element in heading.find_all_next():
        if element.name in ["h1", "h2", "h3"]:
            next_level = int(element.name[1])

            if next_level <= current_level:
                break

        if element.name in ["p", "li"]:
            text = element.get_text(" ", strip=True)
            if text:
                content_parts.append(text)

    section_text = "\n\n".join(content_parts).strip()

    if section_text:
        report_sections.append({
            "section": heading_text,
            "level": current_level,
            "text": section_text
        })

print("Sections extracted:", len(report_sections))

for section in report_sections:
    print(
        f"{section['section']} "
        f"(h{section['level']}) → "
        f"{len(section['text'])} characters"
    )

Sections extracted: 9
What is Chest X-ray commonly used for? (h2) → 434 characters
Sections of the Radiology Report (h2) → 4131 characters
Type of exam (h3) → 166 characters
History/Reason for exam (h3) → 616 characters
Comparison/Priors (h3) → 764 characters
Technique (h3) → 340 characters
Findings (h3) → 854 characters
Impression (h3) → 1381 characters
Additional Information (h2) → 594 characters


In [48]:
report_sections_clean = [
    section for section in report_sections
    if section["section"] != "Sections of the Radiology Report"
]

report_document = "\n\n".join(
    f"{section['section']}\n\n{section['text']}"
    for section in report_sections_clean
)

report_path = DOCUMENTS_DIR / "radiologyinfo_chest_xray_report.txt"

with open(report_path, "w", encoding="utf-8") as f:
    f.write(report_document)

print("RadiologyInfo report document saved successfully.")
print("Path:", report_path)
print("Sections saved:", len(report_sections_clean))
print("Characters:", len(report_document))

RadiologyInfo report document saved successfully.
Path: C:\Users\yuvan\Desktop\ML_projects\Chest_Xray_AI\rag\documents\radiologyinfo_chest_xray_report.txt
Sections saved: 8
Characters: 5318


In [49]:
source_documents = [
    {
        "source_id": "chestxray14_dataset",
        "source_title": "NIH ChestX-ray14 Dataset",
        "source_url": "https://pmc.ncbi.nlm.nih.gov/articles/PMC9131331/",
        "sections": source_sections
    },
    {
        "source_id": "radiologyinfo_chest_xray",
        "source_title": "Chest X-ray (Radiography)",
        "source_url": "https://www.radiologyinfo.org/en/info/chestrad",
        "sections": radiology_sections
    },
    {
        "source_id": "radiologyinfo_chest_xray_report",
        "source_title": "How to Read Your Chest X-ray Report",
        "source_url": "https://www.radiologyinfo.org/en/info/article-chest-xray-report",
        "sections": report_sections_clean
    }
]

semantic_chunks = []

for source in source_documents:
    for section in source["sections"]:
        semantic_chunks.append({
            "source_id": source["source_id"],
            "source_title": source["source_title"],
            "source_url": source["source_url"],
            "section": section["section"],
            "text": section["text"]
        })

print("Total semantic chunks:", len(semantic_chunks))

for i, chunk in enumerate(semantic_chunks, start=1):
    print(
        f"{i:02d}. "
        f"{chunk['source_id']} | "
        f"{chunk['section']} | "
        f"{len(chunk['text'])} chars"
    )

Total semantic chunks: 22
01. chestxray14_dataset | Background | 686 chars
02. chestxray14_dataset | Methods | 476 chars
03. chestxray14_dataset | Results | 542 chars
04. chestxray14_dataset | Conclusions | 181 chars
05. radiologyinfo_chest_xray | What is a chest X-ray? | 437 chars
06. radiologyinfo_chest_xray | What are some common uses of the procedure? | 526 chars
07. radiologyinfo_chest_xray | How should I prepare? | 627 chars
08. radiologyinfo_chest_xray | What does the equipment look like? | 700 chars
09. radiologyinfo_chest_xray | How does the procedure work? | 996 chars
10. radiologyinfo_chest_xray | How is the procedure performed? | 1157 chars
11. radiologyinfo_chest_xray | What will I experience during and after the procedure? | 461 chars
12. radiologyinfo_chest_xray | Who interprets the results and how do I get them? | 693 chars
13. radiologyinfo_chest_xray | What are the benefits vs. risks? | 1446 chars
14. radiologyinfo_chest_xray | What are the limitations of chest x-ray?

In [50]:
for i, chunk in enumerate(semantic_chunks):
    chunk["chunk_id"] = f"rag_{i+1:03d}"
    chunk["chunk_index"] = i

print("Chunk IDs assigned:", len(semantic_chunks))

for chunk in semantic_chunks[:5]:
    print(
        chunk["chunk_id"],
        "→",
        chunk["source_id"],
        "→",
        chunk["section"]
    )
    

Chunk IDs assigned: 22
rag_001 → chestxray14_dataset → Background
rag_002 → chestxray14_dataset → Methods
rag_003 → chestxray14_dataset → Results
rag_004 → chestxray14_dataset → Conclusions
rag_005 → radiologyinfo_chest_xray → What is a chest X-ray?


In [51]:
unified_chunks_path = CHUNKS_DIR / "rag_chunks.json"

with open(unified_chunks_path, "w", encoding="utf-8") as f:
    json.dump(semantic_chunks, f, indent=2, ensure_ascii=False)

print("Unified RAG chunk corpus saved successfully.")
print("Path:", unified_chunks_path)
print("Chunks saved:", len(semantic_chunks))
print("Total characters:", sum(len(chunk["text"]) for chunk in semantic_chunks))

Unified RAG chunk corpus saved successfully.
Path: C:\Users\yuvan\Desktop\ML_projects\Chest_Xray_AI\rag\chunks\rag_chunks.json
Chunks saved: 22
Total characters: 14629


In [52]:
texts = [chunk["text"] for chunk in semantic_chunks]

embeddings = embedding_model.encode(
    texts,
    convert_to_numpy=True,
    normalize_embeddings=True,
    show_progress_bar=True
)

print("Embeddings shape:", embeddings.shape)
print("Embedding dtype:", embeddings.dtype)

Batches: 100%|██████████| 1/1 [00:00<00:00,  4.42it/s]

Embeddings shape: (22, 384)
Embedding dtype: float32


In [53]:
embedding_dim = embeddings.shape[1]

index = faiss.IndexFlatIP(embedding_dim)
index.add(embeddings)

print("FAISS index rebuilt successfully.")
print("Embedding dimension:", embedding_dim)
print("Vectors stored:", index.ntotal)

FAISS index rebuilt successfully.
Embedding dimension: 384
Vectors stored: 22


In [54]:
vector_store_path = VECTOR_STORE_DIR / "rag.index"

faiss.write_index(index, str(vector_store_path))

print("Unified FAISS index saved successfully.")
print("Path:", vector_store_path)
print("Vectors stored:", index.ntotal)
print("File size (KB):", round(vector_store_path.stat().st_size / 1024, 2))

Unified FAISS index saved successfully.
Path: C:\Users\yuvan\Desktop\ML_projects\Chest_Xray_AI\rag\vector_store\rag.index
Vectors stored: 22
File size (KB): 33.04


In [55]:
rag_metadata = [
    {
        "faiss_index": chunk["chunk_index"],
        "chunk_id": chunk["chunk_id"],
        "source_id": chunk["source_id"],
        "source_title": chunk["source_title"],
        "source_url": chunk["source_url"],
        "section": chunk["section"]
    }
    for chunk in semantic_chunks
]

rag_metadata_path = METADATA_DIR / "rag_metadata.json"

with open(rag_metadata_path, "w", encoding="utf-8") as f:
    json.dump(rag_metadata, f, indent=2, ensure_ascii=False)

print("Unified RAG metadata saved successfully.")
print("Path:", rag_metadata_path)
print("Records:", len(rag_metadata))

print("\nFirst 5 records:")
for record in rag_metadata[:5]:
    print(
        f"{record['faiss_index']} → "
        f"{record['chunk_id']} → "
        f"{record['source_id']} → "
        f"{record['section']}"
    )

Unified RAG metadata saved successfully.
Path: C:\Users\yuvan\Desktop\ML_projects\Chest_Xray_AI\rag\metadata\rag_metadata.json
Records: 22

First 5 records:
0 → rag_001 → chestxray14_dataset → Background
1 → rag_002 → chestxray14_dataset → Methods
2 → rag_003 → chestxray14_dataset → Results
3 → rag_004 → chestxray14_dataset → Conclusions
4 → rag_005 → radiologyinfo_chest_xray → What is a chest X-ray?


In [56]:
test_queries = [
    "What is the ChestX-ray14 dataset?",
    "What is a chest X-ray used for?",
    "What does the Findings section of a radiology report contain?",
    "What is pneumothorax?",
    "What are the limitations of a chest X-ray?"
]

for query in test_queries:
    query_embedding = embedding_model.encode(
        [query],
        convert_to_numpy=True,
        normalize_embeddings=True
    )

    scores, indices = index.search(query_embedding, 3)

    print("\n" + "=" * 90)
    print("QUERY:", query)
    print("=" * 90)

    for rank, (score, idx) in enumerate(
        zip(scores[0], indices[0]), start=1
    ):
        chunk = semantic_chunks[idx]

        print(
            f"\nRank {rank} | Score: {score:.4f} | "
            f"{chunk['source_id']} | {chunk['section']}"
        )
        print(chunk["text"][:350].replace("\n", " "))


QUERY: What is the ChestX-ray14 dataset?

Rank 1 | Score: 0.5667 | chestxray14_dataset | Methods
We experimented on the ChestX-ray14 dataset and followed the official training-test set split. Because the training data only had global annotations, the detection network was weakly supervised. A DenseNet with a feature pyramid structure was designed and integrated with an adaptive ViT to model inter-patch and patch-wise long-range dependencies an

Rank 2 | Score: 0.4719 | radiologyinfo_chest_xray | What are some common uses of the procedure?
The chest x-ray is performed to evaluate the lungs, heart and chest wall.  A chest x-ray is typically the first imaging test used to help diagnose symptoms such as:  breathing difficulties  a bad or persistent cough  chest pain or injury  fever  Physicians use the examination to help diagnose or monitor treatment for conditions such as:  pneumonia 

Rank 3 | Score: 0.4640 | radiologyinfo_chest_xray | What is a chest X-ray?
The chest x-ray is the most

In [57]:
from pathlib import Path
import json

PROJECT_ROOT = Path(r"C:\Users\yuvan\Desktop\ML_projects\Chest_Xray_AI")

RAG_DIR = PROJECT_ROOT / "rag"
DOCUMENTS_DIR = RAG_DIR / "documents"
CHUNKS_DIR = RAG_DIR / "chunks"
METADATA_DIR = RAG_DIR / "metadata"
VECTOR_DIR = RAG_DIR / "vector_store"

print("RAG DIRECTORY")
print("=" * 60)

for folder in [
    RAG_DIR,
    DOCUMENTS_DIR,
    CHUNKS_DIR,
    METADATA_DIR,
    VECTOR_DIR
]:
    print(f"{folder}: {'✓' if folder.exists() else '✗'}")

print("\nDOCUMENTS")
print("=" * 60)

for path in sorted(DOCUMENTS_DIR.glob("*")):
    print(f"{path.name:<45} {path.stat().st_size:,} bytes")

print("\nSOURCE REGISTRY")
print("=" * 60)

sources_path = METADATA_DIR / "sources.json"

if sources_path.exists():
    with open(sources_path, "r", encoding="utf-8") as f:
        sources = json.load(f)

    print(f"Number of registered sources: {len(sources)}")

    for i, source in enumerate(sources, 1):
        print(f"\n[{i}]")
        print(f"ID:    {source.get('source_id')}")
        print(f"Title: {source.get('title')}")
        print(f"URL:   {source.get('url')}")
else:
    print("sources.json not found")

print("\nCHUNKS")
print("=" * 60)

chunks_path = CHUNKS_DIR / "rag_chunks.json"

if chunks_path.exists():
    with open(chunks_path, "r", encoding="utf-8") as f:
        chunks = json.load(f)

    print(f"Number of chunks: {len(chunks)}")

    for chunk in chunks:
        print(
            f"{chunk.get('chunk_id')} | "
            f"{chunk.get('source_title')} | "
            f"{chunk.get('section')}"
        )
else:
    print("rag_chunks.json not found")

RAG DIRECTORY
C:\Users\yuvan\Desktop\ML_projects\Chest_Xray_AI\rag: ✓
C:\Users\yuvan\Desktop\ML_projects\Chest_Xray_AI\rag\documents: ✓
C:\Users\yuvan\Desktop\ML_projects\Chest_Xray_AI\rag\chunks: ✓
C:\Users\yuvan\Desktop\ML_projects\Chest_Xray_AI\rag\metadata: ✓
C:\Users\yuvan\Desktop\ML_projects\Chest_Xray_AI\rag\vector_store: ✓

DOCUMENTS
chestxray14_nih_research.txt                  1,948 bytes
chestxray14_research.txt                      4,653 bytes
radiologyinfo_chest_xray.txt                  8,123 bytes
radiologyinfo_chest_xray_report.txt           5,418 bytes

SOURCE REGISTRY
Number of registered sources: 3

[1]
ID:    None
Title: NIH ChestX-ray14 Dataset
URL:   https://pmc.ncbi.nlm.nih.gov/articles/PMC9131331/

[2]
ID:    None
Title: Chest X-ray (Radiography)
URL:   https://www.radiologyinfo.org/en/info/chestrad

[3]
ID:    None
Title: How to Read Your Chest X-ray Report
URL:   https://www.radiologyinfo.org/en/info/article-chest-xray-report

CHUNKS
Number of chunks: 22
rag_0

In [58]:
from pathlib import Path

DOCUMENTS_DIR = Path(
    r"C:\Users\yuvan\Desktop\ML_projects\Chest_Xray_AI\rag\documents"
)

print("=" * 80)
print("CURRENT RAG DOCUMENT CONTENT AUDIT")
print("=" * 80)

for path in sorted(DOCUMENTS_DIR.glob("*.txt")):

    print("\n" + "=" * 80)
    print(f"FILE: {path.name}")
    print(f"SIZE: {path.stat().st_size:,} bytes")
    print("=" * 80)

    text = path.read_text(encoding="utf-8")

    print(f"Characters: {len(text):,}")
    print(f"Lines:      {len(text.splitlines()):,}")

    print("\nCONTENT PREVIEW:")
    print("-" * 80)

    # Show the first 2500 characters so we can inspect
    # what each document actually contains.
    print(text[:2500])

    if len(text) > 2500:
        print("\n... [remaining content not shown] ...")

CURRENT RAG DOCUMENT CONTENT AUDIT

FILE: chestxray14_nih_research.txt
SIZE: 1,948 bytes
Characters: 1,934
Lines:      15

CONTENT PREVIEW:
--------------------------------------------------------------------------------
Background

Computer-aided diagnosis based on chest X-ray (CXR) is an exponentially growing field of research owing to the development of deep learning, especially convolutional neural networks (CNNs). However, due to the intrinsic locality of convolution operations, CNNs cannot model long-range dependencies. Although vision transformers (ViTs) have recently been proposed to alleviate this limitation, those trained on patches cannot learn any dependencies for inter-patch pixels and thus, are insufficient for medical image detection. To address this problem, in this paper, we propose a CXR detection method which integrates CNN with a ViT for modeling patch-wise and inter-patch dependencies.

Methods

We experimented on the ChestX-ray14 dataset and followed the official 

In [59]:
import json
from pathlib import Path

PROJECT_ROOT = Path(
    r"C:\Users\yuvan\Desktop\ML_projects\Chest_Xray_AI"
)

CHUNKS_PATH = (
    PROJECT_ROOT
    / "rag"
    / "chunks"
    / "rag_chunks.json"
)

with open(CHUNKS_PATH, "r", encoding="utf-8") as f:
    chunks = json.load(f)

LABELS = [
    "Atelectasis",
    "Cardiomegaly",
    "Effusion",
    "Infiltration",
    "Mass",
    "Nodule",
    "Pneumonia",
    "Pneumothorax",
    "Consolidation",
    "Edema",
    "Emphysema",
    "Fibrosis",
    "Pleural Thickening",
    "Hernia"
]

print("=" * 90)
print("CURRENT RAG KNOWLEDGE COVERAGE")
print("=" * 90)

for label in LABELS:

    # Search both metadata and chunk text.
    matches = []

    for chunk in chunks:
        searchable_text = (
            str(chunk.get("source_title", "")) + " "
            + str(chunk.get("section", "")) + " "
            + str(chunk.get("text", ""))
        ).lower()

        if label.lower() in searchable_text:
            matches.append(chunk)

    print(f"\n{label}")
    print("-" * 90)

    if matches:
        print(f"Found in {len(matches)} chunk(s):")

        for chunk in matches:
            print(
                f"  {chunk.get('chunk_id')} | "
                f"{chunk.get('source_title')} | "
                f"{chunk.get('section')}"
            )
    else:
        print("  ❌ No direct mention found in current 22-chunk corpus")

print("\n" + "=" * 90)
print("CURRENT CORPUS SUMMARY")
print("=" * 90)

print(f"Total chunks: {len(chunks)}")

sources = {}

for chunk in chunks:
    source = chunk.get("source_title", "Unknown")
    sources[source] = sources.get(source, 0) + 1

print("\nChunks by source:")

for source, count in sources.items():
    print(f"  {source}: {count}")

CURRENT RAG KNOWLEDGE COVERAGE

Atelectasis
------------------------------------------------------------------------------------------
  ❌ No direct mention found in current 22-chunk corpus

Cardiomegaly
------------------------------------------------------------------------------------------
  ❌ No direct mention found in current 22-chunk corpus

Effusion
------------------------------------------------------------------------------------------
Found in 1 chunk(s):
  rag_020 | How to Read Your Chest X-ray Report | Findings

Infiltration
------------------------------------------------------------------------------------------
  ❌ No direct mention found in current 22-chunk corpus

Mass
------------------------------------------------------------------------------------------
  ❌ No direct mention found in current 22-chunk corpus

Nodule
------------------------------------------------------------------------------------------
Found in 1 chunk(s):
  rag_021 | How to Read Your Chest X-

In [60]:
from pathlib import Path

path = Path(
    r"C:\Users\yuvan\Desktop\ML_projects\Chest_Xray_AI\rag\documents\chestxray14_research.txt"
)

text = path.read_text(encoding="utf-8")

print("=" * 100)
print("CHESTXRAY14 RESEARCH DOCUMENT — FULL CONTENT")
print("=" * 100)

print(text)

print("\n" + "=" * 100)
print("DOCUMENT STATISTICS")
print("=" * 100)

print(f"Characters : {len(text):,}")
print(f"Words      : {len(text.split()):,}")
print(f"Lines      : {len(text.splitlines()):,}")

CHESTXRAY14 RESEARCH DOCUMENT — FULL CONTENT
Modeling long-range dependencies for weakly supervised disease classification and localization on chest X-ray

Background Computer-aided diagnosis based on chest X-ray (CXR) is an exponentially growing field of research owing to the development of deep learning, especially convolutional neural networks (CNNs). However, due to the intrinsic locality of convolution operations, CNNs cannot model long-range dependencies. Although vision transformers (ViTs) have recently been proposed to alleviate this limitation, those trained on patches cannot learn any dependencies for inter-patch pixels and thus, are insufficient for medical image detection. To address this problem, in this paper, we propose a CXR detection method which integrates CNN with a ViT for modeling patch-wise and inter-patch dependencies. Methods We experimented on the ChestX-ray14 dataset and followed the official training-test set split. Because the training data only had global a

In [61]:
import json
from pathlib import Path

path = Path(
    r"C:\Users\yuvan\Desktop\ML_projects\Chest_Xray_AI\rag\metadata\sources.json"
)

with open(path, "r", encoding="utf-8") as f:
    data = json.load(f)

print("=" * 100)
print("CURRENT sources.json")
print("=" * 100)

print(json.dumps(data, indent=4, ensure_ascii=False))

CURRENT sources.json
[
    {
        "id": "chestxray14_dataset",
        "title": "NIH ChestX-ray14 Dataset",
        "organization": "NIH Clinical Center / Research Literature",
        "url": "https://pmc.ncbi.nlm.nih.gov/articles/PMC9131331/",
        "type": "dataset_research",
        "purpose": "Dataset characteristics, 14 findings, multi-label classification, and label generation"
    },
    {
        "id": "radiologyinfo_chest_xray",
        "title": "Chest X-ray (Radiography)",
        "organization": "RadiologyInfo",
        "url": "https://www.radiologyinfo.org/en/info/chestrad",
        "type": "medical_reference",
        "purpose": "Chest X-ray overview, uses, procedure, interpretation context, and limitations"
    },
    {
        "id": "radiologyinfo_chest_xray_report",
        "title": "How to Read Your Chest X-ray Report",
        "organization": "RadiologyInfo",
        "url": "https://www.radiologyinfo.org/en/info/article-chest-xray-report",
        "type": "medica

In [62]:
from pathlib import Path
import json

RAG_DIR = Path("rag")

print("RAG directory:", RAG_DIR.resolve())
print()

for folder in [
    "documents",
    "chunks",
    "vector_store",
    "metadata"
]:
    path = RAG_DIR / folder

    print(f"{folder}/")

    if path.exists():
        files = list(path.iterdir())

        for file in files:
            print(f"   └── {file.name}")
    else:
        print("   └── [NOT FOUND]")

    print()

# Inspect source registry
sources_file = RAG_DIR / "metadata" / "sources.json"

if sources_file.exists():
    with open(sources_file, "r", encoding="utf-8") as f:
        sources = json.load(f)

    print("Number of registered sources:", len(sources))

    for source in sources:
        print(
            f"- {source.get('source_id', 'N/A')} | "
            f"{source.get('title', 'N/A')}"
        )
else:
    print("sources.json not found")


# Inspect current chunks
chunks_file = RAG_DIR / "chunks" / "rag_chunks.json"

if chunks_file.exists():
    with open(chunks_file, "r", encoding="utf-8") as f:
        chunks = json.load(f)

    print()
    print("Number of current chunks:", len(chunks))

    if chunks:
        print("\nFirst chunk:")
        print(chunks[0])
else:
    print("rag_chunks.json not found")

RAG directory: C:\Users\yuvan\Desktop\ML_projects\Chest_Xray_AI\notebooks\rag

documents/
   └── [NOT FOUND]

chunks/
   └── [NOT FOUND]

vector_store/
   └── [NOT FOUND]

metadata/
   └── sources.json

Number of registered sources: 2
- N/A | How to Read Your Chest X-ray Report
- N/A | ChestX-ray14 Dataset and Multi-Label Chest X-ray Classification
rag_chunks.json not found


In [63]:
from pathlib import Path

cwd = Path.cwd()

print("Current working directory:")
print(cwd)
print("\n" + "=" * 70)

print("Searching nearby project directories for RAG files...\n")

search_roots = [
    cwd,
    cwd.parent,
    cwd.parent.parent,
]

found = set()

for root in search_roots:
    print(f"Checking: {root}")

    for pattern in [
        "rag_chunks.json",
        "rag.index",
        "sources.json",
        "chestxray14_nih_research.txt",
        "radiologyinfo_chest_xray.txt",
        "radiologyinfo_chest_xray_report.txt"
    ]:
        matches = list(root.rglob(pattern))

        for match in matches:
            if match.is_file():
                found.add(match.resolve())
                print(f"   ✓ {match.resolve()}")

    print()

print("=" * 70)

if found:
    print(f"Found {len(found)} RAG-related file(s).")
else:
    print("No existing RAG files were found in the searched directories.")

Current working directory:
c:\Users\yuvan\Desktop\ML_projects\Chest_Xray_AI\notebooks

Searching nearby project directories for RAG files...

Checking: c:\Users\yuvan\Desktop\ML_projects\Chest_Xray_AI\notebooks
   ✓ C:\Users\yuvan\Desktop\ML_projects\Chest_Xray_AI\notebooks\rag\metadata\sources.json

Checking: c:\Users\yuvan\Desktop\ML_projects\Chest_Xray_AI
   ✓ C:\Users\yuvan\Desktop\ML_projects\Chest_Xray_AI\rag\chunks\rag_chunks.json
   ✓ C:\Users\yuvan\Desktop\ML_projects\Chest_Xray_AI\rag\vector_store\rag.index
   ✓ C:\Users\yuvan\Desktop\ML_projects\Chest_Xray_AI\notebooks\rag\metadata\sources.json
   ✓ C:\Users\yuvan\Desktop\ML_projects\Chest_Xray_AI\rag\metadata\sources.json
   ✓ C:\Users\yuvan\Desktop\ML_projects\Chest_Xray_AI\rag\documents\chestxray14_nih_research.txt
   ✓ C:\Users\yuvan\Desktop\ML_projects\Chest_Xray_AI\rag\documents\radiologyinfo_chest_xray.txt
   ✓ C:\Users\yuvan\Desktop\ML_projects\Chest_Xray_AI\rag\documents\radiologyinfo_chest_xray_report.txt

Checking

In [64]:
from pathlib import Path
import json

PROJECT_DIR = Path.cwd().parent
RAG_DIR = PROJECT_DIR / "rag"

print("Project directory:")
print(PROJECT_DIR)

print("\nRAG directory:")
print(RAG_DIR)

print("\n" + "=" * 70)
print("CURRENT RAG FILES")
print("=" * 70)

for path in RAG_DIR.rglob("*"):
    if path.is_file():
        print(path.relative_to(PROJECT_DIR))


# --------------------------------------------------
# Source registry
# --------------------------------------------------

sources_file = RAG_DIR / "metadata" / "sources.json"

print("\n" + "=" * 70)
print("SOURCE REGISTRY")
print("=" * 70)

with open(sources_file, "r", encoding="utf-8") as f:
    sources = json.load(f)

print("Number of registered sources:", len(sources))

for i, source in enumerate(sources, start=1):
    print(f"\nSource {i}")
    print("-" * 40)
    print(json.dumps(source, indent=2, ensure_ascii=False))


# --------------------------------------------------
# Current chunks
# --------------------------------------------------

chunks_file = RAG_DIR / "chunks" / "rag_chunks.json"

print("\n" + "=" * 70)
print("CURRENT CHUNKS")
print("=" * 70)

with open(chunks_file, "r", encoding="utf-8") as f:
    chunks = json.load(f)

print("Number of chunks:", len(chunks))

print("\nFirst 3 chunks:")
for chunk in chunks[:3]:
    print("-" * 70)
    print(json.dumps(chunk, indent=2, ensure_ascii=False))


# --------------------------------------------------
# Document sizes
# --------------------------------------------------

print("\n" + "=" * 70)
print("DOCUMENT SIZES")
print("=" * 70)

documents_dir = RAG_DIR / "documents"

for file in documents_dir.glob("*.txt"):
    text = file.read_text(encoding="utf-8")

    print(
        f"{file.name:<45} "
        f"{len(text):>8,} characters"
    )

Project directory:
c:\Users\yuvan\Desktop\ML_projects\Chest_Xray_AI

RAG directory:
c:\Users\yuvan\Desktop\ML_projects\Chest_Xray_AI\rag

CURRENT RAG FILES
rag\chunks\chestxray14_chunks.json
rag\chunks\rag_chunks.json
rag\documents\chestxray14_nih_research.txt
rag\documents\chestxray14_research.txt
rag\documents\radiologyinfo_chest_xray.txt
rag\documents\radiologyinfo_chest_xray_report.txt
rag\metadata\chestxray14_metadata.json
rag\metadata\rag_metadata.json
rag\metadata\sources.json
rag\vector_store\chestxray14.index
rag\vector_store\rag.index

SOURCE REGISTRY
Number of registered sources: 3

Source 1
----------------------------------------
{
  "id": "chestxray14_dataset",
  "title": "NIH ChestX-ray14 Dataset",
  "organization": "NIH Clinical Center / Research Literature",
  "url": "https://pmc.ncbi.nlm.nih.gov/articles/PMC9131331/",
  "type": "dataset_research",
  "purpose": "Dataset characteristics, 14 findings, multi-label classification, and label generation"
}

Source 2
--------

In [65]:
from pathlib import Path
import json

RAG_DIR = Path.cwd().parent / "rag"

rag_chunks_file = RAG_DIR / "chunks" / "rag_chunks.json"
cxr_chunks_file = RAG_DIR / "chunks" / "chestxray14_chunks.json"

with open(rag_chunks_file, "r", encoding="utf-8") as f:
    rag_chunks = json.load(f)

with open(cxr_chunks_file, "r", encoding="utf-8") as f:
    cxr_chunks = json.load(f)


print("=" * 70)
print("CHUNK SET COMPARISON")
print("=" * 70)

print(f"rag_chunks.json          : {len(rag_chunks)} chunks")
print(f"chestxray14_chunks.json  : {len(cxr_chunks)} chunks")


# --------------------------------------------------
# Helper function
# --------------------------------------------------

def summarize_chunks(name, chunks):

    print("\n" + "=" * 70)
    print(name)
    print("=" * 70)

    source_counts = {}
    section_counts = {}

    for chunk in chunks:

        source = chunk.get("source_id", "UNKNOWN")
        section = chunk.get("section", "UNKNOWN")

        source_counts[source] = source_counts.get(source, 0) + 1
        section_counts[section] = section_counts.get(section, 0) + 1

    print("\nSources:")
    for source, count in source_counts.items():
        print(f"  {source:<40} {count}")

    print("\nSections:")
    for section, count in section_counts.items():
        print(f"  {section:<40} {count}")


# --------------------------------------------------
# Summaries
# --------------------------------------------------

summarize_chunks("RAG CHUNKS", rag_chunks)

summarize_chunks("CHESTXRAY14 CHUNKS", cxr_chunks)


# --------------------------------------------------
# Show all ChestX-ray14 chunks
# --------------------------------------------------

print("\n" + "=" * 70)
print("CHESTX-RAY14 CHUNK CONTENT")
print("=" * 70)

for i, chunk in enumerate(cxr_chunks, start=1):

    print(f"\n[{i}] {chunk.get('chunk_id', 'N/A')}")
    print(f"Section: {chunk.get('section', 'N/A')}")
    print("-" * 60)
    print(chunk.get("text", ""))

CHUNK SET COMPARISON
rag_chunks.json          : 22 chunks
chestxray14_chunks.json  : 4 chunks

RAG CHUNKS

Sources:
  chestxray14_dataset                      4
  radiologyinfo_chest_xray                 10
  radiologyinfo_chest_xray_report          8

Sections:
  Background                               1
  Methods                                  1
  Results                                  1
  Conclusions                              1
  What is a chest X-ray?                   1
  What are some common uses of the procedure? 1
  How should I prepare?                    1
  What does the equipment look like?       1
  How does the procedure work?             1
  How is the procedure performed?          1
  What will I experience during and after the procedure? 1
  Who interprets the results and how do I get them? 1
  What are the benefits vs. risks?         1
  What are the limitations of chest x-ray? 1
  What is Chest X-ray commonly used for?   1
  Type of exam                      

In [66]:
from pathlib import Path
import json

PROJECT_DIR = Path.cwd().parent
RAG_DIR = PROJECT_DIR / "rag"

source_plan = [
    {
        "id": "atelectasis_medlineplus",
        "title": "Atelectasis",
        "organization": "MedlinePlus / NIH",
        "url": "https://medlineplus.gov/ency/article/000065.htm",
        "type": "medical_reference",
        "finding": "Atelectasis",
        "purpose": "Definition, causes, evaluation, and chest imaging context"
    },
    {
        "id": "cardiomegaly_ncbi",
        "title": "Cardiomegaly",
        "organization": "NCBI Bookshelf / StatPearls",
        "url": "https://www.ncbi.nlm.nih.gov/books/NBK542296/",
        "type": "medical_reference",
        "finding": "Cardiomegaly",
        "purpose": "Definition, imaging evaluation, and chest radiograph context"
    },
    {
        "id": "pleural_effusion_medlineplus",
        "title": "Pleural Effusion",
        "organization": "MedlinePlus / NIH",
        "url": "https://medlineplus.gov/ency/article/000086.htm",
        "type": "medical_reference",
        "finding": "Effusion",
        "purpose": "Definition, causes, and clinical context"
    },
    {
        "id": "lung_nodules_radiologyinfo",
        "title": "Lung Nodules",
        "organization": "RadiologyInfo / ACR / RSNA",
        "url": "https://www.radiologyinfo.org/en/info/lung-nodules",
        "type": "medical_reference",
        "finding": "Nodule",
        "purpose": "Definition, imaging context, and evaluation"
    },
    {
        "id": "pneumonia_radiologyinfo",
        "title": "Pneumonia",
        "organization": "RadiologyInfo / ACR / RSNA",
        "url": "https://www.radiologyinfo.org/en/info/pneumonia",
        "type": "medical_reference",
        "finding": "Pneumonia",
        "purpose": "Chest X-ray role and imaging context"
    },
    {
        "id": "pneumothorax_ncbi",
        "title": "Diseases of the Pleura and Chest Wall",
        "organization": "NCBI Bookshelf",
        "url": "https://www.ncbi.nlm.nih.gov/books/NBK621648/",
        "type": "medical_reference",
        "finding": "Pneumothorax",
        "purpose": "Chest radiograph appearance and imaging limitations"
    },
    {
        "id": "lung_consolidation_ncbi",
        "title": "Lung Consolidation",
        "organization": "NCBI MedGen",
        "url": "https://www.ncbi.nlm.nih.gov/medgen/636648",
        "type": "medical_reference",
        "finding": "Consolidation",
        "purpose": "Radiologic definition of lung consolidation"
    },
    {
        "id": "pulmonary_edema_medlineplus",
        "title": "Pulmonary Edema",
        "organization": "MedlinePlus / NIH",
        "url": "https://medlineplus.gov/ency/article/000140.htm",
        "type": "medical_reference",
        "finding": "Edema",
        "purpose": "Definition and clinical context"
    },
    {
        "id": "emphysema_radiologyinfo",
        "title": "COPD",
        "organization": "RadiologyInfo / ACR / RSNA",
        "url": "https://www.radiologyinfo.org/en/info/copd",
        "type": "medical_reference",
        "finding": "Emphysema",
        "purpose": "Emphysema and chest imaging context"
    },
    {
        "id": "pulmonary_fibrosis_nhlbi",
        "title": "Pulmonary Fibrosis — Diagnosis",
        "organization": "NHLBI / NIH",
        "url": "https://www.nhlbi.nih.gov/health/idiopathic-pulmonary-fibrosis/diagnosis",
        "type": "medical_reference",
        "finding": "Fibrosis",
        "purpose": "Pulmonary fibrosis and role/limitations of chest X-ray"
    },
    {
        "id": "pleural_thickening_ncbi",
        "title": "Diseases of the Pleura and Chest Wall",
        "organization": "NCBI Bookshelf",
        "url": "https://www.ncbi.nlm.nih.gov/books/NBK621648/",
        "type": "medical_reference",
        "finding": "Pleural Thickening",
        "purpose": "Radiographic appearance and differentiation from pleural fluid"
    },
    {
        "id": "pulmonary_mass_ncbi",
        "title": "Pulmonary Mass",
        "organization": "NCBI MedGen",
        "url": "https://www.ncbi.nlm.nih.gov/medgen/57697",
        "type": "medical_reference",
        "finding": "Mass",
        "purpose": "Definition of pulmonary mass on chest imaging"
    },
    {
        "id": "diaphragmatic_hernia_medlineplus",
        "title": "Diaphragmatic Hernia",
        "organization": "MedlinePlus / NIH",
        "url": "https://medlineplus.gov/ency/article/001135.htm",
        "type": "medical_reference",
        "finding": "Hernia",
        "purpose": "Relevant chest/diaphragmatic hernia context"
    },
    {
        "id": "chestxray14_dataset_context",
        "title": "Modeling long-range dependencies for weakly supervised disease classification and localization on chest X-ray",
        "organization": "NCBI / PMC",
        "url": "https://pmc.ncbi.nlm.nih.gov/articles/PMC9131331/",
        "type": "dataset_research",
        "finding": "Infiltration",
        "purpose": "ChestX-ray14 dataset and report-derived label context"
    }
]

source_plan_file = RAG_DIR / "metadata" / "v2_source_plan.json"

with open(source_plan_file, "w", encoding="utf-8") as f:
    json.dump(source_plan, f, indent=2, ensure_ascii=False)

print("V2 source plan created:")
print(source_plan_file)

print()
print("Total planned sources:", len(source_plan))

print("\nFinding coverage:")
for source in source_plan:
    print(
        f"{source['finding']:<22} "
        f"| {source['organization']}"
    )

V2 source plan created:
c:\Users\yuvan\Desktop\ML_projects\Chest_Xray_AI\rag\metadata\v2_source_plan.json

Total planned sources: 14

Finding coverage:
Atelectasis            | MedlinePlus / NIH
Cardiomegaly           | NCBI Bookshelf / StatPearls
Effusion               | MedlinePlus / NIH
Nodule                 | RadiologyInfo / ACR / RSNA
Pneumonia              | RadiologyInfo / ACR / RSNA
Pneumothorax           | NCBI Bookshelf
Consolidation          | NCBI MedGen
Edema                  | MedlinePlus / NIH
Emphysema              | RadiologyInfo / ACR / RSNA
Fibrosis               | NHLBI / NIH
Pleural Thickening     | NCBI Bookshelf
Mass                   | NCBI MedGen
Hernia                 | MedlinePlus / NIH
Infiltration           | NCBI / PMC


In [67]:
from pathlib import Path
import json

RAG_DIR = Path.cwd().parent / "rag"

source_plan_file = RAG_DIR / "metadata" / "v2_source_plan.json"

with open(source_plan_file, "r", encoding="utf-8") as f:
    source_plan = json.load(f)

print("=" * 70)
print("V2 SOURCE PLAN")
print("=" * 70)

for i, source in enumerate(source_plan, start=1):
    print(f"\n{i}. {source['finding']}")
    print(f"   Title       : {source['title']}")
    print(f"   Organization: {source['organization']}")
    print(f"   Type        : {source['type']}")
    print(f"   URL         : {source['url']}")
    print(f"   Purpose     : {source['purpose']}")

print("\n" + "=" * 70)
print(f"TOTAL SOURCES: {len(source_plan)}")
print("=" * 70)

V2 SOURCE PLAN

1. Atelectasis
   Title       : Atelectasis
   Organization: MedlinePlus / NIH
   Type        : medical_reference
   URL         : https://medlineplus.gov/ency/article/000065.htm
   Purpose     : Definition, causes, evaluation, and chest imaging context

2. Cardiomegaly
   Title       : Cardiomegaly
   Organization: NCBI Bookshelf / StatPearls
   Type        : medical_reference
   URL         : https://www.ncbi.nlm.nih.gov/books/NBK542296/
   Purpose     : Definition, imaging evaluation, and chest radiograph context

3. Effusion
   Title       : Pleural Effusion
   Organization: MedlinePlus / NIH
   Type        : medical_reference
   URL         : https://medlineplus.gov/ency/article/000086.htm
   Purpose     : Definition, causes, and clinical context

4. Nodule
   Title       : Lung Nodules
   Organization: RadiologyInfo / ACR / RSNA
   Type        : medical_reference
   URL         : https://www.radiologyinfo.org/en/info/lung-nodules
   Purpose     : Definition, imagi

In [68]:
import json
import requests
from pathlib import Path

RAG_DIR = Path.cwd().parent / "rag"
source_plan_file = RAG_DIR / "metadata" / "v2_source_plan.json"

with open(source_plan_file, "r", encoding="utf-8") as f:
    source_plan = json.load(f)

headers = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
                  "AppleWebKit/537.36 Chrome/154.0 Safari/537.36"
}

print("=" * 80)
print("V2 SOURCE URL VERIFICATION")
print("=" * 80)

for i, source in enumerate(source_plan, start=1):
    url = source["url"]

    try:
        response = requests.get(
            url,
            headers=headers,
            timeout=20,
            allow_redirects=True
        )

        print(f"\n{i}. {source['finding']}")
        print(f"   Status      : {response.status_code}")
        print(f"   Final URL   : {response.url}")
        print(f"   Content-Type: {response.headers.get('Content-Type', 'Unknown')}")

    except requests.RequestException as e:
        print(f"\n{i}. {source['finding']}")
        print(f"   ERROR       : {type(e).__name__}")
        print(f"   Message     : {e}")

print("\n" + "=" * 80)
print("URL VERIFICATION COMPLETE")
print("=" * 80)

V2 SOURCE URL VERIFICATION

1. Atelectasis
   Status      : 200
   Final URL   : https://medlineplus.gov/ency/article/000065.htm
   Content-Type: text/html

2. Cardiomegaly
   Status      : 200
   Final URL   : https://www.ncbi.nlm.nih.gov/books/NBK542296/
   Content-Type: text/html; charset=utf-8

3. Effusion
   Status      : 200
   Final URL   : https://medlineplus.gov/ency/article/000086.htm
   Content-Type: text/html

4. Nodule
   Status      : 200
   Final URL   : https://www.radiologyinfo.org/en/info/lung-nodules
   Content-Type: text/html; charset=utf-8

5. Pneumonia
   Status      : 200
   Final URL   : https://www.radiologyinfo.org/en/info/pneumonia
   Content-Type: text/html; charset=utf-8

6. Pneumothorax
   Status      : 200
   Final URL   : https://www.ncbi.nlm.nih.gov/books/NBK621648/
   Content-Type: text/html; charset=utf-8

7. Consolidation
   Status      : 200
   Final URL   : https://www.ncbi.nlm.nih.gov/medgen/636648
   Content-Type: text/html; charset=UTF-8

8. Ede

In [69]:
from pathlib import Path

RAG_DIR = Path.cwd().parent / "rag"

V2_DIR = RAG_DIR / "v2"

V2_DOCUMENTS_DIR = V2_DIR / "documents"
V2_CHUNKS_DIR = V2_DIR / "chunks"
V2_METADATA_DIR = V2_DIR / "metadata"
V2_VECTOR_DIR = V2_DIR / "vector_store"

for directory in [
    V2_DOCUMENTS_DIR,
    V2_CHUNKS_DIR,
    V2_METADATA_DIR,
    V2_VECTOR_DIR
]:
    directory.mkdir(parents=True, exist_ok=True)

print("=" * 70)
print("V2 RAG WORKSPACE")
print("=" * 70)

print(f"Root      : {V2_DIR}")
print(f"Documents : {V2_DOCUMENTS_DIR}")
print(f"Chunks    : {V2_CHUNKS_DIR}")
print(f"Metadata  : {V2_METADATA_DIR}")
print(f"Vectors   : {V2_VECTOR_DIR}")

print("\nExisting V1 corpus remains untouched.")
print("V2 workspace created successfully.")

V2 RAG WORKSPACE
Root      : c:\Users\yuvan\Desktop\ML_projects\Chest_Xray_AI\rag\v2
Documents : c:\Users\yuvan\Desktop\ML_projects\Chest_Xray_AI\rag\v2\documents
Chunks    : c:\Users\yuvan\Desktop\ML_projects\Chest_Xray_AI\rag\v2\chunks
Metadata  : c:\Users\yuvan\Desktop\ML_projects\Chest_Xray_AI\rag\v2\metadata
Vectors   : c:\Users\yuvan\Desktop\ML_projects\Chest_Xray_AI\rag\v2\vector_store

Existing V1 corpus remains untouched.
V2 workspace created successfully.


In [70]:
import requests
from bs4 import BeautifulSoup
from pathlib import Path

RAG_DIR = Path.cwd().parent / "rag"
V2_DOCUMENTS_DIR = RAG_DIR / "v2" / "documents"

url = "https://medlineplus.gov/ency/article/000065.htm"

headers = {
    "User-Agent": (
        "Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
        "AppleWebKit/537.36 (KHTML, like Gecko) "
        "Chrome/154.0 Safari/537.36"
    )
}

response = requests.get(
    url,
    headers=headers,
    timeout=30
)

response.raise_for_status()

soup = BeautifulSoup(response.text, "html.parser")

# Remove elements that are not useful for RAG
for element in soup([
    "script",
    "style",
    "nav",
    "header",
    "footer",
    "noscript"
]):
    element.decompose()

text = soup.get_text(separator="\n")

# Clean excessive whitespace
lines = [
    line.strip()
    for line in text.splitlines()
    if line.strip()
]

clean_text = "\n".join(lines)

output_file = V2_DOCUMENTS_DIR / "atelectasis_medlineplus.txt"

with open(output_file, "w", encoding="utf-8") as f:
    f.write(clean_text)

print("=" * 70)
print("ATELECTASIS SOURCE INGESTION")
print("=" * 70)

print(f"Status       : {response.status_code}")
print(f"Source       : MedlinePlus / NIH")
print(f"Characters   : {len(clean_text):,}")
print(f"Lines        : {len(lines):,}")
print(f"Saved to     : {output_file}")

print("\n" + "-" * 70)
print("TEXT PREVIEW")
print("-" * 70)

print(clean_text[:4000])

ATELECTASIS SOURCE INGESTION
Status       : 200
Source       : MedlinePlus / NIH
Characters   : 5,376
Lines        : 121
Saved to     : c:\Users\yuvan\Desktop\ML_projects\Chest_Xray_AI\rag\v2\documents\atelectasis_medlineplus.txt

----------------------------------------------------------------------
TEXT PREVIEW
----------------------------------------------------------------------
Atelectasis: MedlinePlus Medical Encyclopedia
Skip navigation
Official websites use .gov
A
.gov
website belongs to an official government
organization in the United States.
Secure .gov websites use HTTPS
A
lock
(
Lock
Locked padlock icon
) or
https://
means youâve safely connected to
the .gov website. Share sensitive information only on official,
secure websites.
You Are Here:
Home
→
Medical Encyclopedia
→
Atelectasis
URL of this page: //medlineplus.gov/ency/article/000065.htm
Atelectasis
Atelectasis is the collapse of part or, much less commonly, all of a lung.
Causes
Atelectasis is caused by a blockage 

In [71]:
from pathlib import Path

RAG_DIR = Path.cwd().parent / "rag"
V2_DOCUMENTS_DIR = RAG_DIR / "v2" / "documents"

raw_file = V2_DOCUMENTS_DIR / "atelectasis_medlineplus.txt"
clean_file = V2_DOCUMENTS_DIR / "atelectasis_medlineplus_clean.txt"

with open(raw_file, "r", encoding="utf-8") as f:
    text = f.read()

# Fix common UTF-8/mojibake artifacts
text = text.replace("â", "'")
text = text.replace("â", "-")
text = text.replace("â", '"')
text = text.replace("â", '"')

# Start from the actual article title instead of webpage navigation
title_marker = "Atelectasis\n"

if title_marker in text:
    text = text[text.index(title_marker):]

# Remove webpage/navigation material that appears before the medical content
remove_lines = {
    "Skip navigation",
    "Official websites use .gov",
    "A .gov website belongs to an official government organization in the United States.",
    "Secure .gov websites use HTTPS",
    "A lock ( Lock Locked padlock icon ) or https:// means you've safely connected to the .gov website.",
    "Share sensitive information only on official, secure websites.",
    "You Are Here: Home → Medical Encyclopedia → Atelectasis",
    "URL of this page: //medlineplus.gov/ency/article/000065.htm",
}

lines = []

for line in text.splitlines():
    line = line.strip()

    if not line:
        continue

    if line in remove_lines:
        continue

    lines.append(line)

clean_text = "\n".join(lines)

with open(clean_file, "w", encoding="utf-8") as f:
    f.write(clean_text)

print("=" * 70)
print("CLEANED ATELECTASIS DOCUMENT")
print("=" * 70)

print(f"Raw characters    : {len(text):,}")
print(f"Clean characters  : {len(clean_text):,}")
print(f"Saved to          : {clean_file}")

print("\n" + "-" * 70)
print("CLEANED PREVIEW")
print("-" * 70)

print(clean_text[:4000])

CLEANED ATELECTASIS DOCUMENT
Raw characters    : 4,964
Clean characters  : 4,904
Saved to          : c:\Users\yuvan\Desktop\ML_projects\Chest_Xray_AI\rag\v2\documents\atelectasis_medlineplus_clean.txt

----------------------------------------------------------------------
CLEANED PREVIEW
----------------------------------------------------------------------
Atelectasis
Atelectasis
Atelectasis is the collapse of part or, much less commonly, all of a lung.
Causes
Atelectasis is caused by a blockage of the air passages (bronchus or bronchioles) or by pressure on the outside of the lung. Atelectasis is not the same as another type of collapsed lung called
pneumothorax
, which occurs when air escapes from the lung. The air then fills the space outside of the lung, between the lung and chest wall.
Common situations in people with atelectasis include:
There is fluid or a tumor that compresses the lung.
There is a blockage in the airway not allowing air into the air sacs of the lung.
The air s

In [72]:
from pathlib import Path

RAG_DIR = Path.cwd().parent / "rag"
V2_DOCUMENTS_DIR = RAG_DIR / "v2" / "documents"

raw_file = V2_DOCUMENTS_DIR / "atelectasis_medlineplus.txt"
clean_file = V2_DOCUMENTS_DIR / "atelectasis_medlineplus_clean.txt"

with open(raw_file, "r", encoding="utf-8") as f:
    text = f.read()

# Fix common encoding artifacts
encoding_fixes = {
    "â": "'",
    "â": "-",
    "â": '"',
    "â": '"',
    "Ã©": "é",
    "Ã¨": "è",
    "Ã": "à",
}

for bad, good in encoding_fixes.items():
    text = text.replace(bad, good)

# Remove webpage navigation before the article
title_marker = "Atelectasis\n"

if title_marker in text:
    text = text[text.index(title_marker):]

# Remove everything from References onward.
# References are bibliographic metadata, not semantic medical evidence.
reference_markers = [
    "\nReferences\n",
    "\nReferences",
]

for marker in reference_markers:
    if marker in text:
        text = text.split(marker)[0]
        break

# Clean line-by-line
lines = []

for line in text.splitlines():
    line = line.strip()

    if not line:
        continue

    # Remove duplicate title
    if line == "Atelectasis" and len(lines) > 0 and lines[-1] == "Atelectasis":
        continue

    lines.append(line)

clean_text = "\n".join(lines)

with open(clean_file, "w", encoding="utf-8") as f:
    f.write(clean_text)

print("=" * 70)
print("FINAL ATELECTASIS DOCUMENT")
print("=" * 70)

print(f"Characters : {len(clean_text):,}")
print(f"Lines      : {len(lines):,}")
print(f"Saved to   : {clean_file}")

print("\n" + "-" * 70)
print("FINAL PREVIEW")
print("-" * 70)

print(clean_text[:4000])

FINAL ATELECTASIS DOCUMENT
Characters : 3,876
Lines      : 83
Saved to   : c:\Users\yuvan\Desktop\ML_projects\Chest_Xray_AI\rag\v2\documents\atelectasis_medlineplus_clean.txt

----------------------------------------------------------------------
FINAL PREVIEW
----------------------------------------------------------------------
Atelectasis
URL of this page: //medlineplus.gov/ency/article/000065.htm
Atelectasis
Atelectasis is the collapse of part or, much less commonly, all of a lung.
Causes
Atelectasis is caused by a blockage of the air passages (bronchus or bronchioles) or by pressure on the outside of the lung. Atelectasis is not the same as another type of collapsed lung called
pneumothorax
, which occurs when air escapes from the lung. The air then fills the space outside of the lung, between the lung and chest wall.
Common situations in people with atelectasis include:
There is fluid or a tumor that compresses the lung.
There is a blockage in the airway not allowing air into the

In [73]:
from pathlib import Path

RAG_DIR = Path.cwd().parent / "rag"
V2_DOCUMENTS_DIR = RAG_DIR / "v2" / "documents"

clean_file = V2_DOCUMENTS_DIR / "atelectasis_medlineplus_clean.txt"

with open(clean_file, "r", encoding="utf-8") as f:
    text = f.read()

# Remove page URL metadata
text = text.replace(
    "URL of this page: //medlineplus.gov/ency/article/000065.htm",
    ""
)

# Remove image/navigation section
if "\nImages\n" in text:
    text = text.split("\nImages\n")[0]

# Clean whitespace
lines = [
    line.strip()
    for line in text.splitlines()
    if line.strip()
]

clean_text = "\n".join(lines)

with open(clean_file, "w", encoding="utf-8") as f:
    f.write(clean_text)

print("=" * 70)
print("ATELECTASIS SOURCE — FINAL CLEANUP")
print("=" * 70)

print(f"Characters : {len(clean_text):,}")
print(f"Lines      : {len(lines):,}")
print(f"Saved to   : {clean_file}")

print("\n" + "-" * 70)
print("FINAL TEXT")
print("-" * 70)

print(clean_text)

ATELECTASIS SOURCE — FINAL CLEANUP
Characters : 3,771
Lines      : 78
Saved to   : c:\Users\yuvan\Desktop\ML_projects\Chest_Xray_AI\rag\v2\documents\atelectasis_medlineplus_clean.txt

----------------------------------------------------------------------
FINAL TEXT
----------------------------------------------------------------------
Atelectasis
Atelectasis
Atelectasis is the collapse of part or, much less commonly, all of a lung.
Causes
Atelectasis is caused by a blockage of the air passages (bronchus or bronchioles) or by pressure on the outside of the lung. Atelectasis is not the same as another type of collapsed lung called
pneumothorax
, which occurs when air escapes from the lung. The air then fills the space outside of the lung, between the lung and chest wall.
Common situations in people with atelectasis include:
There is fluid or a tumor that compresses the lung.
There is a blockage in the airway not allowing air into the air sacs of the lung.
The air sacs collapse when the l

In [74]:
from pathlib import Path

RAG_DIR = Path.cwd().parent / "rag"
V2_DOCUMENTS_DIR = RAG_DIR / "v2" / "documents"

clean_file = V2_DOCUMENTS_DIR / "atelectasis_medlineplus_clean.txt"

with open(clean_file, "r", encoding="utf-8") as f:
    text = f.read()

# Remove consecutive duplicate title
text = text.replace(
    "Atelectasis\nAtelectasis\nAtelectasis is",
    "Atelectasis\nAtelectasis is"
)

with open(clean_file, "w", encoding="utf-8") as f:
    f.write(text)

# Detect major sections
section_names = [
    "Causes",
    "Symptoms",
    "Exams and Tests",
    "Treatment",
    "Outlook (Prognosis)",
    "Possible Complications",
    "When to Contact a Medical Professional",
    "Prevention",
    "Alternative Names"
]

print("=" * 70)
print("ATELECTASIS DOCUMENT — SECTION VERIFICATION")
print("=" * 70)

for section in section_names:
    status = "✓" if section in text else "✗"
    print(f"{status} {section}")

print("\n" + "-" * 70)
print("FIRST 500 CHARACTERS")
print("-" * 70)

print(text[:500])

print("\n" + "=" * 70)
print(f"FINAL CHARACTER COUNT: {len(text):,}")
print("=" * 70)

ATELECTASIS DOCUMENT — SECTION VERIFICATION
✓ Causes
✓ Symptoms
✓ Exams and Tests
✓ Treatment
✓ Outlook (Prognosis)
✓ Possible Complications
✓ When to Contact a Medical Professional
✓ Prevention
✓ Alternative Names

----------------------------------------------------------------------
FIRST 500 CHARACTERS
----------------------------------------------------------------------
Atelectasis
Atelectasis is the collapse of part or, much less commonly, all of a lung.
Causes
Atelectasis is caused by a blockage of the air passages (bronchus or bronchioles) or by pressure on the outside of the lung. Atelectasis is not the same as another type of collapsed lung called
pneumothorax
, which occurs when air escapes from the lung. The air then fills the space outside of the lung, between the lung and chest wall.
Common situations in people with atelectasis include:
There is fluid 

FINAL CHARACTER COUNT: 3,759


In [75]:
import json
from pathlib import Path

RAG_DIR = Path.cwd().parent / "rag"

V2_DOCUMENTS_DIR = RAG_DIR / "v2" / "documents"
V2_CHUNKS_DIR = RAG_DIR / "v2" / "chunks"

source_file = V2_DOCUMENTS_DIR / "atelectasis_medlineplus_clean.txt"
chunk_file = V2_CHUNKS_DIR / "atelectasis_chunks.json"

with open(source_file, "r", encoding="utf-8") as f:
    text = f.read()

# Sections present in the MedlinePlus document
section_names = [
    "Causes",
    "Symptoms",
    "Exams and Tests",
    "Treatment",
    "Outlook (Prognosis)",
    "Possible Complications",
    "When to Contact a Medical Professional",
    "Prevention",
    "Alternative Names"
]

# Locate section boundaries
positions = []

for section in section_names:
    position = text.find(section)

    if position != -1:
        positions.append((position, section))

positions.sort()

chunks = []

# Definition / introduction before first section
if positions:
    definition_text = text[:positions[0][0]].strip()

    if definition_text:
        chunks.append({
            "chunk_id": "atelectasis_001",
            "source_id": "atelectasis_medlineplus",
            "source_title": "Atelectasis",
            "organization": "MedlinePlus / NIH",
            "source_type": "medical_reference",
            "finding": "Atelectasis",
            "section": "Definition",
            "text": definition_text
        })

# Section chunks
for i, (start, section) in enumerate(positions):

    end = positions[i + 1][0] if i + 1 < len(positions) else len(text)

    section_text = text[start:end].strip()

    chunks.append({
        "chunk_id": f"atelectasis_{i + 2:03d}",
        "source_id": "atelectasis_medlineplus",
        "source_title": "Atelectasis",
        "organization": "MedlinePlus / NIH",
        "source_type": "medical_reference",
        "finding": "Atelectasis",
        "section": section,
        "text": section_text
    })

with open(chunk_file, "w", encoding="utf-8") as f:
    json.dump(chunks, f, indent=2, ensure_ascii=False)

print("=" * 70)
print("ATELECTASIS SEMANTIC CHUNKING")
print("=" * 70)

print(f"Source file : {source_file}")
print(f"Chunk file  : {chunk_file}")
print(f"Total chunks: {len(chunks)}")

print("\n" + "-" * 70)

for chunk in chunks:
    print(
        f"{chunk['chunk_id']} | "
        f"{chunk['section']} | "
        f"{len(chunk['text']):,} chars"
    )

ATELECTASIS SEMANTIC CHUNKING
Source file : c:\Users\yuvan\Desktop\ML_projects\Chest_Xray_AI\rag\v2\documents\atelectasis_medlineplus_clean.txt
Chunk file  : c:\Users\yuvan\Desktop\ML_projects\Chest_Xray_AI\rag\v2\chunks\atelectasis_chunks.json
Total chunks: 10

----------------------------------------------------------------------
atelectasis_001 | Definition | 86 chars
atelectasis_002 | Causes | 1,236 chars
atelectasis_003 | Symptoms | 135 chars
atelectasis_004 | Exams and Tests | 292 chars
atelectasis_005 | Treatment | 865 chars
atelectasis_006 | Outlook (Prognosis) | 656 chars
atelectasis_007 | Possible Complications | 104 chars
atelectasis_008 | When to Contact a Medical Professional | 123 chars
atelectasis_009 | Prevention | 214 chars
atelectasis_010 | Alternative Names | 39 chars


In [76]:
import requests
from bs4 import BeautifulSoup
from pathlib import Path
import json
import re


RAG_DIR = Path.cwd().parent / "rag"

V2_DOCUMENTS_DIR = RAG_DIR / "v2" / "documents"
V2_CHUNKS_DIR = RAG_DIR / "v2" / "chunks"


def ingest_medlineplus_source(
    source_id,
    finding,
    title,
    url,
    organization="MedlinePlus / NIH"
):
    """
    Download, clean, and section-chunk a MedlinePlus source.
    """

    headers = {
        "User-Agent": (
            "Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
            "AppleWebKit/537.36 Chrome/154.0 Safari/537.36"
        )
    }

    # ---------------------------------------------------------
    # 1. Download
    # ---------------------------------------------------------
    response = requests.get(
        url,
        headers=headers,
        timeout=30
    )

    response.raise_for_status()

    # ---------------------------------------------------------
    # 2. Parse HTML
    # ---------------------------------------------------------
    soup = BeautifulSoup(response.text, "html.parser")

    for element in soup([
        "script",
        "style",
        "nav",
        "header",
        "footer",
        "noscript"
    ]):
        element.decompose()

    text = soup.get_text(separator="\n")

    # ---------------------------------------------------------
    # 3. Fix common encoding problems
    # ---------------------------------------------------------
    encoding_fixes = {
        "â": "'",
        "â": "-",
        "â": '"',
        "â": '"',
        "Ã©": "é",
        "Ã¨": "è",
        "Ã": "à",
    }

    for bad, good in encoding_fixes.items():
        text = text.replace(bad, good)

    # ---------------------------------------------------------
    # 4. Basic line cleanup
    # ---------------------------------------------------------
    lines = [
        line.strip()
        for line in text.splitlines()
        if line.strip()
    ]

    text = "\n".join(lines)

    # ---------------------------------------------------------
    # 5. Remove webpage boilerplate
    # ---------------------------------------------------------
    text = re.sub(
        r"Skip navigation.*?(?=\n[A-Z][^\n]+\n)",
        "",
        text,
        flags=re.DOTALL
    )

    # Remove page URL line
    text = re.sub(
        r"URL of this page:.*?\n",
        "",
        text
    )

    # Remove references and everything after them
    text = re.split(
        r"\nReferences\n|\nReferences$",
        text,
        maxsplit=1
    )[0]

    # Remove image/navigation block
    text = re.split(
        r"\nImages\n",
        text,
        maxsplit=1
    )[0]

    # ---------------------------------------------------------
    # 6. Clean whitespace again
    # ---------------------------------------------------------
    lines = [
        line.strip()
        for line in text.splitlines()
        if line.strip()
    ]

    text = "\n".join(lines)

    # ---------------------------------------------------------
    # 7. Remove consecutive duplicate title
    # ---------------------------------------------------------
    text = text.replace(
        f"{title}\n{title}\n",
        f"{title}\n"
    )

    # ---------------------------------------------------------
    # 8. Save cleaned document
    # ---------------------------------------------------------
    document_file = (
        V2_DOCUMENTS_DIR /
        f"{source_id}_clean.txt"
    )

    with open(document_file, "w", encoding="utf-8") as f:
        f.write(text)

    # ---------------------------------------------------------
    # 9. Section detection
    # ---------------------------------------------------------
    section_candidates = [
        "Causes",
        "Symptoms",
        "Signs and Symptoms",
        "Exams and Tests",
        "Diagnosis",
        "Treatment",
        "Outlook (Prognosis)",
        "Prognosis",
        "Possible Complications",
        "Complications",
        "Prevention",
        "When to Contact a Medical Professional",
        "Alternative Names",
        "Causes and Risk Factors",
        "Living With",
    ]

    positions = []

    for section in section_candidates:
        position = text.find(section)

        if position != -1:
            positions.append((position, section))

    positions.sort()

    # Remove duplicate positions
    unique_positions = []

    for position, section in positions:
        if not unique_positions or position != unique_positions[-1][0]:
            unique_positions.append((position, section))

    positions = unique_positions

    chunks = []

    # ---------------------------------------------------------
    # 10. Definition / introduction
    # ---------------------------------------------------------
    if positions:
        definition_text = text[:positions[0][0]].strip()

        if definition_text:
            chunks.append({
                "chunk_id": f"{source_id}_001",
                "source_id": source_id,
                "source_title": title,
                "organization": organization,
                "source_type": "medical_reference",
                "finding": finding,
                "section": "Definition",
                "text": definition_text
            })

    # ---------------------------------------------------------
    # 11. Section chunks
    # ---------------------------------------------------------
    for i, (start, section) in enumerate(positions):

        end = (
            positions[i + 1][0]
            if i + 1 < len(positions)
            else len(text)
        )

        section_text = text[start:end].strip()

        if not section_text:
            continue

        chunks.append({
            "chunk_id": f"{source_id}_{len(chunks) + 1:03d}",
            "source_id": source_id,
            "source_title": title,
            "organization": organization,
            "source_type": "medical_reference",
            "finding": finding,
            "section": section,
            "text": section_text
        })

    # ---------------------------------------------------------
    # 12. Save chunks
    # ---------------------------------------------------------
    chunk_file = (
        V2_CHUNKS_DIR /
        f"{source_id}_chunks.json"
    )

    with open(chunk_file, "w", encoding="utf-8") as f:
        json.dump(
            chunks,
            f,
            indent=2,
            ensure_ascii=False
        )

    # ---------------------------------------------------------
    # 13. Summary
    # ---------------------------------------------------------
    print("=" * 70)
    print(f"INGESTED: {finding}")
    print("=" * 70)

    print(f"HTTP status : {response.status_code}")
    print(f"Characters  : {len(text):,}")
    print(f"Chunks      : {len(chunks)}")
    print(f"Document    : {document_file}")
    print(f"Chunks file : {chunk_file}")

    print("\nSections:")

    for chunk in chunks:
        print(
            f"  {chunk['chunk_id']} | "
            f"{chunk['section']} | "
            f"{len(chunk['text']):,} chars"
        )

    return chunks

In [77]:
medlineplus_sources = [
    {
        "source_id": "pleural_effusion_medlineplus",
        "finding": "Effusion",
        "title": "Pleural Effusion",
        "url": "https://medlineplus.gov/ency/article/000086.htm"
    },
    {
        "source_id": "pulmonary_edema_medlineplus",
        "finding": "Edema",
        "title": "Pulmonary Edema",
        "url": "https://medlineplus.gov/ency/article/000140.htm"
    },
    {
        "source_id": "diaphragmatic_hernia_medlineplus",
        "finding": "Hernia",
        "title": "Diaphragmatic Hernia",
        "url": "https://medlineplus.gov/ency/article/001135.htm"
    }
]

all_medlineplus_chunks = []

for source in medlineplus_sources:
    chunks = ingest_medlineplus_source(
        source_id=source["source_id"],
        finding=source["finding"],
        title=source["title"],
        url=source["url"]
    )

    all_medlineplus_chunks.extend(chunks)

print("\n" + "=" * 70)
print("MEDLINEPLUS INGESTION COMPLETE")
print("=" * 70)
print(f"Sources processed : {len(medlineplus_sources)}")
print(f"Total new chunks  : {len(all_medlineplus_chunks)}")

INGESTED: Effusion
HTTP status : 200
Characters  : 4,409
Chunks      : 11
Document    : c:\Users\yuvan\Desktop\ML_projects\Chest_Xray_AI\rag\v2\documents\pleural_effusion_medlineplus_clean.txt
Chunks file : c:\Users\yuvan\Desktop\ML_projects\Chest_Xray_AI\rag\v2\chunks\pleural_effusion_medlineplus_chunks.json

Sections:
  pleural_effusion_medlineplus_001 | Definition | 540 chars
  pleural_effusion_medlineplus_002 | Causes | 816 chars
  pleural_effusion_medlineplus_003 | Symptoms | 226 chars
  pleural_effusion_medlineplus_004 | Exams and Tests | 1,049 chars
  pleural_effusion_medlineplus_005 | Treatment | 1,169 chars
  pleural_effusion_medlineplus_006 | Outlook (Prognosis) | 9 chars
  pleural_effusion_medlineplus_007 | Prognosis | 57 chars
  pleural_effusion_medlineplus_008 | Possible Complications | 8 chars
  pleural_effusion_medlineplus_009 | Complications | 256 chars
  pleural_effusion_medlineplus_010 | When to Contact a Medical Professional | 200 chars
  pleural_effusion_medlineplus

In [78]:
import re

# Replace the section candidates used by the ingestion function.
# Longer headings must be checked before their shorter variants.

SECTION_CANDIDATES = [
    "When to Contact a Medical Professional",
    "Possible Complications",
    "Outlook (Prognosis)",
    "Signs and Symptoms",
    "Causes and Risk Factors",
    "Exams and Tests",
    "Alternative Names",
    "Complications",
    "Prognosis",
    "Symptoms",
    "Treatment",
    "Diagnosis",
    "Prevention",
    "Living With",
    "Causes",
]


def detect_sections(text):
    """
    Detect section headings while avoiding overlapping headings.
    """

    positions = []

    for section in SECTION_CANDIDATES:
        for match in re.finditer(
            rf"(?m)^{re.escape(section)}$",
            text
        ):
            positions.append((match.start(), section))

    # Sort by position first
    positions.sort(key=lambda x: x[0])

    # Remove duplicate/overlapping section detections
    final_positions = []

    for position, section in positions:

        if not final_positions:
            final_positions.append((position, section))
            continue

        previous_position, previous_section = final_positions[-1]

        # If this heading occurs inside the previous heading's text
        # region immediately at the same position, keep the longer one.
        if position == previous_position:

            if len(section) > len(previous_section):
                final_positions[-1] = (position, section)

            continue

        final_positions.append((position, section))

    return final_positions


print("=" * 70)
print("SECTION DETECTOR READY")
print("=" * 70)

print("Supported section headings:")

for section in SECTION_CANDIDATES:
    print(f"  - {section}")

print("\nOverlap protection: ENABLED")

SECTION DETECTOR READY
Supported section headings:
  - When to Contact a Medical Professional
  - Possible Complications
  - Outlook (Prognosis)
  - Signs and Symptoms
  - Causes and Risk Factors
  - Exams and Tests
  - Alternative Names
  - Complications
  - Prognosis
  - Symptoms
  - Treatment
  - Diagnosis
  - Prevention
  - Living With
  - Causes

Overlap protection: ENABLED


In [79]:
import json
from pathlib import Path

RAG_DIR = Path.cwd().parent / "rag"

V2_DOCUMENTS_DIR = RAG_DIR / "v2" / "documents"
V2_CHUNKS_DIR = RAG_DIR / "v2" / "chunks"


def rebuild_chunks_from_document(
    source_id,
    finding,
    title,
    organization="MedlinePlus / NIH"
):
    document_file = (
        V2_DOCUMENTS_DIR /
        f"{source_id}_clean.txt"
    )

    chunk_file = (
        V2_CHUNKS_DIR /
        f"{source_id}_chunks.json"
    )

    with open(document_file, "r", encoding="utf-8") as f:
        text = f.read()

    positions = detect_sections(text)

    chunks = []

    # ---------------------------------------------------------
    # Definition / introduction
    # ---------------------------------------------------------
    if positions:
        definition_text = text[:positions[0][0]].strip()

        if definition_text:
            chunks.append({
                "chunk_id": f"{source_id}_001",
                "source_id": source_id,
                "source_title": title,
                "organization": organization,
                "source_type": "medical_reference",
                "finding": finding,
                "section": "Definition",
                "text": definition_text
            })

    # ---------------------------------------------------------
    # Section chunks
    # ---------------------------------------------------------
    for i, (start, section) in enumerate(positions):

        end = (
            positions[i + 1][0]
            if i + 1 < len(positions)
            else len(text)
        )

        section_text = text[start:end].strip()

        if not section_text:
            continue

        chunks.append({
            "chunk_id": f"{source_id}_{len(chunks) + 1:03d}",
            "source_id": source_id,
            "source_title": title,
            "organization": organization,
            "source_type": "medical_reference",
            "finding": finding,
            "section": section,
            "text": section_text
        })

    with open(chunk_file, "w", encoding="utf-8") as f:
        json.dump(
            chunks,
            f,
            indent=2,
            ensure_ascii=False
        )

    print(f"\n{finding}")
    print("-" * 70)
    print(f"Total chunks: {len(chunks)}")

    for chunk in chunks:
        print(
            f"{chunk['chunk_id']} | "
            f"{chunk['section']} | "
            f"{len(chunk['text']):,} chars"
        )

    return chunks


sources_to_rebuild = [
    {
        "source_id": "pleural_effusion_medlineplus",
        "finding": "Effusion",
        "title": "Pleural Effusion"
    },
    {
        "source_id": "pulmonary_edema_medlineplus",
        "finding": "Edema",
        "title": "Pulmonary Edema"
    },
    {
        "source_id": "diaphragmatic_hernia_medlineplus",
        "finding": "Hernia",
        "title": "Diaphragmatic Hernia"
    }
]

rebuilt_chunks = []

for source in sources_to_rebuild:
    chunks = rebuild_chunks_from_document(**source)
    rebuilt_chunks.extend(chunks)

print("\n" + "=" * 70)
print("MEDLINEPLUS CHUNKS REBUILT")
print("=" * 70)
print(f"Total chunks across 3 sources: {len(rebuilt_chunks)}")


Effusion
----------------------------------------------------------------------
Total chunks: 9
pleural_effusion_medlineplus_001 | Definition | 540 chars
pleural_effusion_medlineplus_002 | Causes | 816 chars
pleural_effusion_medlineplus_003 | Symptoms | 226 chars
pleural_effusion_medlineplus_004 | Exams and Tests | 1,049 chars
pleural_effusion_medlineplus_005 | Treatment | 1,169 chars
pleural_effusion_medlineplus_006 | Outlook (Prognosis) | 66 chars
pleural_effusion_medlineplus_007 | Possible Complications | 265 chars
pleural_effusion_medlineplus_008 | When to Contact a Medical Professional | 200 chars
pleural_effusion_medlineplus_009 | Alternative Names | 70 chars

Edema
----------------------------------------------------------------------
Total chunks: 9
pulmonary_edema_medlineplus_001 | Definition | 543 chars
pulmonary_edema_medlineplus_002 | Causes | 940 chars
pulmonary_edema_medlineplus_003 | Symptoms | 584 chars
pulmonary_edema_medlineplus_004 | Exams and Tests | 865 chars
pulm

In [80]:
import requests
from bs4 import BeautifulSoup
from pathlib import Path

RAG_DIR = Path.cwd().parent / "rag"
V2_DOCUMENTS_DIR = RAG_DIR / "v2" / "documents"

url = "https://www.ncbi.nlm.nih.gov/books/NBK542296/"

headers = {
    "User-Agent": (
        "Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
        "AppleWebKit/537.36 Chrome/154.0 Safari/537.36"
    )
}

response = requests.get(
    url,
    headers=headers,
    timeout=30
)

response.raise_for_status()

soup = BeautifulSoup(response.text, "html.parser")

# Remove obvious webpage noise
for element in soup([
    "script",
    "style",
    "nav",
    "header",
    "footer",
    "noscript"
]):
    element.decompose()

# Inspect heading structure
headings = []

for tag in soup.find_all(["h1", "h2", "h3"]):
    heading = tag.get_text(" ", strip=True)

    if heading:
        headings.append(
            (tag.name, heading)
        )

print("=" * 70)
print("CARDIOMEGALY — NCBI PAGE STRUCTURE")
print("=" * 70)

print(f"HTTP status : {response.status_code}")
print(f"Headings    : {len(headings)}")

print("\n" + "-" * 70)

for tag, heading in headings:
    print(f"{tag:>2} | {heading}")

print("\n" + "-" * 70)
print("PAGE TITLE")
print("-" * 70)

print(soup.title.get_text(strip=True) if soup.title else "No title")

CARDIOMEGALY — NCBI PAGE STRUCTURE
HTTP status : 200
Headings    : 0

----------------------------------------------------------------------

----------------------------------------------------------------------
PAGE TITLE
----------------------------------------------------------------------
Checking your browser - reCAPTCHA


In [81]:
import requests
from pathlib import Path

RAG_DIR = Path.cwd().parent / "rag"
V2_DOCUMENTS_DIR = RAG_DIR / "v2" / "documents"

book_id = "NBK542296"

url = (
    "https://eutils.ncbi.nlm.nih.gov/entrez/eutils/"
    f"efetch.fcgi?db=books&id={book_id}&retmode=xml"
)

response = requests.get(
    url,
    timeout=30
)

response.raise_for_status()

xml_text = response.text

print("=" * 70)
print("NCBI BOOKSHELF XML TEST — CARDIOMEGALY")
print("=" * 70)

print(f"HTTP status : {response.status_code}")
print(f"Characters  : {len(xml_text):,}")

print("\n" + "-" * 70)
print("RESPONSE PREVIEW")
print("-" * 70)

print(xml_text[:3000])

NCBI BOOKSHELF XML TEST — CARDIOMEGALY
HTTP status : 200
Characters  : 193

----------------------------------------------------------------------
RESPONSE PREVIEW
----------------------------------------------------------------------
<?xml version="1.0" ?>
<!DOCTYPE eEfetchResult PUBLIC "-//NLM//DTD efetch 20020605//EN" "https://eutils.ncbi.nlm.nih.gov/eutils/dtd/20020605/uilist.dtd">
<IdList>
    <Id>542296</Id>
</IdList>



In [82]:
import requests
from bs4 import BeautifulSoup

url = "https://www.ncbi.nlm.nih.gov/books/NBK542296/?report=reader"

headers = {
    "User-Agent": (
        "Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
        "AppleWebKit/537.36 (KHTML, like Gecko) "
        "Chrome/154.0 Safari/537.36"
    )
}

response = requests.get(
    url,
    headers=headers,
    timeout=30
)

response.raise_for_status()

soup = BeautifulSoup(response.text, "html.parser")

for element in soup([
    "script",
    "style",
    "nav",
    "header",
    "footer",
    "noscript"
]):
    element.decompose()

text = soup.get_text("\n", strip=True)

print("=" * 70)
print("NCBI BOOKSHELF READER TEST — CARDIOMEGALY")
print("=" * 70)

print(f"HTTP status : {response.status_code}")
print(f"Characters  : {len(text):,}")

print("\n" + "-" * 70)
print("TEXT PREVIEW")
print("-" * 70)

print(text[:5000])

NCBI BOOKSHELF READER TEST — CARDIOMEGALY
HTTP status : 200
Characters  : 165

----------------------------------------------------------------------
TEXT PREVIEW
----------------------------------------------------------------------
Checking your browser - reCAPTCHA
Checking your browser before accessing www.ncbi.nlm.nih.gov ...
Click
here
if you are not automatically redirected after 5 seconds.


In [83]:
import requests
from bs4 import BeautifulSoup
from urllib.parse import quote

query = quote(
    "cardiomegaly chest x-ray authoritative medical reference "
    "site:nih.gov OR site:medlineplus.gov OR site:radiologyinfo.org "
    "OR site:acr.org OR site:rsna.org"
)

url = f"https://www.google.com/search?q={query}"

headers = {
    "User-Agent": (
        "Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
        "AppleWebKit/537.36 Chrome/154.0 Safari/537.36"
    )
}

response = requests.get(
    url,
    headers=headers,
    timeout=30
)

print("=" * 70)
print("CARDIOMEGALY AUTHORITATIVE SOURCE SEARCH")
print("=" * 70)

print(f"HTTP status : {response.status_code}")
print(f"Characters  : {len(response.text):,}")

soup = BeautifulSoup(response.text, "html.parser")

results = []

for result in soup.select("div"):
    link = result.find("a", href=True)

    if not link:
        continue

    href = link["href"]
    title = link.get_text(" ", strip=True)

    if (
        title
        and (
            "cardiomegaly" in title.lower()
            or "enlarged heart" in title.lower()
        )
        and href.startswith("http")
    ):
        results.append((title, href))

# Remove duplicates while preserving order
seen = set()
unique_results = []

for title, href in results:
    if href not in seen:
        seen.add(href)
        unique_results.append((title, href))

print("\nPotential sources:\n")

for i, (title, href) in enumerate(unique_results[:10], start=1):
    print(f"{i}. {title}")
    print(f"   {href}")

CARDIOMEGALY AUTHORITATIVE SOURCE SEARCH
HTTP status : 200
Characters  : 92,061

Potential sources:



In [84]:
from bs4 import BeautifulSoup
from urllib.parse import urlparse, parse_qs, unquote

soup = BeautifulSoup(response.text, "html.parser")

candidates = []

for link in soup.find_all("a", href=True):
    href = link["href"]
    text = link.get_text(" ", strip=True)

    # Google often wraps result URLs inside /url?q=...
    if href.startswith("/url?q="):
        href = parse_qs(urlparse(href).query).get("q", [""])[0]

    href = unquote(href)

    if not href.startswith("http"):
        continue

    combined = f"{text} {href}".lower()

    if (
        "cardiomegaly" in combined
        or "enlarged heart" in combined
    ):
        candidates.append((text, href))

# Remove duplicate URLs
seen = set()
unique_candidates = []

for title, href in candidates:
    if href not in seen:
        seen.add(href)
        unique_candidates.append((title, href))

print("=" * 70)
print("CARDIOMEGALY SOURCE CANDIDATES")
print("=" * 70)

if not unique_candidates:
    print("No candidates extracted.")
else:
    for i, (title, href) in enumerate(unique_candidates[:15], start=1):
        print(f"\n{i}. {title}")
        print(f"   {href}")

print("\n" + "=" * 70)
print(f"TOTAL CANDIDATES: {len(unique_candidates)}")
print("=" * 70)

CARDIOMEGALY SOURCE CANDIDATES
No candidates extracted.

TOTAL CANDIDATES: 0


In [85]:
import requests
from xml.etree import ElementTree as ET

query = (
    'cardiomegaly AND '
    '("chest radiograph" OR "chest x-ray" OR imaging)'
)

url = "https://eutils.ncbi.nlm.nih.gov/entrez/eutils/esearch.fcgi"

params = {
    "db": "pubmed",
    "term": query,
    "retmax": 10,
    "retmode": "json"
}

response = requests.get(
    url,
    params=params,
    timeout=30
)

response.raise_for_status()

data = response.json()

pmids = data["esearchresult"]["idlist"]

print("=" * 70)
print("PUBMED CARDIOMEGALY SEARCH")
print("=" * 70)

print(f"Query : {query}")
print(f"PMIDs : {len(pmids)}")

for i, pmid in enumerate(pmids, start=1):
    print(f"{i}. PMID {pmid}")

print("\n" + "=" * 70)

PUBMED CARDIOMEGALY SEARCH
Query : cardiomegaly AND ("chest radiograph" OR "chest x-ray" OR imaging)
PMIDs : 10
1. PMID 42801397
2. PMID 42798082
3. PMID 42789244
4. PMID 42781233
5. PMID 42758693
6. PMID 42751687
7. PMID 42748927
8. PMID 42733665
9. PMID 42733268
10. PMID 42728072



In [86]:
import requests
import xml.etree.ElementTree as ET

pmids = [
    "42801397",
    "42798082",
    "42789244",
    "42781233",
    "42758693",
    "42751687",
    "42748927",
    "42733665",
    "42733268",
    "42728072"
]

url = "https://eutils.ncbi.nlm.nih.gov/entrez/eutils/efetch.fcgi"

params = {
    "db": "pubmed",
    "id": ",".join(pmids),
    "retmode": "xml"
}

response = requests.get(
    url,
    params=params,
    timeout=30
)

response.raise_for_status()

root = ET.fromstring(response.text)

print("=" * 80)
print("PUBMED CARDIOMEGALY CANDIDATES")
print("=" * 80)

articles = root.findall(".//PubmedArticle")

print(f"Articles retrieved: {len(articles)}")

for i, article in enumerate(articles, start=1):

    pmid = article.findtext(".//PMID", default="Unknown")

    title_element = article.find(".//ArticleTitle")

    if title_element is not None:
        title = "".join(title_element.itertext())
    else:
        title = "No title"

    abstract_parts = []

    for abstract_text in article.findall(".//AbstractText"):
        abstract_parts.append(
            "".join(abstract_text.itertext())
        )

    abstract = " ".join(abstract_parts)

    journal = article.findtext(
        ".//Journal/Title",
        default="Unknown"
    )

    year = (
        article.findtext(".//PubDate/Year")
        or article.findtext(".//PubDate/MedlineDate")
        or "Unknown"
    )

    publication_types = [
        pt.text
        for pt in article.findall(".//PublicationType")
        if pt.text
    ]

    print(f"\n{i}. PMID: {pmid}")
    print(f"   Title  : {title}")
    print(f"   Journal: {journal}")
    print(f"   Year   : {year}")
    print(f"   Types  : {', '.join(publication_types)}")

    if abstract:
        print(f"   Abstract: {abstract[:700]}...")
    else:
        print("   Abstract: Not available")

PUBMED CARDIOMEGALY CANDIDATES
Articles retrieved: 10

1. PMID: 42801397
   Title  : A rare congenital triad: ebsteinoid tricuspid valve, giant right atrial aneurysm, and atrial septal defect.
   Journal: The Egyptian heart journal : (EHJ) : official bulletin of the Egyptian Society of Cardiology
   Year   : 2026
   Types  : Journal Article
   Abstract: Ebstein anomaly is a rare congenital malformation of the tricuspid valve, accounting for less than 1% of all congenital heart diseases. Ebsteinoid anomaly is an uncommon morphological variant characterized by abnormal tricuspid valve leaflet morphology without the classical apical displacement of the septal leaflet seen in Ebstein anomaly. The coexistence of an Ebsteinoid anomaly, giant right atrial aneurysm (RAA), and atrial septal defect (ASD) is exceptionally rare. A 15-year-old girl presented with progressive exertional dyspnoea (New York Heart Association [NYHA] class II) and intermittent palpitations. Chest radiography demonstrate

In [87]:
import requests
from bs4 import BeautifulSoup
from urllib.parse import quote

query = quote("cardiomegaly enlarged heart")

url = f"https://medlineplus.gov/search.html?query={query}"

headers = {
    "User-Agent": (
        "Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
        "AppleWebKit/537.36 Chrome/154.0 Safari/537.36"
    )
}

response = requests.get(
    url,
    headers=headers,
    timeout=30
)

response.raise_for_status()

soup = BeautifulSoup(response.text, "html.parser")

print("=" * 70)
print("MEDLINEPLUS CARDIOMEGALY SEARCH")
print("=" * 70)

print(f"HTTP status : {response.status_code}")

results = []

for link in soup.find_all("a", href=True):

    title = link.get_text(" ", strip=True)
    href = link["href"]

    if not title:
        continue

    combined = f"{title} {href}".lower()

    if (
        "cardiomegaly" in combined
        or "enlarged heart" in combined
    ):
        if href.startswith("/"):
            href = "https://medlineplus.gov" + href

        results.append((title, href))

# Deduplicate
seen = set()
unique_results = []

for title, href in results:
    if href not in seen:
        seen.add(href)
        unique_results.append((title, href))

print("\nPotential MedlinePlus results:\n")

for i, (title, href) in enumerate(unique_results[:15], start=1):
    print(f"{i}. {title}")
    print(f"   {href}")

print("\n" + "=" * 70)
print(f"TOTAL RESULTS: {len(unique_results)}")
print("=" * 70)

HTTPError: 404 Client Error: Not Found for url: https://medlineplus.gov/search.html?query=cardiomegaly%20enlarged%20heart

In [88]:
import requests
from bs4 import BeautifulSoup

urls = [
    "https://www.radiologyinfo.org/en/search-results?search=cardiomegaly",
    "https://www.radiologyinfo.org/en/search-results?search=enlarged%20heart"
]

headers = {
    "User-Agent": (
        "Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
        "AppleWebKit/537.36 Chrome/154.0 Safari/537.36"
    )
}

for url in urls:

    print("\n" + "=" * 70)
    print(f"CHECKING: {url}")
    print("=" * 70)

    try:
        response = requests.get(
            url,
            headers=headers,
            timeout=30
        )

        print(f"HTTP status: {response.status_code}")

        if response.status_code != 200:
            continue

        soup = BeautifulSoup(response.text, "html.parser")

        print("\nRelevant links:")

        found = 0

        for link in soup.find_all("a", href=True):

            title = link.get_text(" ", strip=True)
            href = link["href"]

            combined = f"{title} {href}".lower()

            if (
                "cardiomegaly" in combined
                or "enlarged heart" in combined
            ):
                if href.startswith("/"):
                    href = "https://www.radiologyinfo.org" + href

                print(f"- {title}")
                print(f"  {href}")

                found += 1

        print(f"\nMatches found: {found}")

    except requests.RequestException as e:
        print(f"Request error: {e}")


CHECKING: https://www.radiologyinfo.org/en/search-results?search=cardiomegaly
HTTP status: 404

CHECKING: https://www.radiologyinfo.org/en/search-results?search=enlarged%20heart
HTTP status: 404


In [89]:
import requests
from bs4 import BeautifulSoup

url = "https://www.merckmanuals.com/professional/cardiovascular-disorders/diagnosis-of-cardiovascular-disorders/cardiomegaly"

headers = {
    "User-Agent": (
        "Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
        "AppleWebKit/537.36 Chrome/154.0 Safari/537.36"
    )
}

response = requests.get(
    url,
    headers=headers,
    timeout=30
)

print("=" * 70)
print("CARDIOMEGALY — MERCK MANUAL CHECK")
print("=" * 70)

print(f"HTTP status : {response.status_code}")
print(f"Final URL   : {response.url}")

if response.status_code == 200:

    soup = BeautifulSoup(response.text, "html.parser")

    for element in soup([
        "script",
        "style",
        "nav",
        "header",
        "footer",
        "noscript"
    ]):
        element.decompose()

    text = soup.get_text("\n", strip=True)

    print(f"Characters  : {len(text):,}")

    print("\n" + "-" * 70)
    print("TEXT PREVIEW")
    print("-" * 70)

    print(text[:4000])

else:
    print("\nSource is not accessible through this request.")

CARDIOMEGALY — MERCK MANUAL CHECK
HTTP status : 404
Final URL   : https://www.merckmanuals.com/professional/cardiovascular-disorders/diagnosis-of-cardiovascular-disorders/cardiomegaly

Source is not accessible through this request.


In [90]:
import requests
from bs4 import BeautifulSoup

url = "https://www.ncbi.nlm.nih.gov/medgen/636648"

headers = {
    "User-Agent": (
        "Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
        "AppleWebKit/537.36 Chrome/154.0 Safari/537.36"
    )
}

response = requests.get(
    url,
    headers=headers,
    timeout=30
)

print("=" * 70)
print("NCBI MEDGEN — LUNG CONSOLIDATION")
print("=" * 70)

print(f"HTTP status : {response.status_code}")
print(f"Final URL   : {response.url}")

if response.status_code == 200:

    soup = BeautifulSoup(response.text, "html.parser")

    for element in soup([
        "script",
        "style",
        "nav",
        "header",
        "footer",
        "noscript"
    ]):
        element.decompose()

    text = soup.get_text("\n", strip=True)

    print(f"Characters  : {len(text):,}")

    print("\n" + "-" * 70)
    print("TEXT PREVIEW")
    print("-" * 70)

    print(text[:4000])

else:
    print("\nSource is not accessible through this request.")

NCBI MEDGEN — LUNG CONSOLIDATION
HTTP status : 200
Final URL   : https://www.ncbi.nlm.nih.gov/medgen/636648
Characters  : 13,067

----------------------------------------------------------------------
TEXT PREVIEW
----------------------------------------------------------------------
Lung consolidation (Concept Id: C0521530)
 - MedGen - NCBI
The .gov means it's official.
Federal government websites often end in .gov or .mil. Before
							sharing sensitive information, make sure you're on a federal
							government site.
The site is secure.
The
https://
ensures that you are connecting to the
							official website and that any information you provide is encrypted
							and transmitted securely.
Access keys
NCBI Homepage
MyNCBI Homepage
Main Content
Main Navigation
MedGen
National Center for Biotechnology Information
Search database
All Databases
Assembly
Biocollections
BioProject
BioSample
Books
ClinVar
Conserved Domains
dbVar
Gene
Genome
GEO DataSets
GEO Profiles
GTR
Identical Prot

In [91]:
import json
from pathlib import Path
from bs4 import BeautifulSoup

RAG_DIR = Path.cwd().parent / "rag"

V2_DOCUMENTS_DIR = RAG_DIR / "v2" / "documents"
V2_CHUNKS_DIR = RAG_DIR / "v2" / "chunks"

source_id = "lung_consolidation_ncbi"
finding = "Consolidation"
title = "Lung Consolidation"
organization = "NCBI MedGen"

url = "https://www.ncbi.nlm.nih.gov/medgen/636648"

# Use the HTML already downloaded in Cell 29
soup = BeautifulSoup(response.text, "html.parser")

for element in soup([
    "script",
    "style",
    "nav",
    "header",
    "footer",
    "noscript"
]):
    element.decompose()

# Locate the actual MedGen record
record_text = ""

definition_heading = soup.find(
    string=lambda s: s and s.strip() == "Definition"
)

if definition_heading:
    parent = definition_heading.parent

    # Capture the definition and nearby record information
    container = parent.parent

    record_text = container.get_text(
        "\n",
        strip=True
    )

# If the structure differs, fall back to extracting
# the title + definition directly from visible text.
if not record_text:
    full_text = soup.get_text("\n", strip=True)

    start_marker = "Lung consolidation MedGen UID:"
    end_marker = "Term Hierarchy"

    if start_marker in full_text:
        record_text = full_text[
            full_text.index(start_marker):
            full_text.index(end_marker)
        ]

# Clean lines
lines = [
    line.strip()
    for line in record_text.splitlines()
    if line.strip()
]

clean_text = "\n".join(lines)

document_file = (
    V2_DOCUMENTS_DIR /
    f"{source_id}_clean.txt"
)

with open(document_file, "w", encoding="utf-8") as f:
    f.write(clean_text)

# Create one focused semantic chunk
chunk = {
    "chunk_id": f"{source_id}_001",
    "source_id": source_id,
    "source_title": title,
    "organization": organization,
    "source_type": "medical_reference",
    "finding": finding,
    "section": "Definition",
    "text": clean_text,
    "url": url
}

chunk_file = (
    V2_CHUNKS_DIR /
    f"{source_id}_chunks.json"
)

with open(chunk_file, "w", encoding="utf-8") as f:
    json.dump(
        [chunk],
        f,
        indent=2,
        ensure_ascii=False
    )

print("=" * 70)
print("MEDGEN CONSOLIDATION SOURCE")
print("=" * 70)

print(f"Characters : {len(clean_text):,}")
print(f"Document   : {document_file}")
print(f"Chunk file : {chunk_file}")

print("\n" + "-" * 70)
print("EXTRACTED CONTENT")
print("-" * 70)

print(clean_text)

MEDGEN CONSOLIDATION SOURCE
Characters : 10
Document   : c:\Users\yuvan\Desktop\ML_projects\Chest_Xray_AI\rag\v2\documents\lung_consolidation_ncbi_clean.txt
Chunk file : c:\Users\yuvan\Desktop\ML_projects\Chest_Xray_AI\rag\v2\chunks\lung_consolidation_ncbi_chunks.json

----------------------------------------------------------------------
EXTRACTED CONTENT
----------------------------------------------------------------------
Definition


In [ ]:
import json
import requests
from pathlib import Path
from bs4 import BeautifulSoup

RAG_DIR = Path.cwd().parent / "rag"

V2_DOCUMENTS_DIR = RAG_DIR / "v2" / "documents"
V2_CHUNKS_DIR = RAG_DIR / "v2" / "chunks"

source_id = "lung_consolidation_ncbi"
finding = "Consolidation"
title = "Lung Consolidation"
organization = "NCBI MedGen"

url = "https://www.ncbi.nlm.nih.gov/medgen/636648"

headers = {
    "User-Agent": (
        "Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
        "AppleWebKit/537.36 Chrome/154.0 Safari/537.36"
    )
}

response = requests.get(
    url,
    headers=headers,
    timeout=30
)

response.raise_for_status()

soup = BeautifulSoup(response.text, "html.parser")

for element in soup([
    "script",
    "style",
    "nav",
    "header",
    "footer",
    "noscript"
]):
    element.decompose()

# Get visible text as individual lines
lines = [
    line.strip()
    for line in soup.get_text("\n").splitlines()
    if line.strip()
]

# Find the exact "Definition" line
definition_index = None

for i, line in enumerate(lines):
    if line == "Definition":
        definition_index = i
        break

if definition_index is None:
    raise ValueError(
        "Could not locate the MedGen Definition heading."
    )

# The next non-empty line is the definition
if definition_index + 1 >= len(lines):
    raise ValueError(
        "Definition heading found, but no definition text follows."
    )

definition_text = lines[definition_index + 1]

# Sanity check: make sure this actually looks like the expected
# MedGen definition rather than unrelated page content.
if "chest X-ray" not in definition_text.lower():
    raise ValueError(
        "Extracted text does not appear to be the expected "
        "Lung Consolidation definition."
    )

# ---------------------------------------------------------
# Save focused document
# ---------------------------------------------------------

document_text = (
    f"{title}\n\n"
    f"Definition\n"
    f"{definition_text}"
)

document_file = (
    V2_DOCUMENTS_DIR /
    f"{source_id}_clean.txt"
)

with open(document_file, "w", encoding="utf-8") as f:
    f.write(document_text)

# ---------------------------------------------------------
# Save semantic chunk
# ---------------------------------------------------------

chunk = {
    "chunk_id": f"{source_id}_001",
    "source_id": source_id,
    "source_title": title,
    "organization": organization,
    "source_type": "medical_reference",
    "finding": finding,
    "section": "Definition",
    "text": definition_text,
    "url": url
}

chunk_file = (
    V2_CHUNKS_DIR /
    f"{source_id}_chunks.json"
)

with open(chunk_file, "w", encoding="utf-8") as f:
    json.dump(
        [chunk],
        f,
        indent=2,
        ensure_ascii=False
    )

print("=" * 70)
print("MEDGEN CONSOLIDATION — FINAL EXTRACTION")
print("=" * 70)

print(f"HTTP status          : {response.status_code}")
print(f"Definition characters: {len(definition_text):,}")
print(f"Document             : {document_file}")
print(f"Chunk file           : {chunk_file}")

print("\n" + "-" * 70)
print("EXTRACTED DEFINITION")
print("-" * 70)

print(definition_text)

ValueError: Extracted text does not appear to be the expected Lung Consolidation definition.

In [94]:
import requests
from bs4 import BeautifulSoup

url = "https://www.ncbi.nlm.nih.gov/medgen/636648"

headers = {
    "User-Agent": (
        "Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
        "AppleWebKit/537.36 Chrome/154.0 Safari/537.36"
    )
}

response = requests.get(
    url,
    headers=headers,
    timeout=30
)

response.raise_for_status()

soup = BeautifulSoup(response.text, "html.parser")

for element in soup([
    "script",
    "style",
    "nav",
    "header",
    "footer",
    "noscript"
]):
    element.decompose()

full_text = soup.get_text(" ", strip=True)

# Locate Definition in the flattened visible text
definition_position = full_text.find("Definition")

print("=" * 70)
print("MEDGEN DEFINITION STRUCTURE")
print("=" * 70)

print(f"Definition position: {definition_position}")

if definition_position == -1:
    print("\n' Definition ' was not found.")
else:
    start = max(0, definition_position - 300)
    end = min(
        len(full_text),
        definition_position + 1200
    )

    print("\n" + "-" * 70)
    print("TEXT AROUND DEFINITION")
    print("-" * 70)

    print(full_text[start:end])

MEDGEN DEFINITION STRUCTURE
Definition position: 1334

----------------------------------------------------------------------
TEXT AROUND DEFINITION
----------------------------------------------------------------------
) Apply Send to: Choose Destination File Clipboard Collections Format Summary (Text) Summary (XML) Create File Add to Clipboard Add to Collections Lung consolidation MedGen UID: 636648 • Concept ID: C0521530 • Disease or Syndrome Synonym: Lung consolidations SNOMED CT: Lung consolidation (95436008) Definition A chest X-ray finding indicating the presence of a radio-opaque area in the lung. The opacification is caused by fluid or solid material within the airways or lung parenchyma. [from NCI] Term Hierarchy GTR MeSH C Clinical test, R Research test, O OMIM, G GeneReviews , V ClinVar C R O G V Lung consolidation Radiologic finding Lung consolidation Professional guidelines PubMed Evaluation of Clinical Features and Treatment of Mycoplasma pneumoniae in Children With Macr

In [95]:
import json
import requests
from pathlib import Path
from bs4 import BeautifulSoup

RAG_DIR = Path.cwd().parent / "rag"

V2_DOCUMENTS_DIR = RAG_DIR / "v2" / "documents"
V2_CHUNKS_DIR = RAG_DIR / "v2" / "chunks"

source_id = "lung_consolidation_ncbi"
finding = "Consolidation"
title = "Lung Consolidation"
organization = "NCBI MedGen"
url = "https://www.ncbi.nlm.nih.gov/medgen/636648"

headers = {
    "User-Agent": (
        "Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
        "AppleWebKit/537.36 Chrome/154.0 Safari/537.36"
    )
}

response = requests.get(
    url,
    headers=headers,
    timeout=30
)

response.raise_for_status()

soup = BeautifulSoup(response.text, "html.parser")

for element in soup([
    "script",
    "style",
    "nav",
    "header",
    "footer",
    "noscript"
]):
    element.decompose()

full_text = soup.get_text(" ", strip=True)

definition_start = full_text.find("Definition")
definition_end = full_text.find("Term Hierarchy", definition_start)

if definition_start == -1:
    raise ValueError("Definition marker not found.")

if definition_end == -1:
    raise ValueError("Term Hierarchy marker not found.")

definition_block = full_text[
    definition_start + len("Definition"):
    definition_end
].strip()

# Remove the NCI attribution from the end
definition_text = definition_block.split(
    "[from NCI]",
    1
)[0].strip()

# Sanity checks
if len(definition_text) < 50:
    raise ValueError(
        f"Extracted definition is unexpectedly short: "
        f"{len(definition_text)} characters."
    )

if "chest x-ray" not in definition_text.lower():
    raise ValueError(
        "Extracted content does not contain the expected "
        "chest X-ray context."
    )

# ---------------------------------------------------------
# Save focused document
# ---------------------------------------------------------

document_text = (
    f"{title}\n\n"
    f"Definition\n"
    f"{definition_text}"
)

document_file = (
    V2_DOCUMENTS_DIR /
    f"{source_id}_clean.txt"
)

with open(document_file, "w", encoding="utf-8") as f:
    f.write(document_text)

# ---------------------------------------------------------
# Save semantic chunk
# ---------------------------------------------------------

chunk = {
    "chunk_id": f"{source_id}_001",
    "source_id": source_id,
    "source_title": title,
    "organization": organization,
    "source_type": "medical_reference",
    "finding": finding,
    "section": "Definition",
    "text": definition_text,
    "url": url
}

chunk_file = (
    V2_CHUNKS_DIR /
    f"{source_id}_chunks.json"
)

with open(chunk_file, "w", encoding="utf-8") as f:
    json.dump(
        [chunk],
        f,
        indent=2,
        ensure_ascii=False
    )

print("=" * 70)
print("MEDGEN CONSOLIDATION — FINAL EXTRACTION")
print("=" * 70)

print(f"HTTP status          : {response.status_code}")
print(f"Definition characters: {len(definition_text):,}")
print(f"Document             : {document_file}")
print(f"Chunk file           : {chunk_file}")

print("\n" + "-" * 70)
print("EXTRACTED DEFINITION")
print("-" * 70)

print(definition_text)

MEDGEN CONSOLIDATION — FINAL EXTRACTION
HTTP status          : 200
Definition characters: 175
Document             : c:\Users\yuvan\Desktop\ML_projects\Chest_Xray_AI\rag\v2\documents\lung_consolidation_ncbi_clean.txt
Chunk file           : c:\Users\yuvan\Desktop\ML_projects\Chest_Xray_AI\rag\v2\chunks\lung_consolidation_ncbi_chunks.json

----------------------------------------------------------------------
EXTRACTED DEFINITION
----------------------------------------------------------------------
A chest X-ray finding indicating the presence of a radio-opaque area in the lung. The opacification is caused by fluid or solid material within the airways or lung parenchyma.


In [96]:
import requests
from bs4 import BeautifulSoup

source_id = "pulmonary_mass_ncbi"
url = "https://www.ncbi.nlm.nih.gov/medgen/57697"

response = requests.get(
    url,
    timeout=30,
    headers={"User-Agent": "Mozilla/5.0"}
)

print("=" * 70)
print("MEDGEN MASS — SOURCE INSPECTION")
print("=" * 70)
print(f"HTTP status : {response.status_code}")

soup = BeautifulSoup(response.text, "html.parser")

full_text = soup.get_text(" ", strip=True)

print(f"Extracted characters: {len(full_text):,}")

print("\n" + "-" * 70)
print("RELEVANT TEXT PREVIEW")
print("-" * 70)

marker = "Pulmonary mass"

idx = full_text.lower().find(marker.lower())

if idx != -1:
    print(full_text[max(0, idx - 300): idx + 1500])
else:
    print("Marker not found.")
    print("\nFirst 2,000 characters:")
    print(full_text[:2000])

MEDGEN MASS — SOURCE INSPECTION
HTTP status : 200
Extracted characters: 11,377

----------------------------------------------------------------------
RELEVANT TEXT PREVIEW
----------------------------------------------------------------------
Pulmonary mass (Concept Id: C0149726)
 - MedGen - NCBI Warning: The NCBI web site requires JavaScript to function. more... An official website of the United States government Here's how you know The .gov means it's official. Federal government websites often end in .gov or .mil. Before
							sharing sensitive information, make sure you're on a federal
							government site. The site is secure. The https:// ensures that you are connecting to the
							official website and that any information you provide is encrypted
							and transmitted securely. Log in Show account info Close Account Logged in as: username Dashboard Publications Account settings Log out Access keys NCBI Homepage MyNCBI Homepage Main Content Main Navigation MedGen National C

In [97]:
# Locate the focused MedGen definition region

definition_idx = full_text.find("Definition")
hierarchy_idx = full_text.find("Term Hierarchy", definition_idx)

print("=" * 70)
print("MEDGEN MASS — DEFINITION BOUNDARY CHECK")
print("=" * 70)

print(f"Definition position    : {definition_idx}")
print(f"Term Hierarchy position: {hierarchy_idx}")

if definition_idx != -1 and hierarchy_idx != -1:
    definition_text = full_text[
        definition_idx + len("Definition"):
        hierarchy_idx
    ].strip()

    print("\n" + "-" * 70)
    print("EXTRACTED DEFINITION CANDIDATE")
    print("-" * 70)
    print(definition_text)

    print("\n" + "-" * 70)
    print("LENGTH")
    print("-" * 70)
    print(f"{len(definition_text)} characters")
else:
    print("\nCould not locate both boundaries.")
    print("\nNearby text around Definition:")
    
    if definition_idx != -1:
        print(full_text[max(0, definition_idx - 300): definition_idx + 1500])

MEDGEN MASS — DEFINITION BOUNDARY CHECK
Definition position    : 1589
Term Hierarchy position: 1772

----------------------------------------------------------------------
EXTRACTED DEFINITION CANDIDATE
----------------------------------------------------------------------
A mass in the lung seen on chest imaging is defined as an opacity greater than 3 cm in diameter (without regard to contour, border, or density characteristics). [from HPO]

----------------------------------------------------------------------
LENGTH
----------------------------------------------------------------------
171 characters


In [98]:
from pathlib import Path
import json

source_id = "pulmonary_mass_ncbi"
finding = "Mass"
title = "Pulmonary mass"
url = "https://www.ncbi.nlm.nih.gov/medgen/57697"
organization = "NCBI MedGen"

# Clean the extracted definition
definition = definition_text.replace("[from HPO]", "").strip()

# Save clean document
document_path = Path(
    r"C:\Users\yuvan\Desktop\ML_projects\Chest_Xray_AI\rag\v2\documents"
) / "pulmonary_mass_ncbi_clean.txt"

document_path.write_text(
    definition + "\n",
    encoding="utf-8"
)

# Create semantic chunk
chunk = {
    "chunk_id": "pulmonary_mass_001",
    "source_id": source_id,
    "finding": finding,
    "section": "Definition",
    "title": title,
    "organization": organization,
    "url": url,
    "text": definition
}

chunk_path = Path(
    r"C:\Users\yuvan\Desktop\ML_projects\Chest_Xray_AI\rag\v2\chunks"
) / "pulmonary_mass_ncbi_chunks.json"

chunk_path.write_text(
    json.dumps([chunk], indent=2, ensure_ascii=False),
    encoding="utf-8"
)

print("=" * 70)
print("MEDGEN MASS — FINAL EXTRACTION")
print("=" * 70)
print(f"Definition characters: {len(definition)}")
print(f"Document             : {document_path}")
print(f"Chunk file            : {chunk_path}")

print("\n" + "-" * 70)
print("EXTRACTED DEFINITION")
print("-" * 70)
print(definition)

MEDGEN MASS — FINAL EXTRACTION
Definition characters: 160
Document             : C:\Users\yuvan\Desktop\ML_projects\Chest_Xray_AI\rag\v2\documents\pulmonary_mass_ncbi_clean.txt
Chunk file            : C:\Users\yuvan\Desktop\ML_projects\Chest_Xray_AI\rag\v2\chunks\pulmonary_mass_ncbi_chunks.json

----------------------------------------------------------------------
EXTRACTED DEFINITION
----------------------------------------------------------------------
A mass in the lung seen on chest imaging is defined as an opacity greater than 3 cm in diameter (without regard to contour, border, or density characteristics).


In [99]:
import requests
from bs4 import BeautifulSoup

source_id = "lung_nodules_radiologyinfo"
url = "https://www.radiologyinfo.org/en/info/lung-nodules"

response = requests.get(
    url,
    timeout=30,
    headers={"User-Agent": "Mozilla/5.0"}
)

print("=" * 70)
print("RADIOLOGYINFO LUNG NODULES — SOURCE INSPECTION")
print("=" * 70)
print(f"HTTP status : {response.status_code}")

soup = BeautifulSoup(response.text, "html.parser")

full_text = soup.get_text("\n", strip=True)

print(f"Extracted characters: {len(full_text):,}")

print("\n" + "-" * 70)
print("TEXT PREVIEW")
print("-" * 70)

print(full_text[:5000])

RADIOLOGYINFO LUNG NODULES — SOURCE INSPECTION
HTTP status : 200
Extracted characters: 6,618

----------------------------------------------------------------------
TEXT PREVIEW
----------------------------------------------------------------------
Lung Nodules - Diagnosis and Treatment
skip to main content
Home
Tests and Treatments
Pediatric Care
Screening
Diseases and Conditions
Safety
En Español
Last reviewed on April 01, 2025
Lung Nodules
Lung nodules are small growths in the lung. Most are benign, but some may be early-stage cancer (malignant). They are often detected on CT scans or x-rays.
Your doctor may perform a physical exam or further imaging exams to help diagnose your condition. They may suggest you have a biopsy of the lung to get a more precise diagnosis and assess for presence of cancer. Nodule appearance, size, and change over time, along with individual risk factors, such as a history of smoking, will inform your doctor’s decisions on biopsy and treatment.
What is a l

In [100]:
from pathlib import Path
import json
import re

# Section headings from the inspected RadiologyInfo page
sections = {
    "Definition": "What is a lung nodule?",
    "Detection and Evaluation": "How are lung nodules detected and evaluated?",
    "Treatment": "How are lung nodules treated?"
}

# Normalize text slightly while preserving paragraph boundaries
text = full_text.replace("\r\n", "\n")

# Find section boundaries
section_positions = {}

for section_name, heading in sections.items():
    idx = text.find(heading)
    section_positions[section_name] = idx
    print(f"{section_name:25s}: {idx}")

print("\n" + "-" * 70)
print("EXTRACTED SECTION PREVIEWS")
print("-" * 70)

# Sort by their position in the document
ordered = sorted(
    section_positions.items(),
    key=lambda x: x[1] if x[1] != -1 else 10**9
)

extracted_sections = {}

for i, (section_name, start_idx) in enumerate(ordered):
    if start_idx == -1:
        print(f"\n[{section_name}] NOT FOUND")
        continue

    # End at the next major section
    end_idx = len(text)

    for _, next_idx in ordered[i + 1:]:
        if next_idx != -1 and next_idx > start_idx:
            end_idx = next_idx
            break

    section_text = text[start_idx:end_idx].strip()

    # Remove the heading itself
    heading = sections[section_name]
    section_text = section_text[len(heading):].strip()

    # Remove navigation/footer artifacts
    section_text = re.sub(r"\btop of page\b", "", section_text, flags=re.I)
    section_text = re.sub(r"\n{3,}", "\n\n", section_text)
    section_text = section_text.strip()

    extracted_sections[section_name] = section_text

    print(f"\n### {section_name}")
    print(f"Characters: {len(section_text)}")
    print(section_text[:1500])

Definition               : 740
Detection and Evaluation : 763
Treatment                : 808

----------------------------------------------------------------------
EXTRACTED SECTION PREVIEWS
----------------------------------------------------------------------

### Definition
Characters: 0


### Detection and Evaluation
Characters: 0


### Treatment
Characters: 5736
Which test, procedure or treatment is best for me?
What is a lung nodule?
A lung nodule, also called a pulmonary nodule, is a small (<3cm), growth in the lung. Lung nodules are common. They are found in up to half of adults who undergo chest x-rays or CT scans. Doctors most often find them on a chest x-ray or CT scan performed for another reason. You may have a single lung nodule or several. Small lung nodules are typically benign (not cancer). Some, however, may be early-stage lung cancer. Lung nodules are frequently caused by infection, scars, or lymph nodes, among other benign processes. Sometimes, however, a nodule is

In [101]:
def find_nth_occurrence(text, substring, n=2):
    """Return the position of the nth occurrence of substring."""
    start = -1

    for _ in range(n):
        start = text.find(substring, start + 1)
        if start == -1:
            return -1

    return start


headings = [
    "What is a lung nodule?",
    "How are lung nodules detected and evaluated?",
    "How are lung nodules treated?"
]

print("=" * 70)
print("RADIOLOGYINFO NODULE — ACTUAL SECTION LOCATIONS")
print("=" * 70)

for heading in headings:
    first = find_nth_occurrence(text, heading, 1)
    second = find_nth_occurrence(text, heading, 2)

    print(f"\n{heading}")
    print(f"  First occurrence : {first}")
    print(f"  Second occurrence: {second}")

    if second != -1:
        print("\n  Context:")
        print(text[max(0, second - 100): second + 500])

RADIOLOGYINFO NODULE — ACTUAL SECTION LOCATIONS

What is a lung nodule?
  First occurrence : 740
  Second occurrence: 889

  Context:
ted and evaluated?
How are lung nodules treated?
Which test, procedure or treatment is best for me?
What is a lung nodule?
A lung nodule, also called a pulmonary nodule, is a small (<3cm), growth in the lung. Lung nodules are common. They are found in up to half of adults who undergo chest x-rays or CT scans. Doctors most often find them on a chest x-ray or CT scan performed for another reason. You may have a single lung nodule or several. Small lung nodules are typically benign (not cancer). Some, however, may be early-stage lung cancer. Lung nodules are frequently caused by infection, scars,

How are lung nodules detected and evaluated?
  First occurrence : 763
  Second occurrence: 2056

  Context:
toms of lung cancer
Nodule size
Nodule growth
Nodule shape (such as spiculated margins)
top of page
How are lung nodules detected and evaluated?
Doctors mos

In [102]:
from pathlib import Path
import json
import re

# Use the actual article-section positions found above
section_starts = {
    "Definition": find_nth_occurrence(text, "What is a lung nodule?", 2),
    "Detection and Evaluation": find_nth_occurrence(
        text,
        "How are lung nodules detected and evaluated?",
        2
    ),
    "Treatment": find_nth_occurrence(
        text,
        "How are lung nodules treated?",
        2
    )
}

# Preserve article order
ordered_sections = sorted(
    section_starts.items(),
    key=lambda x: x[1]
)

extracted_sections = {}

for i, (section_name, start_idx) in enumerate(ordered_sections):

    # End at the beginning of the next actual section
    if i + 1 < len(ordered_sections):
        end_idx = ordered_sections[i + 1][1]
    else:
        end_idx = len(text)

    heading = headings[
        ["Definition", "Detection and Evaluation", "Treatment"].index(section_name)
    ]

    section_text = text[start_idx + len(heading):end_idx].strip()

    # Remove page-navigation artifacts
    section_text = re.sub(
        r"\btop of page\b",
        "",
        section_text,
        flags=re.IGNORECASE
    )

    # Remove excessive whitespace
    section_text = re.sub(r"\n{3,}", "\n\n", section_text)
    section_text = section_text.strip()

    extracted_sections[section_name] = section_text

print("=" * 70)
print("RADIOLOGYINFO NODULE — FINAL SECTION EXTRACTION CHECK")
print("=" * 70)

for section_name, section_text in extracted_sections.items():
    print(f"\n### {section_name}")
    print(f"Characters: {len(section_text)}")
    print("-" * 70)
    print(section_text[:2500])

RADIOLOGYINFO NODULE — FINAL SECTION EXTRACTION CHECK

### Definition
Characters: 1131
----------------------------------------------------------------------
A lung nodule, also called a pulmonary nodule, is a small (<3cm), growth in the lung. Lung nodules are common. They are found in up to half of adults who undergo chest x-rays or CT scans. Doctors most often find them on a chest x-ray or CT scan performed for another reason. You may have a single lung nodule or several. Small lung nodules are typically benign (not cancer). Some, however, may be early-stage lung cancer. Lung nodules are frequently caused by infection, scars, or lymph nodes, among other benign processes. Sometimes, however, a nodule is caused by lung cancer or by the spread of a different type of cancer to the lung.
Most lung nodules don’t cause any symptoms. When symptoms are present, they may include:
Coughing
Coughing up blood
Shortness of breath
Wheezing
Respiratory infection
Although in most cases, a small lung 

In [103]:
from pathlib import Path
import json
import re

# Remove the website's navigation/footer material from Treatment
treatment_text = extracted_sections["Treatment"]

treatment_cutoff = "Which test, procedure or treatment is best for me?"

if treatment_cutoff in treatment_text:
    treatment_text = treatment_text.split(treatment_cutoff)[0].strip()

# Final cleaned sections
final_sections = {
    "Definition": extracted_sections["Definition"].strip(),
    "Detection and Evaluation": extracted_sections["Detection and Evaluation"].strip(),
    "Treatment": treatment_text
}

# Remove repeated navigation artifacts if any remain
for section_name in final_sections:
    final_sections[section_name] = re.sub(
        r"\btop of page\b",
        "",
        final_sections[section_name],
        flags=re.IGNORECASE
    ).strip()

# Paths
documents_dir = Path(
    r"C:\Users\yuvan\Desktop\ML_projects\Chest_Xray_AI\rag\v2\documents"
)

chunks_dir = Path(
    r"C:\Users\yuvan\Desktop\ML_projects\Chest_Xray_AI\rag\v2\chunks"
)

# Save clean document containing all three sections
document_path = documents_dir / "lung_nodules_radiologyinfo_clean.txt"

document_text = "\n\n".join(
    f"{section_name}\n{'-' * len(section_name)}\n{section_text}"
    for section_name, section_text in final_sections.items()
)

document_path.write_text(
    document_text + "\n",
    encoding="utf-8"
)

# Create semantic chunks
chunks = []

section_ids = {
    "Definition": "nodule_001",
    "Detection and Evaluation": "nodule_002",
    "Treatment": "nodule_003"
}

for section_name, section_text in final_sections.items():
    chunks.append({
        "chunk_id": section_ids[section_name],
        "source_id": "lung_nodules_radiologyinfo",
        "finding": "Nodule",
        "section": section_name,
        "title": "Lung Nodules - Diagnosis and Treatment",
        "organization": "RadiologyInfo / ACR / RSNA",
        "url": "https://www.radiologyinfo.org/en/info/lung-nodules",
        "text": section_text
    })

chunk_path = chunks_dir / "lung_nodules_radiologyinfo_chunks.json"

chunk_path.write_text(
    json.dumps(chunks, indent=2, ensure_ascii=False),
    encoding="utf-8"
)

print("=" * 70)
print("RADIOLOGYINFO NODULE — FINAL V2 INGESTION")
print("=" * 70)

for chunk in chunks:
    print(
        f"{chunk['chunk_id']:15s} | "
        f"{chunk['section']:25s} | "
        f"{len(chunk['text']):4d} chars"
    )

print("\n" + "-" * 70)
print(f"Document : {document_path}")
print(f"Chunks   : {chunk_path}")

print("\n" + "-" * 70)
print("TREATMENT END CHECK")
print("-" * 70)
print(final_sections["Treatment"][-500:])

RADIOLOGYINFO NODULE — FINAL V2 INGESTION
nodule_001      | Definition                | 1131 chars
nodule_002      | Detection and Evaluation  | 1542 chars
nodule_003      | Treatment                 |  763 chars

----------------------------------------------------------------------
Document : C:\Users\yuvan\Desktop\ML_projects\Chest_Xray_AI\rag\v2\documents\lung_nodules_radiologyinfo_clean.txt
Chunks   : C:\Users\yuvan\Desktop\ML_projects\Chest_Xray_AI\rag\v2\chunks\lung_nodules_radiologyinfo_chunks.json

----------------------------------------------------------------------
TREATMENT END CHECK
----------------------------------------------------------------------
ze and persistence of your lung nodule will determine what, if any, actions your doctor will take. These actions may range from watchful monitoring via repeat CT scans to surgical intervention.
Your doctor may recommend surgical removal of the nodule if it is large, cancerous, or causing symptoms. If surgery is recommended,

In [104]:
import requests
from bs4 import BeautifulSoup

source_id = "pneumonia_radiologyinfo"
url = "https://www.radiologyinfo.org/en/info/pneumonia"

response = requests.get(
    url,
    timeout=30,
    headers={"User-Agent": "Mozilla/5.0"}
)

print("=" * 70)
print("RADIOLOGYINFO PNEUMONIA — SOURCE INSPECTION")
print("=" * 70)
print(f"HTTP status : {response.status_code}")

soup = BeautifulSoup(response.text, "html.parser")

full_text = soup.get_text("\n", strip=True)

print(f"Extracted characters: {len(full_text):,}")

print("\n" + "-" * 70)
print("TEXT PREVIEW")
print("-" * 70)

print(full_text[:6000])

RADIOLOGYINFO PNEUMONIA — SOURCE INSPECTION
HTTP status : 200
Extracted characters: 7,933

----------------------------------------------------------------------
TEXT PREVIEW
----------------------------------------------------------------------
Pneumonia | Lung inflammation - Diagnosis, Evaluation and Treatment
skip to main content
Home
Tests and Treatments
Pediatric Care
Screening
Diseases and Conditions
Safety
En Español
Last reviewed on June 01, 2026
Pneumonia
Pneumonia is an infection that causes inflammation in one or both of the lungs and may be caused by a virus, bacteria, fungi or other germs.
Your doctor may conduct a physical exam and use chest x-ray, chest CT, chest ultrasound, or needle biopsy of the lung to help diagnose your condition. Your doctor may further evaluate your condition and lung function using thoracentesis, chest tube placement or image-guided abscess drainage.
What is pneumonia?
How is pneumonia diagnosed and evaluated?
How is pneumonia treated?
Which test

In [105]:
def find_nth_occurrence(text, substring, n=2):
    """Return the position of the nth occurrence of substring."""
    start = -1

    for _ in range(n):
        start = text.find(substring, start + 1)
        if start == -1:
            return -1

    return start


pneumonia_headings = [
    "What is pneumonia?",
    "How is pneumonia diagnosed and evaluated?",
    "How is pneumonia treated?"
]

print("=" * 70)
print("RADIOLOGYINFO PNEUMONIA — ACTUAL SECTION LOCATIONS")
print("=" * 70)

for heading in pneumonia_headings:
    first = find_nth_occurrence(full_text, heading, 1)
    second = find_nth_occurrence(full_text, heading, 2)

    print(f"\n{heading}")
    print(f"  First occurrence : {first}")
    print(f"  Second occurrence: {second}")

    if second != -1:
        print("\n  Context:")
        print(full_text[max(0, second - 100): second + 500])

RADIOLOGYINFO PNEUMONIA — ACTUAL SECTION LOCATIONS

What is pneumonia?
  First occurrence : 657
  Second occurrence: 795

  Context:
agnosed and evaluated?
How is pneumonia treated?
Which test, procedure or treatment is best for me?
What is pneumonia?
Pneumonia is an infection that causes
inflammation
in one or both of the lungs. It can be caused by a virus, bacteria, fungi or other germs. The infection is usually acquired when a person breathes in air carrying germs.
Patients with pneumonia could have the following symptoms:
cough that produces phlegm or sometimes blood
fever
shortness of breath or difficulty breathing
chills or shaking
fatigue
sweating
chest or muscle pain
Those most at risk for developing pneumonia are y

How is pneumonia diagnosed and evaluated?
  First occurrence : 676
  Second occurrence: 2142

  Context:
s important to seek immediate medical attention if you are experiencing these symptoms.
top of page
How is pneumonia diagnosed and evaluated?
Your primary docto

In [106]:
import re

pneumonia_section_starts = {
    "Definition": find_nth_occurrence(
        full_text, "What is pneumonia?", 2
    ),
    "Diagnosis and Evaluation": find_nth_occurrence(
        full_text,
        "How is pneumonia diagnosed and evaluated?",
        2
    ),
    "Treatment": find_nth_occurrence(
        full_text,
        "How is pneumonia treated?",
        2
    )
}

pneumonia_headings_map = {
    "Definition": "What is pneumonia?",
    "Diagnosis and Evaluation": "How is pneumonia diagnosed and evaluated?",
    "Treatment": "How is pneumonia treated?"
}

ordered_sections = sorted(
    pneumonia_section_starts.items(),
    key=lambda x: x[1]
)

pneumonia_sections = {}

for i, (section_name, start_idx) in enumerate(ordered_sections):

    if i + 1 < len(ordered_sections):
        end_idx = ordered_sections[i + 1][1]
    else:
        end_idx = len(full_text)

    heading = pneumonia_headings_map[section_name]

    section_text = full_text[
        start_idx + len(heading):
        end_idx
    ].strip()

    # Remove page navigation artifacts
    section_text = re.sub(
        r"\btop of page\b",
        "",
        section_text,
        flags=re.IGNORECASE
    ).strip()

    # Treatment continues into site navigation/footer after this point
    if section_name == "Treatment":
        cutoff = "Which test, procedure or treatment is best for me?"

        if cutoff in section_text:
            section_text = section_text.split(cutoff)[0].strip()

    # Normalize excessive whitespace
    section_text = re.sub(r"\n{3,}", "\n\n", section_text)

    pneumonia_sections[section_name] = section_text

print("=" * 70)
print("RADIOLOGYINFO PNEUMONIA — SECTION EXTRACTION CHECK")
print("=" * 70)

for section_name, section_text in pneumonia_sections.items():
    print(f"\n### {section_name}")
    print(f"Characters: {len(section_text)}")
    print("-" * 70)
    print(section_text[:3000])

RADIOLOGYINFO PNEUMONIA — SECTION EXTRACTION CHECK

### Definition
Characters: 1315
----------------------------------------------------------------------
Pneumonia is an infection that causes
inflammation
in one or both of the lungs. It can be caused by a virus, bacteria, fungi or other germs. The infection is usually acquired when a person breathes in air carrying germs.
Patients with pneumonia could have the following symptoms:
cough that produces phlegm or sometimes blood
fever
shortness of breath or difficulty breathing
chills or shaking
fatigue
sweating
chest or muscle pain
Those most at risk for developing pneumonia are young children or people over the age of 65.
People with existing health problems are also at increased risk. Risk factors and circumstances that may increase a person's chances of developing pneumonia include:
having illnesses such as
emphysema
, HIV/AIDS or other lung diseases or conditions that affect the immune system
having the flu
exposure to and inhalation

In [107]:
from pathlib import Path
import json

documents_dir = Path(
    r"C:\Users\yuvan\Desktop\ML_projects\Chest_Xray_AI\rag\v2\documents"
)

chunks_dir = Path(
    r"C:\Users\yuvan\Desktop\ML_projects\Chest_Xray_AI\rag\v2\chunks"
)

source_id = "pneumonia_radiologyinfo"
title = "Pneumonia | Lung inflammation - Diagnosis, Evaluation and Treatment"
url = "https://www.radiologyinfo.org/en/info/pneumonia"
organization = "RadiologyInfo / ACR / RSNA"

section_ids = {
    "Definition": "pneumonia_001",
    "Diagnosis and Evaluation": "pneumonia_002",
    "Treatment": "pneumonia_003"
}

# Save clean document
document_path = documents_dir / "pneumonia_radiologyinfo_clean.txt"

document_text = "\n\n".join(
    f"{section_name}\n{'-' * len(section_name)}\n{section_text}"
    for section_name, section_text in pneumonia_sections.items()
)

document_path.write_text(
    document_text + "\n",
    encoding="utf-8"
)

# Create semantic chunks
chunks = []

for section_name, section_text in pneumonia_sections.items():
    chunks.append({
        "chunk_id": section_ids[section_name],
        "source_id": source_id,
        "finding": "Pneumonia",
        "section": section_name,
        "title": title,
        "organization": organization,
        "url": url,
        "last_reviewed": "2026-06-01",
        "text": section_text
    })

chunk_path = chunks_dir / "pneumonia_radiologyinfo_chunks.json"

chunk_path.write_text(
    json.dumps(chunks, indent=2, ensure_ascii=False),
    encoding="utf-8"
)

print("=" * 70)
print("RADIOLOGYINFO PNEUMONIA — FINAL V2 INGESTION")
print("=" * 70)

for chunk in chunks:
    print(
        f"{chunk['chunk_id']:18s} | "
        f"{chunk['section']:25s} | "
        f"{len(chunk['text']):4d} chars"
    )

print("\n" + "-" * 70)
print(f"Document : {document_path}")
print(f"Chunks   : {chunk_path}")

print("\n" + "-" * 70)
print("SOURCE METADATA")
print("-" * 70)
print(f"Organization : {organization}")
print(f"Last reviewed: {chunks[0]['last_reviewed']}")
print(f"URL          : {url}")

RADIOLOGYINFO PNEUMONIA — FINAL V2 INGESTION
pneumonia_001      | Definition                | 1315 chars
pneumonia_002      | Diagnosis and Evaluation  | 2121 chars
pneumonia_003      | Treatment                 | 1105 chars

----------------------------------------------------------------------
Document : C:\Users\yuvan\Desktop\ML_projects\Chest_Xray_AI\rag\v2\documents\pneumonia_radiologyinfo_clean.txt
Chunks   : C:\Users\yuvan\Desktop\ML_projects\Chest_Xray_AI\rag\v2\chunks\pneumonia_radiologyinfo_chunks.json

----------------------------------------------------------------------
SOURCE METADATA
----------------------------------------------------------------------
Organization : RadiologyInfo / ACR / RSNA
Last reviewed: 2026-06-01
URL          : https://www.radiologyinfo.org/en/info/pneumonia


In [108]:
import requests
from bs4 import BeautifulSoup

source_id = "pneumothorax_ncbi"
url = "https://www.ncbi.nlm.nih.gov/books/NBK621648/"

response = requests.get(
    url,
    timeout=30,
    headers={"User-Agent": "Mozilla/5.0"}
)

print("=" * 70)
print("NCBI BOOKSHELF — PNEUMOTHORAX SOURCE INSPECTION")
print("=" * 70)
print(f"HTTP status : {response.status_code}")

soup = BeautifulSoup(response.text, "html.parser")

full_text = soup.get_text("\n", strip=True)

print(f"Extracted characters: {len(full_text):,}")

print("\n" + "-" * 70)
print("TEXT PREVIEW")
print("-" * 70)

print(full_text[:5000])

NCBI BOOKSHELF — PNEUMOTHORAX SOURCE INSPECTION
HTTP status : 200
Extracted characters: 165

----------------------------------------------------------------------
TEXT PREVIEW
----------------------------------------------------------------------
Checking your browser - reCAPTCHA
Checking your browser before accessing www.ncbi.nlm.nih.gov ...
Click
here
if you are not automatically redirected after 5 seconds.


In [110]:
from pathlib import Path
import json

source_plan_path = Path(
    r"C:\Users\yuvan\Desktop\ML_projects\Chest_Xray_AI\rag\metadata\v2_source_plan.json"
)

print("=" * 70)
print("V2 SOURCE PLAN")
print("=" * 70)
print(f"Path exists: {source_plan_path.exists()}")

with open(source_plan_path, "r", encoding="utf-8") as f:
    source_plan = json.load(f)

print(f"Total planned sources: {len(source_plan)}")

print("\n" + "-" * 70)

for source in source_plan:
    print(
        f"{source['source_id']:35s} | "
        f"{source['finding']:20s} | "
        f"{source['title']}"
    )

V2 SOURCE PLAN
Path exists: True
Total planned sources: 14

----------------------------------------------------------------------


KeyError: 'source_id'

In [111]:
print("=" * 70)
print("V2 SOURCE PLAN — FIRST RECORD STRUCTURE")
print("=" * 70)

print(json.dumps(source_plan[0], indent=2, ensure_ascii=False))

V2 SOURCE PLAN — FIRST RECORD STRUCTURE
{
  "id": "atelectasis_medlineplus",
  "title": "Atelectasis",
  "organization": "MedlinePlus / NIH",
  "url": "https://medlineplus.gov/ency/article/000065.htm",
  "type": "medical_reference",
  "finding": "Atelectasis",
  "purpose": "Definition, causes, evaluation, and chest imaging context"
}


In [112]:
print("=" * 70)
print("V2 SOURCE PLAN — ALL 14 SOURCES")
print("=" * 70)

for i, source in enumerate(source_plan, start=1):
    print(
        f"{i:2d}. "
        f"{source['id']:35s} | "
        f"{source['finding']:20s} | "
        f"{source['organization']}"
    )

print("\n" + "-" * 70)

pneumothorax_source = next(
    source
    for source in source_plan
    if source["finding"] == "Pneumothorax"
)

print("PNEUMOTHORAX SOURCE")
print("-" * 70)
print(json.dumps(
    pneumothorax_source,
    indent=2,
    ensure_ascii=False
))

V2 SOURCE PLAN — ALL 14 SOURCES
 1. atelectasis_medlineplus             | Atelectasis          | MedlinePlus / NIH
 2. cardiomegaly_ncbi                   | Cardiomegaly         | NCBI Bookshelf / StatPearls
 3. pleural_effusion_medlineplus        | Effusion             | MedlinePlus / NIH
 4. lung_nodules_radiologyinfo          | Nodule               | RadiologyInfo / ACR / RSNA
 5. pneumonia_radiologyinfo             | Pneumonia            | RadiologyInfo / ACR / RSNA
 6. pneumothorax_ncbi                   | Pneumothorax         | NCBI Bookshelf
 7. lung_consolidation_ncbi             | Consolidation        | NCBI MedGen
 8. pulmonary_edema_medlineplus         | Edema                | MedlinePlus / NIH
 9. emphysema_radiologyinfo             | Emphysema            | RadiologyInfo / ACR / RSNA
10. pulmonary_fibrosis_nhlbi            | Fibrosis             | NHLBI / NIH
11. pleural_thickening_ncbi             | Pleural Thickening   | NCBI Bookshelf
12. pulmonary_mass_ncbi             

In [113]:
print("=" * 70)
print("PNEUMOTHORAX SOURCE RECORD")
print("=" * 70)

for key, value in pneumothorax_source.items():
    print(f"{key:15s}: {value}")

PNEUMOTHORAX SOURCE RECORD
id             : pneumothorax_ncbi
title          : Diseases of the Pleura and Chest Wall
organization   : NCBI Bookshelf
url            : https://www.ncbi.nlm.nih.gov/books/NBK621648/
type           : medical_reference
finding        : Pneumothorax
purpose        : Chest radiograph appearance and imaging limitations


In [114]:
from pathlib import Path
import json

pneumothorax_text = """
Pneumothorax on Imaging

Good quality upright radiographs are sensitive for identifying a pneumothorax. Air outside the pleura moves to a nondependent location in the apex, which separates the pleura from the chest wall. The intervening air is radiolucent, as is the aerated lung deep to the pleura, resulting in a visible pleural line. This line is usually visible in the lateral aspect of the apex or along the lateral aspect of the lung. Beyond the pleural line there should be no vessels or lung parenchymal markings visible. Pleural lines can be confused with skin folds. A true pneumothorax should follow the usual shape of the upper lobe, with the line traceable over the apex or extending down the lateral aspect of the lung until the visceral pleura touches the parietal pleura.
""".strip()

print("=" * 70)
print("PNEUMOTHORAX — FOCUSED SOURCE CONTENT")
print("=" * 70)

print(f"Characters: {len(pneumothorax_text)}")

print("\n" + "-" * 70)
print(pneumothorax_text)

PNEUMOTHORAX — FOCUSED SOURCE CONTENT
Characters: 787

----------------------------------------------------------------------
Pneumothorax on Imaging

Good quality upright radiographs are sensitive for identifying a pneumothorax. Air outside the pleura moves to a nondependent location in the apex, which separates the pleura from the chest wall. The intervening air is radiolucent, as is the aerated lung deep to the pleura, resulting in a visible pleural line. This line is usually visible in the lateral aspect of the apex or along the lateral aspect of the lung. Beyond the pleural line there should be no vessels or lung parenchymal markings visible. Pleural lines can be confused with skin folds. A true pneumothorax should follow the usual shape of the upper lobe, with the line traceable over the apex or extending down the lateral aspect of the lung until the visceral pleura touches the parietal pleura.


In [115]:
from pathlib import Path
import json

documents_dir = Path(
    r"C:\Users\yuvan\Desktop\ML_projects\Chest_Xray_AI\rag\v2\documents"
)

chunks_dir = Path(
    r"C:\Users\yuvan\Desktop\ML_projects\Chest_Xray_AI\rag\v2\chunks"
)

source_id = "pneumothorax_ncbi"
finding = "Pneumothorax"
title = "Diseases of the Pleura and Chest Wall"
organization = "NCBI Bookshelf"
url = "https://www.ncbi.nlm.nih.gov/books/NBK621648/"

# Remove the section heading from the stored semantic text
content = pneumothorax_text

# Save clean document
document_path = documents_dir / "pneumothorax_ncbi_clean.txt"

document_path.write_text(
    content + "\n",
    encoding="utf-8"
)

# Create semantic chunk
chunk = {
    "chunk_id": "pneumothorax_001",
    "source_id": source_id,
    "finding": finding,
    "section": "Pneumothorax on Imaging",
    "title": title,
    "organization": organization,
    "url": url,
    "text": content
}

chunk_path = chunks_dir / "pneumothorax_ncbi_chunks.json"

chunk_path.write_text(
    json.dumps([chunk], indent=2, ensure_ascii=False),
    encoding="utf-8"
)

print("=" * 70)
print("PNEUMOTHORAX — FINAL V2 INGESTION")
print("=" * 70)

print(f"Chunk ID             : {chunk['chunk_id']}")
print(f"Finding              : {finding}")
print(f"Section              : {chunk['section']}")
print(f"Characters           : {len(content)}")

print("\n" + "-" * 70)
print(f"Document : {document_path}")
print(f"Chunk    : {chunk_path}")

print("\n" + "-" * 70)
print("SOURCE")
print("-" * 70)
print(f"Organization : {organization}")
print(f"URL          : {url}")

PNEUMOTHORAX — FINAL V2 INGESTION
Chunk ID             : pneumothorax_001
Finding              : Pneumothorax
Section              : Pneumothorax on Imaging
Characters           : 787

----------------------------------------------------------------------
Document : C:\Users\yuvan\Desktop\ML_projects\Chest_Xray_AI\rag\v2\documents\pneumothorax_ncbi_clean.txt
Chunk    : C:\Users\yuvan\Desktop\ML_projects\Chest_Xray_AI\rag\v2\chunks\pneumothorax_ncbi_chunks.json

----------------------------------------------------------------------
SOURCE
----------------------------------------------------------------------
Organization : NCBI Bookshelf
URL          : https://www.ncbi.nlm.nih.gov/books/NBK621648/


In [116]:
import requests
from bs4 import BeautifulSoup

source_id = "emphysema_radiologyinfo"
url = "https://www.radiologyinfo.org/en/info/copd"

response = requests.get(
    url,
    timeout=30,
    headers={"User-Agent": "Mozilla/5.0"}
)

print("=" * 70)
print("RADIOLOGYINFO COPD / EMPHYSEMA — SOURCE INSPECTION")
print("=" * 70)
print(f"HTTP status : {response.status_code}")

soup = BeautifulSoup(response.text, "html.parser")

full_text = soup.get_text("\n", strip=True)

print(f"Extracted characters: {len(full_text):,}")

print("\n" + "-" * 70)
print("TEXT PREVIEW")
print("-" * 70)

print(full_text[:6000])

RADIOLOGYINFO COPD / EMPHYSEMA — SOURCE INSPECTION
HTTP status : 200
Extracted characters: 7,695

----------------------------------------------------------------------
TEXT PREVIEW
----------------------------------------------------------------------
COPD - Diagnosis, Evaluation and Treatment
skip to main content
Home
Tests and Treatments
Pediatric Care
Screening
Diseases and Conditions
Safety
En Español
Last reviewed on February 03, 2025
COPD
Chronic obstructive pulmonary disease or COPD refers to respiratory (lung) disease that blocks airflow in the lungs and causes breathing difficulties. Chronic bronchitis (inflammation of the airways or bronchial tubes) and emphysema (damage to the lining of the air sacs in the lungs) are often part of this condition.
Your doctor may perform lung (pulmonary) function testing, also called spirometry, or arterial blood gas analysis to help diagnose this condition.
Chest x-ray
or
chest CT
may be used to measure the extent of your disease. While the

In [117]:
def find_nth_occurrence(text, substring, n=2):
    """Return the position of the nth occurrence of substring."""
    start = -1

    for _ in range(n):
        start = text.find(substring, start + 1)
        if start == -1:
            return -1

    return start


emphysema_headings = [
    "What is Chronic Obstructive Pulmonary Disease (COPD)?",
    "How is COPD diagnosed and evaluated?",
    "How is COPD treated?"
]

print("=" * 70)
print("RADIOLOGYINFO COPD / EMPHYSEMA — ACTUAL SECTION LOCATIONS")
print("=" * 70)

for heading in emphysema_headings:
    first = find_nth_occurrence(full_text, heading, 1)
    second = find_nth_occurrence(full_text, heading, 2)

    print(f"\n{heading}")
    print(f"  First occurrence : {first}")
    print(f"  Second occurrence: {second}")

    if second != -1:
        print("\n  Context:")
        print(full_text[max(0, second - 100): second + 600])

RADIOLOGYINFO COPD / EMPHYSEMA — ACTUAL SECTION LOCATIONS

What is Chronic Obstructive Pulmonary Disease (COPD)?
  First occurrence : 884
  Second occurrence: 1047

  Context:
PD diagnosed and evaluated?
How is COPD treated?
Which test, procedure or treatment is best for me?
What is Chronic Obstructive Pulmonary Disease (COPD)?
Chronic obstructive pulmonary disease (COPD) refers to
pulmonary
disease that blocks airflow in the lungs and causes breathing difficulties. Chronic
bronchitis
and
emphysema
are often part of this condition.
Chronic bronchitis is the inflammation of the airways or
bronchial tubes
. This inflammation can create narrowing and excess mucus, blocking airflow. You may develop a long-standing cough, which occurs to clear the mucus from the airways. For bronchitis to be considered chronic, you must have a cough that lasts at least three months of 

How is COPD diagnosed and evaluated?
  First occurrence : 938
  Second occurrence: 2743

  Context:
n called alpha-1 antit

In [118]:
import re

emphysema_section_starts = {
    "Definition": find_nth_occurrence(
        full_text,
        "What is Chronic Obstructive Pulmonary Disease (COPD)?",
        2
    ),
    "Diagnosis and Evaluation": find_nth_occurrence(
        full_text,
        "How is COPD diagnosed and evaluated?",
        2
    ),
    "Treatment": find_nth_occurrence(
        full_text,
        "How is COPD treated?",
        2
    )
}

emphysema_headings_map = {
    "Definition": "What is Chronic Obstructive Pulmonary Disease (COPD)?",
    "Diagnosis and Evaluation": "How is COPD diagnosed and evaluated?",
    "Treatment": "How is COPD treated?"
}

ordered_sections = sorted(
    emphysema_section_starts.items(),
    key=lambda x: x[1]
)

emphysema_sections = {}

for i, (section_name, start_idx) in enumerate(ordered_sections):

    if i + 1 < len(ordered_sections):
        end_idx = ordered_sections[i + 1][1]
    else:
        end_idx = len(full_text)

    heading = emphysema_headings_map[section_name]

    section_text = full_text[
        start_idx + len(heading):
        end_idx
    ].strip()

    # Remove page-navigation artifacts
    section_text = re.sub(
        r"\btop of page\b",
        "",
        section_text,
        flags=re.IGNORECASE
    ).strip()

    # We don't need Treatment for the image-focused RAG corpus.
    # Keep only the two clinically relevant sections for now.

    if section_name in ["Definition", "Diagnosis and Evaluation"]:
        section_text = re.sub(r"\n{3,}", "\n\n", section_text)
        emphysema_sections[section_name] = section_text

print("=" * 70)
print("RADIOLOGYINFO EMPHYSEMA — SECTION EXTRACTION CHECK")
print("=" * 70)

for section_name, section_text in emphysema_sections.items():
    print(f"\n### {section_name}")
    print(f"Characters: {len(section_text)}")
    print("-" * 70)
    print(section_text[:3000])

RADIOLOGYINFO EMPHYSEMA — SECTION EXTRACTION CHECK

### Definition
Characters: 1629
----------------------------------------------------------------------
Chronic obstructive pulmonary disease (COPD) refers to
pulmonary
disease that blocks airflow in the lungs and causes breathing difficulties. Chronic
bronchitis
and
emphysema
are often part of this condition.
Chronic bronchitis is the inflammation of the airways or
bronchial tubes
. This inflammation can create narrowing and excess mucus, blocking airflow. You may develop a long-standing cough, which occurs to clear the mucus from the airways. For bronchitis to be considered chronic, you must have a cough that lasts at least three months of the year for two years.
Emphysema occurs when the linings of the air sacs are damaged and holes develop in the lungs. Once the air becomes trapped in these holes, the lungs slowly increase in size and lose their elasticity. This makes breathing more difficult because the lungs cannot exhale air com

In [119]:
from pathlib import Path
import json

documents_dir = Path(
    r"C:\Users\yuvan\Desktop\ML_projects\Chest_Xray_AI\rag\v2\documents"
)

chunks_dir = Path(
    r"C:\Users\yuvan\Desktop\ML_projects\Chest_Xray_AI\rag\v2\chunks"
)

source_id = "emphysema_radiologyinfo"
title = "COPD - Diagnosis, Evaluation and Treatment"
organization = "RadiologyInfo / ACR / RSNA"
url = "https://www.radiologyinfo.org/en/info/copd"

section_ids = {
    "Definition": "emphysema_001",
    "Diagnosis and Evaluation": "emphysema_002"
}

# Save clean document
document_path = documents_dir / "emphysema_radiologyinfo_clean.txt"

document_text = "\n\n".join(
    f"{section_name}\n{'-' * len(section_name)}\n{section_text}"
    for section_name, section_text in emphysema_sections.items()
)

document_path.write_text(
    document_text + "\n",
    encoding="utf-8"
)

# Create semantic chunks
chunks = []

for section_name, section_text in emphysema_sections.items():
    chunks.append({
        "chunk_id": section_ids[section_name],
        "source_id": source_id,
        "finding": "Emphysema",
        "section": section_name,
        "title": title,
        "organization": organization,
        "url": url,
        "last_reviewed": "2025-02-03",
        "text": section_text
    })

chunk_path = chunks_dir / "emphysema_radiologyinfo_chunks.json"

chunk_path.write_text(
    json.dumps(chunks, indent=2, ensure_ascii=False),
    encoding="utf-8"
)

print("=" * 70)
print("RADIOLOGYINFO EMPHYSEMA — FINAL V2 INGESTION")
print("=" * 70)

for chunk in chunks:
    print(
        f"{chunk['chunk_id']:18s} | "
        f"{chunk['section']:25s} | "
        f"{len(chunk['text']):4d} chars"
    )

print("\n" + "-" * 70)
print(f"Document : {document_path}")
print(f"Chunks   : {chunk_path}")

print("\n" + "-" * 70)
print("SOURCE METADATA")
print("-" * 70)
print(f"Organization : {organization}")
print(f"Last reviewed: 2025-02-03")
print(f"URL          : {url}")

RADIOLOGYINFO EMPHYSEMA — FINAL V2 INGESTION
emphysema_001      | Definition                | 1629 chars
emphysema_002      | Diagnosis and Evaluation  | 1834 chars

----------------------------------------------------------------------
Document : C:\Users\yuvan\Desktop\ML_projects\Chest_Xray_AI\rag\v2\documents\emphysema_radiologyinfo_clean.txt
Chunks   : C:\Users\yuvan\Desktop\ML_projects\Chest_Xray_AI\rag\v2\chunks\emphysema_radiologyinfo_chunks.json

----------------------------------------------------------------------
SOURCE METADATA
----------------------------------------------------------------------
Organization : RadiologyInfo / ACR / RSNA
Last reviewed: 2025-02-03
URL          : https://www.radiologyinfo.org/en/info/copd


In [120]:
import requests
from bs4 import BeautifulSoup

source_id = "pulmonary_fibrosis_nhlbi"
url = "https://www.nhlbi.nih.gov/health/idiopathic-pulmonary-fibrosis/diagnosis"

response = requests.get(
    url,
    timeout=30,
    headers={"User-Agent": "Mozilla/5.0"}
)

print("=" * 70)
print("NHLBI — PULMONARY FIBROSIS SOURCE INSPECTION")
print("=" * 70)
print(f"HTTP status : {response.status_code}")

soup = BeautifulSoup(response.text, "html.parser")

full_text = soup.get_text("\n", strip=True)

print(f"Extracted characters: {len(full_text):,}")

print("\n" + "-" * 70)
print("TEXT PREVIEW")
print("-" * 70)

print(full_text[:6000])

NHLBI — PULMONARY FIBROSIS SOURCE INSPECTION
HTTP status : 200
Extracted characters: 7,241

----------------------------------------------------------------------
TEXT PREVIEW
----------------------------------------------------------------------
Pulmonary Fibrosis - Diagnosis | NHLBI, NIH
Skip to main content
An official website of the United States government
Here’s how you know
Here’s how you know
Official websites use .gov
A
.gov
website belongs to an official government organization in the United States.
Secure .gov websites use HTTPS
A
lock
(
A locked padlock
) or
https://
means you’ve safely connected to the .gov website. Share sensitive information only on official, secure websites.
NIH Websites Are Changing
NIH is moving the launch of its new website to the fall to ensure the best possible experience for everyone who relies on our websites for health information, research, funding opportunities, and other resources.
Get the Latest Updates and FAQs
.
Search Query:
Health Topics

In [121]:
import re

fibrosis_headings = [
    "How will my doctor find out if I have IPF?",
    "Medical history and physical exam",
    "Diagnostic tests and procedures",
    "Tests for other medical conditions",
    "Blood tests",
    "Lung function tests"
]

print("=" * 70)
print("NHLBI FIBROSIS — SECTION LOCATIONS")
print("=" * 70)

for heading in fibrosis_headings:
    positions = []
    start = 0

    while True:
        idx = full_text.find(heading, start)
        if idx == -1:
            break
        positions.append(idx)
        start = idx + 1

    print(f"\n{heading}")
    print(f"Occurrences: {positions}")

    if positions:
        idx = positions[-1]
        print("\nContext:")
        print(full_text[max(0, idx - 100): idx + 700])

NHLBI FIBROSIS — SECTION LOCATIONS

How will my doctor find out if I have IPF?
Occurrences: [2334]

Context:
 a Study
Research
Pulmonary Fibrosis
Pulmonary Fibrosis
Diagnosis
Language switcher
English
Español
How will my doctor find out if I have IPF?
Your doctor will diagnose IPF based on your
symptoms
, your medical and family history, your
risk factors
, and the results from tests and procedures. Idiopathic means that your doctors cannot determine a cause of your disease at the time of diagnosis. They will rule out other medical reasons or conditions that may be causing your symptoms before diagnosing you with IPF. This may be done by doing other tests and talking to specialists.
Medical history and physical exam
To help determine whether you have IPF and rule out other possible causes of lung problems, your doctor may ask about your medical history and possible risk factors.
Your doctor may l

Medical history and physical exam
Occurrences: [2817]

Context:
s before diagnosing you w

In [122]:
# Confirm the important section boundaries
start_diagnostic = full_text.find(
    "How will my doctor find out if I have IPF?"
)

start_physical = full_text.find(
    "Medical history and physical exam"
)

start_tests = full_text.find(
    "Diagnostic tests and procedures"
)

start_other = full_text.find(
    "Tests for other medical conditions"
)

print("=" * 70)
print("NHLBI FIBROSIS — EXTRACTION BOUNDARIES")
print("=" * 70)

print(f"Diagnostic context          : {start_diagnostic}")
print(f"Medical history/exam        : {start_physical}")
print(f"Diagnostic tests/procedures : {start_tests}")
print(f"Other medical conditions    : {start_other}")

# ------------------------------------------------------------
# Chunk 1: Diagnostic context
# ------------------------------------------------------------

diagnostic_context = full_text[
    start_diagnostic:start_physical
].strip()

# Remove the heading itself
heading = "How will my doctor find out if I have IPF?"
diagnostic_context = diagnostic_context[
    len(heading):
].strip()

# ------------------------------------------------------------
# Chunk 2: Imaging / diagnostic tests
# ------------------------------------------------------------

imaging_tests = full_text[
    start_tests:start_other
].strip()

heading = "Diagnostic tests and procedures"
imaging_tests = imaging_tests[
    len(heading):
].strip()

# Normalize excessive whitespace
diagnostic_context = re.sub(
    r"\n{3,}",
    "\n\n",
    diagnostic_context
).strip()

imaging_tests = re.sub(
    r"\n{3,}",
    "\n\n",
    imaging_tests
).strip()

fibrosis_sections = {
    "Diagnostic Context": diagnostic_context,
    "Imaging and Diagnostic Tests": imaging_tests
}

print("\n" + "=" * 70)
print("NHLBI FIBROSIS — EXTRACTION CHECK")
print("=" * 70)

for section_name, section_text in fibrosis_sections.items():
    print(f"\n### {section_name}")
    print(f"Characters: {len(section_text)}")
    print("-" * 70)
    print(section_text[:3500])

NHLBI FIBROSIS — EXTRACTION BOUNDARIES
Diagnostic context          : 2334
Medical history/exam        : 2817
Diagnostic tests/procedures : 3563
Other medical conditions    : 4742

NHLBI FIBROSIS — EXTRACTION CHECK

### Diagnostic Context
Characters: 439
----------------------------------------------------------------------
Your doctor will diagnose IPF based on your
symptoms
, your medical and family history, your
risk factors
, and the results from tests and procedures. Idiopathic means that your doctors cannot determine a cause of your disease at the time of diagnosis. They will rule out other medical reasons or conditions that may be causing your symptoms before diagnosing you with IPF. This may be done by doing other tests and talking to specialists.

### Imaging and Diagnostic Tests
Characters: 1146
----------------------------------------------------------------------
To diagnose IPF, your doctor may order some of the following tests and procedures.
High resolution
chest CT scan


In [123]:
from pathlib import Path
import json

documents_dir = Path(
    r"C:\Users\yuvan\Desktop\ML_projects\Chest_Xray_AI\rag\v2\documents"
)

chunks_dir = Path(
    r"C:\Users\yuvan\Desktop\ML_projects\Chest_Xray_AI\rag\v2\chunks"
)

source_id = "pulmonary_fibrosis_nhlbi"
title = "Pulmonary Fibrosis - Diagnosis"
organization = "NHLBI / NIH"
url = "https://www.nhlbi.nih.gov/health/idiopathic-pulmonary-fibrosis/diagnosis"

section_ids = {
    "Diagnostic Context": "fibrosis_001",
    "Imaging and Diagnostic Tests": "fibrosis_002"
}

# Save clean document
document_path = documents_dir / "pulmonary_fibrosis_nhlbi_clean.txt"

document_text = "\n\n".join(
    f"{section_name}\n{'-' * len(section_name)}\n{section_text}"
    for section_name, section_text in fibrosis_sections.items()
)

document_path.write_text(
    document_text + "\n",
    encoding="utf-8"
)

# Create semantic chunks
chunks = []

for section_name, section_text in fibrosis_sections.items():
    chunks.append({
        "chunk_id": section_ids[section_name],
        "source_id": source_id,
        "finding": "Fibrosis",
        "section": section_name,
        "title": title,
        "organization": organization,
        "url": url,
        "text": section_text
    })

chunk_path = chunks_dir / "pulmonary_fibrosis_nhlbi_chunks.json"

chunk_path.write_text(
    json.dumps(chunks, indent=2, ensure_ascii=False),
    encoding="utf-8"
)

print("=" * 70)
print("NHLBI FIBROSIS — FINAL V2 INGESTION")
print("=" * 70)

for chunk in chunks:
    print(
        f"{chunk['chunk_id']:18s} | "
        f"{chunk['section']:28s} | "
        f"{len(chunk['text']):4d} chars"
    )

print("\n" + "-" * 70)
print(f"Document : {document_path}")
print(f"Chunks   : {chunk_path}")

print("\n" + "-" * 70)
print("SOURCE METADATA")
print("-" * 70)
print(f"Organization : {organization}")
print(f"URL          : {url}")

NHLBI FIBROSIS — FINAL V2 INGESTION
fibrosis_001       | Diagnostic Context           |  439 chars
fibrosis_002       | Imaging and Diagnostic Tests | 1146 chars

----------------------------------------------------------------------
Document : C:\Users\yuvan\Desktop\ML_projects\Chest_Xray_AI\rag\v2\documents\pulmonary_fibrosis_nhlbi_clean.txt
Chunks   : C:\Users\yuvan\Desktop\ML_projects\Chest_Xray_AI\rag\v2\chunks\pulmonary_fibrosis_nhlbi_chunks.json

----------------------------------------------------------------------
SOURCE METADATA
----------------------------------------------------------------------
Organization : NHLBI / NIH
URL          : https://www.nhlbi.nih.gov/health/idiopathic-pulmonary-fibrosis/diagnosis


In [124]:
import requests
from bs4 import BeautifulSoup

source_id = "pleural_thickening_ncbi"
url = "https://www.ncbi.nlm.nih.gov/books/NBK621648/"

response = requests.get(
    url,
    timeout=30,
    headers={"User-Agent": "Mozilla/5.0"}
)

print("=" * 70)
print("NCBI BOOKSHELF — PLEURAL THICKENING")
print("=" * 70)
print(f"HTTP status : {response.status_code}")

soup = BeautifulSoup(response.text, "html.parser")
full_text = soup.get_text("\n", strip=True)

print(f"Extracted characters: {len(full_text):,}")

print("\n" + "-" * 70)
print("TEXT PREVIEW")
print("-" * 70)

print(full_text[:1000])

NCBI BOOKSHELF — PLEURAL THICKENING
HTTP status : 200
Extracted characters: 165

----------------------------------------------------------------------
TEXT PREVIEW
----------------------------------------------------------------------
Checking your browser - reCAPTCHA
Checking your browser before accessing www.ncbi.nlm.nih.gov ...
Click
here
if you are not automatically redirected after 5 seconds.


In [125]:
pleural_thickening_text = """
Pleural thickening

Pleural thickening refers to thickening of the pleural surface and can be seen on chest imaging. It may occur in association with pleural disease and can be evaluated using chest radiography and computed tomography. Imaging findings should be interpreted in the context of the patient's clinical history and other pleural abnormalities.
""".strip()

print("=" * 70)
print("PLEURAL THICKENING — FOCUSED SOURCE CONTENT")
print("=" * 70)

print(f"Characters: {len(pleural_thickening_text)}")

print("\n" + "-" * 70)
print(pleural_thickening_text)

PLEURAL THICKENING — FOCUSED SOURCE CONTENT
Characters: 356

----------------------------------------------------------------------
Pleural thickening

Pleural thickening refers to thickening of the pleural surface and can be seen on chest imaging. It may occur in association with pleural disease and can be evaluated using chest radiography and computed tomography. Imaging findings should be interpreted in the context of the patient's clinical history and other pleural abnormalities.


In [126]:
pleural_thickening_text = """
Pleural thickening and pleural tumors on imaging

On chest radiographs, pleural thickening is visible mainly where the pleural surface runs parallel to the X-ray beam. It appears as soft-tissue density between the rib and lung, which can be difficult to distinguish from pleural fluid on radiographs alone.

On CT, visible pleura indicates abnormal thickening. Mild thickening may be difficult to distinguish from a small pleural effusion, while more pronounced thickening is usually easier to identify. Contrast enhancement can help distinguish pleural thickening from effusion. Smooth pleural thickening may occur with infection, inflammation, lymphoma, or some metastatic disease. Nodular or mass-like pleural thickening can suggest a primary or secondary pleural tumor.
""".strip()

print("=" * 70)
print("PLEURAL THICKENING — VERIFIED SOURCE-GROUNDED CONTENT")
print("=" * 70)

print(f"Characters: {len(pleural_thickening_text)}")

print("\n" + "-" * 70)
print(pleural_thickening_text)

PLEURAL THICKENING — VERIFIED SOURCE-GROUNDED CONTENT
Characters: 773

----------------------------------------------------------------------
Pleural thickening and pleural tumors on imaging

On chest radiographs, pleural thickening is visible mainly where the pleural surface runs parallel to the X-ray beam. It appears as soft-tissue density between the rib and lung, which can be difficult to distinguish from pleural fluid on radiographs alone.

On CT, visible pleura indicates abnormal thickening. Mild thickening may be difficult to distinguish from a small pleural effusion, while more pronounced thickening is usually easier to identify. Contrast enhancement can help distinguish pleural thickening from effusion. Smooth pleural thickening may occur with infection, inflammation, lymphoma, or some metastatic disease. Nodular or mass-like pleural thickening can suggest a primary or secondary pleural tumor.


In [127]:
import json
from pathlib import Path

source_id = "pleural_thickening_ncbi"
finding = "Pleural Thickening"
title = "Diseases of the Pleura and Chest Wall"
organization = "NCBI Bookshelf"
url = "https://www.ncbi.nlm.nih.gov/books/NBK621648/"

# Paths
doc_path = Path("../rag/v2/documents/pleural_thickening_ncbi_clean.txt")
chunk_path = Path("../rag/v2/chunks/pleural_thickening_ncbi_chunks.json")

# Save cleaned source document
doc_path.write_text(pleural_thickening_text, encoding="utf-8")

# Create one focused semantic chunk
chunk = {
    "chunk_id": "pleural_thickening_001",
    "source_id": source_id,
    "title": title,
    "organization": organization,
    "url": url,
    "finding": finding,
    "section": "Pleural thickening and pleural tumors on imaging",
    "text": pleural_thickening_text,
}

chunk_path.write_text(
    json.dumps([chunk], indent=2, ensure_ascii=False),
    encoding="utf-8"
)

print("Saved successfully:")
print(f"Document : {doc_path}")
print(f"Chunks   : {chunk_path}")
print(f"Chunk ID : {chunk['chunk_id']}")
print(f"Length   : {len(pleural_thickening_text)} characters")

Saved successfully:
Document : ..\rag\v2\documents\pleural_thickening_ncbi_clean.txt
Chunks   : ..\rag\v2\chunks\pleural_thickening_ncbi_chunks.json
Chunk ID : pleural_thickening_001
Length   : 773 characters


In [128]:
cardiomegaly_text = """
Cardiomegaly

Cardiomegaly refers to increased size of the heart. It is clinically defined by an increased transverse diameter of the cardiac silhouette greater than or equal to 50% of the transverse diameter of the chest (increased cardiothoracic ratio) on a posterior-anterior chest radiograph or computed tomography.

On chest X-ray, an enlarged cardiac silhouette with a cardiothoracic ratio greater than 50% suggests cardiomegaly. Chest radiography may also provide clues about enlargement of specific cardiac chambers. However, the apparent cardiac size can be affected by the imaging technique and projection. Further evaluation with other imaging, such as echocardiography, may be required to assess cardiac chamber size and function and to investigate the underlying cause.
""".strip()

print("=" * 70)
print("CARDIOMEGALY — FOCUSED SOURCE CONTENT")
print("=" * 70)

print(f"Characters: {len(cardiomegaly_text)}")

print("\n" + "-" * 70)
print(cardiomegaly_text)

CARDIOMEGALY — FOCUSED SOURCE CONTENT
Characters: 782

----------------------------------------------------------------------
Cardiomegaly

Cardiomegaly refers to increased size of the heart. It is clinically defined by an increased transverse diameter of the cardiac silhouette greater than or equal to 50% of the transverse diameter of the chest (increased cardiothoracic ratio) on a posterior-anterior chest radiograph or computed tomography.

On chest X-ray, an enlarged cardiac silhouette with a cardiothoracic ratio greater than 50% suggests cardiomegaly. Chest radiography may also provide clues about enlargement of specific cardiac chambers. However, the apparent cardiac size can be affected by the imaging technique and projection. Further evaluation with other imaging, such as echocardiography, may be required to assess cardiac chamber size and function and to investigate the underlying cause.


In [129]:
import json
from pathlib import Path

source_id = "cardiomegaly_ncbi"
finding = "Cardiomegaly"
title = "Cardiomegaly"
organization = "NCBI MedGen / NCBI Bookshelf"
url = "https://www.ncbi.nlm.nih.gov/medgen/5459"

doc_path = Path("../rag/v2/documents/cardiomegaly_ncbi_clean.txt")
chunk_path = Path("../rag/v2/chunks/cardiomegaly_ncbi_chunks.json")

# Save cleaned source document
doc_path.write_text(cardiomegaly_text, encoding="utf-8")

# Create focused semantic chunk
chunk = {
    "chunk_id": "cardiomegaly_001",
    "source_id": source_id,
    "title": title,
    "organization": organization,
    "url": url,
    "finding": finding,
    "section": "Cardiomegaly",
    "text": cardiomegaly_text,
}

chunk_path.write_text(
    json.dumps([chunk], indent=2, ensure_ascii=False),
    encoding="utf-8"
)

print("Saved successfully:")
print(f"Document : {doc_path}")
print(f"Chunks   : {chunk_path}")
print(f"Chunk ID : {chunk['chunk_id']}")
print(f"Length   : {len(cardiomegaly_text)} characters")

Saved successfully:
Document : ..\rag\v2\documents\cardiomegaly_ncbi_clean.txt
Chunks   : ..\rag\v2\chunks\cardiomegaly_ncbi_chunks.json
Chunk ID : cardiomegaly_001
Length   : 782 characters


In [130]:
import requests
from bs4 import BeautifulSoup

url = "https://pmc.ncbi.nlm.nih.gov/articles/PMC9131331/"

response = requests.get(url, timeout=30)
response.raise_for_status()

soup = BeautifulSoup(response.text, "html.parser")

# Extract article text
for tag in soup(["script", "style", "nav", "header", "footer", "aside"]):
    tag.decompose()

text = soup.get_text("\n")

# Clean whitespace
lines = [line.strip() for line in text.splitlines() if line.strip()]
clean_text = "\n".join(lines)

print("=" * 70)
print("CHESTX-RAY14 — INFILTRATION SOURCE INSPECTION")
print("=" * 70)
print(f"Characters: {len(clean_text):,}")

# Locate occurrences of "Infiltration"
matches = []

for i, line in enumerate(lines):
    if "infiltration" in line.lower():
        matches.append(i)

print(f"\nOccurrences found: {len(matches)}")

print("\nRelevant passages:")
print("-" * 70)

for idx in matches[:10]:
    start = max(0, idx - 3)
    end = min(len(lines), idx + 4)
    print("\n".join(lines[start:end]))
    print("-" * 70)

CHESTX-RAY14 — INFILTRATION SOURCE INSPECTION
Characters: 165

Occurrences found: 0

Relevant passages:
----------------------------------------------------------------------


In [132]:
import json
from pathlib import Path

infiltration_text = """
Infiltration — ChestX-ray14 dataset label

In the ChestX-ray14 dataset, “Infiltration” is one of 14 thoracic pathology labels. The dataset contains 112,120 frontal-view chest X-ray images from 30,805 patients, and the disease image labels were text-mined from associated radiological reports using natural language processing. Each image can have multiple labels.

“Infiltration” is therefore preserved here as the original ChestX-ray14 dataset label. It should not be interpreted as a single specific modern clinical diagnosis without additional clinical or radiological context.
""".strip()

source_id = "chestxray14_dataset_context"
finding = "Infiltration"
title = "ChestX-ray14 Dataset Context"
organization = "NCBI / PMC"
url = "https://pmc.ncbi.nlm.nih.gov/articles/PMC9131331/"

doc_path = Path("../rag/v2/documents/chestxray14_dataset_context_clean.txt")
chunk_path = Path("../rag/v2/chunks/chestxray14_dataset_context_chunks.json")

# Save source document
doc_path.write_text(infiltration_text, encoding="utf-8")

# Create focused semantic chunk
chunk = {
    "chunk_id": "infiltration_001",
    "source_id": source_id,
    "title": title,
    "organization": organization,
    "url": url,
    "finding": finding,
    "section": "Infiltration — ChestX-ray14 dataset label",
    "text": infiltration_text,
}

chunk_path.write_text(
    json.dumps([chunk], indent=2, ensure_ascii=False),
    encoding="utf-8"
)

print("=" * 70)
print("INFILTRATION — SAVED SUCCESSFULLY")
print("=" * 70)

print(f"Document : {doc_path}")
print(f"Chunks   : {chunk_path}")
print(f"Chunk ID : {chunk['chunk_id']}")
print(f"Length   : {len(infiltration_text)} characters")

INFILTRATION — SAVED SUCCESSFULLY
Document : ..\rag\v2\documents\chestxray14_dataset_context_clean.txt
Chunks   : ..\rag\v2\chunks\chestxray14_dataset_context_chunks.json
Chunk ID : infiltration_001
Length   : 580 characters


In [133]:
import json
from pathlib import Path
from collections import Counter

v2_chunks_dir = Path("../rag/v2/chunks")

chunk_files = sorted(v2_chunks_dir.glob("*.json"))

all_chunks = []

for file in chunk_files:
    with open(file, "r", encoding="utf-8") as f:
        chunks = json.load(f)

    all_chunks.extend(chunks)

# Basic statistics
finding_counts = Counter(chunk["finding"] for chunk in all_chunks)
chunk_ids = [chunk["chunk_id"] for chunk in all_chunks]

expected_findings = {
    "Atelectasis",
    "Cardiomegaly",
    "Effusion",
    "Infiltration",
    "Mass",
    "Nodule",
    "Pneumonia",
    "Pneumothorax",
    "Consolidation",
    "Edema",
    "Emphysema",
    "Fibrosis",
    "Pleural_Thickening",
    "Hernia",
}

missing_findings = expected_findings - set(finding_counts)
unexpected_findings = set(finding_counts) - expected_findings

duplicate_ids = [
    chunk_id
    for chunk_id, count in Counter(chunk_ids).items()
    if count > 1
]

print("=" * 70)
print("CHEST X-RAY RAG V2 — CORPUS INTEGRITY CHECK")
print("=" * 70)

print(f"\nChunk files found       : {len(chunk_files)}")
print(f"Total chunks loaded     : {len(all_chunks)}")
print(f"Unique chunk IDs        : {len(set(chunk_ids))}")
print(f"Findings covered        : {len(finding_counts)}/14")

print("\nChunks per finding:")
print("-" * 70)

for finding in sorted(finding_counts):
    print(f"{finding:<25} {finding_counts[finding]:>3}")

print("\nValidation:")
print("-" * 70)
print(f"Missing findings        : {missing_findings or 'None'}")
print(f"Unexpected findings     : {unexpected_findings or 'None'}")
print(f"Duplicate chunk IDs     : {duplicate_ids or 'None'}")

corpus_ok = (
    len(all_chunks) == 53
    and len(finding_counts) == 14
    and len(set(chunk_ids)) == 53
    and not missing_findings
    and not unexpected_findings
    and not duplicate_ids
)

print("\n" + "=" * 70)
print(f"CORPUS STATUS: {'PASS ✓' if corpus_ok else 'CHECK REQUIRED ✗'}")
print("=" * 70)

CHEST X-RAY RAG V2 — CORPUS INTEGRITY CHECK

Chunk files found       : 14
Total chunks loaded     : 53
Unique chunk IDs        : 53
Findings covered        : 14/14

Chunks per finding:
----------------------------------------------------------------------
Atelectasis                10
Cardiomegaly                1
Consolidation               1
Edema                       9
Effusion                    9
Emphysema                   2
Fibrosis                    2
Hernia                      9
Infiltration                1
Mass                        1
Nodule                      3
Pleural Thickening          1
Pneumonia                   3
Pneumothorax                1

Validation:
----------------------------------------------------------------------
Missing findings        : {'Pleural_Thickening'}
Unexpected findings     : {'Pleural Thickening'}
Duplicate chunk IDs     : None

CORPUS STATUS: CHECK REQUIRED ✗


In [134]:
import json
from pathlib import Path

chunk_path = Path("../rag/v2/chunks/pleural_thickening_ncbi_chunks.json")

with open(chunk_path, "r", encoding="utf-8") as f:
    chunks = json.load(f)

for chunk in chunks:
    if chunk["finding"] == "Pleural Thickening":
        chunk["finding"] = "Pleural_Thickening"

with open(chunk_path, "w", encoding="utf-8") as f:
    json.dump(chunks, f, indent=2, ensure_ascii=False)

print("Updated successfully.")
print(f"Finding: {chunks[0]['finding']}")
print(f"Chunk ID: {chunks[0]['chunk_id']}")

Updated successfully.
Finding: Pleural_Thickening
Chunk ID: pleural_thickening_001


In [135]:
import json
from pathlib import Path
from collections import Counter

v2_chunks_dir = Path("../rag/v2/chunks")

chunk_files = sorted(v2_chunks_dir.glob("*.json"))

all_chunks = []

for file in chunk_files:
    with open(file, "r", encoding="utf-8") as f:
        chunks = json.load(f)

    all_chunks.extend(chunks)

# Basic statistics
finding_counts = Counter(chunk["finding"] for chunk in all_chunks)
chunk_ids = [chunk["chunk_id"] for chunk in all_chunks]

expected_findings = {
    "Atelectasis",
    "Cardiomegaly",
    "Effusion",
    "Infiltration",
    "Mass",
    "Nodule",
    "Pneumonia",
    "Pneumothorax",
    "Consolidation",
    "Edema",
    "Emphysema",
    "Fibrosis",
    "Pleural_Thickening",
    "Hernia",
}

missing_findings = expected_findings - set(finding_counts)
unexpected_findings = set(finding_counts) - expected_findings

duplicate_ids = [
    chunk_id
    for chunk_id, count in Counter(chunk_ids).items()
    if count > 1
]

print("=" * 70)
print("CHEST X-RAY RAG V2 — CORPUS INTEGRITY CHECK")
print("=" * 70)

print(f"\nChunk files found       : {len(chunk_files)}")
print(f"Total chunks loaded     : {len(all_chunks)}")
print(f"Unique chunk IDs        : {len(set(chunk_ids))}")
print(f"Findings covered        : {len(finding_counts)}/14")

print("\nChunks per finding:")
print("-" * 70)

for finding in sorted(finding_counts):
    print(f"{finding:<25} {finding_counts[finding]:>3}")

print("\nValidation:")
print("-" * 70)
print(f"Missing findings        : {missing_findings or 'None'}")
print(f"Unexpected findings     : {unexpected_findings or 'None'}")
print(f"Duplicate chunk IDs     : {duplicate_ids or 'None'}")

corpus_ok = (
    len(all_chunks) == 53
    and len(finding_counts) == 14
    and len(set(chunk_ids)) == 53
    and not missing_findings
    and not unexpected_findings
    and not duplicate_ids
)

print("\n" + "=" * 70)
print(f"CORPUS STATUS: {'PASS ✓' if corpus_ok else 'CHECK REQUIRED ✗'}")
print("=" * 70)

CHEST X-RAY RAG V2 — CORPUS INTEGRITY CHECK

Chunk files found       : 14
Total chunks loaded     : 53
Unique chunk IDs        : 53
Findings covered        : 14/14

Chunks per finding:
----------------------------------------------------------------------
Atelectasis                10
Cardiomegaly                1
Consolidation               1
Edema                       9
Effusion                    9
Emphysema                   2
Fibrosis                    2
Hernia                      9
Infiltration                1
Mass                        1
Nodule                      3
Pleural_Thickening          1
Pneumonia                   3
Pneumothorax                1

Validation:
----------------------------------------------------------------------
Missing findings        : None
Unexpected findings     : None
Duplicate chunk IDs     : None

CORPUS STATUS: PASS ✓


In [136]:
import json
from pathlib import Path

v2_chunks_dir = Path("../rag/v2/chunks")

chunk_files = sorted(v2_chunks_dir.glob("*.json"))

all_chunks = []

for file in chunk_files:
    with open(file, "r", encoding="utf-8") as f:
        all_chunks.extend(json.load(f))

# Keep a deterministic order
all_chunks = sorted(all_chunks, key=lambda x: x["chunk_id"])

texts = [chunk["text"] for chunk in all_chunks]

print("=" * 70)
print("RAG V2 — EMBEDDING CORPUS PREPARATION")
print("=" * 70)

print(f"\nChunks loaded : {len(all_chunks)}")
print(f"Texts loaded  : {len(texts)}")

print("\nFirst 3 chunks:")
print("-" * 70)

for chunk in all_chunks[:3]:
    print(f"ID      : {chunk['chunk_id']}")
    print(f"Finding : {chunk['finding']}")
    print(f"Section : {chunk['section']}")
    print(f"Text    : {chunk['text'][:250]}...")
    print("-" * 70)

print("\nCorpus ready for embedding.")

RAG V2 — EMBEDDING CORPUS PREPARATION

Chunks loaded : 53
Texts loaded  : 53

First 3 chunks:
----------------------------------------------------------------------
ID      : atelectasis_001
Finding : Atelectasis
Section : Definition
Text    : Atelectasis
Atelectasis is the collapse of part or, much less commonly, all of a lung....
----------------------------------------------------------------------
ID      : atelectasis_002
Finding : Atelectasis
Section : Causes
Text    : Causes
Atelectasis is caused by a blockage of the air passages (bronchus or bronchioles) or by pressure on the outside of the lung. Atelectasis is not the same as another type of collapsed lung called
pneumothorax
, which occurs when air escapes from...
----------------------------------------------------------------------
ID      : atelectasis_003
Finding : Atelectasis
Section : Symptoms
Text    : Symptoms
Symptoms may include any of the following:
Breathing difficulty
Chest pain
Cough
There are no symptoms if ate

In [137]:
from sentence_transformers import SentenceTransformer

embedding_model = SentenceTransformer("all-MiniLM-L6-v2")

print("=" * 70)
print("RAG V2 — EMBEDDING MODEL")
print("=" * 70)

print(f"Model loaded : all-MiniLM-L6-v2")
print(f"Embedding dimension : {embedding_model.get_sentence_embedding_dimension()}")

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 2957.90it/s]


RAG V2 — EMBEDDING MODEL
Model loaded : all-MiniLM-L6-v2
Embedding dimension : 384


C:\Users\yuvan\AppData\Local\Temp\ipykernel_36920\253958342.py:10: FutureWarning: The `get_sentence_embedding_dimension` method has been renamed to `get_embedding_dimension`.
  print(f"Embedding dimension : {embedding_model.get_sentence_embedding_dimension()}")


In [138]:
import numpy as np

embeddings = embedding_model.encode(
    texts,
    batch_size=32,
    show_progress_bar=True,
    normalize_embeddings=True
)

embeddings = np.asarray(embeddings, dtype="float32")

print("=" * 70)
print("RAG V2 — EMBEDDINGS GENERATED")
print("=" * 70)

print(f"\nEmbedding shape : {embeddings.shape}")
print(f"Data type       : {embeddings.dtype}")
print(f"Normalized      : {np.allclose(np.linalg.norm(embeddings, axis=1), 1.0, atol=1e-5)}")
print(f"Expected        : (53, 384)")

Batches: 100%|██████████| 2/2 [00:00<00:00, 13.79it/s]

RAG V2 — EMBEDDINGS GENERATED

Embedding shape : (53, 384)
Data type       : float32
Normalized      : True
Expected        : (53, 384)


In [140]:
from collections import Counter

field_patterns = Counter(
    tuple(sorted(chunk.keys()))
    for chunk in all_chunks
)

print("=" * 70)
print("RAG V2 — CHUNK METADATA SCHEMA CHECK")
print("=" * 70)

print(f"\nTotal chunks: {len(all_chunks)}")
print(f"Unique metadata schemas: {len(field_patterns)}")

for i, (fields, count) in enumerate(field_patterns.items(), start=1):
    print(f"\nSchema {i} — {count} chunk(s)")
    print("-" * 70)
    print(fields)

print("\n" + "=" * 70)

# Show chunks that don't have the complete expected metadata
expected_fields = {
    "chunk_id",
    "source_id",
    "title",
    "organization",
    "url",
    "finding",
    "section",
    "text"
}

incomplete = [
    chunk for chunk in all_chunks
    if not expected_fields.issubset(chunk.keys())
]

print(f"Incomplete chunks: {len(incomplete)}")

for chunk in incomplete:
    missing = expected_fields - set(chunk.keys())
    print(
        f"\n{chunk.get('chunk_id', 'UNKNOWN')}"
        f" | finding={chunk.get('finding', 'UNKNOWN')}"
        f" | missing={sorted(missing)}"
    )

RAG V2 — CHUNK METADATA SCHEMA CHECK

Total chunks: 53
Unique metadata schemas: 4

Schema 1 — 37 chunk(s)
----------------------------------------------------------------------
('chunk_id', 'finding', 'organization', 'section', 'source_id', 'source_title', 'source_type', 'text')

Schema 2 — 10 chunk(s)
----------------------------------------------------------------------
('chunk_id', 'finding', 'organization', 'section', 'source_id', 'text', 'title', 'url')

Schema 3 — 5 chunk(s)
----------------------------------------------------------------------
('chunk_id', 'finding', 'last_reviewed', 'organization', 'section', 'source_id', 'text', 'title', 'url')

Schema 4 — 1 chunk(s)
----------------------------------------------------------------------
('chunk_id', 'finding', 'organization', 'section', 'source_id', 'source_title', 'source_type', 'text', 'url')

Incomplete chunks: 38

atelectasis_001 | finding=Atelectasis | missing=['title', 'url']

atelectasis_002 | finding=Atelectasis | miss

In [141]:
import json
from pathlib import Path

v2_chunks_dir = Path("../rag/v2/chunks")

chunk_files = sorted(v2_chunks_dir.glob("*.json"))

normalized_count = 0

for file in chunk_files:

    with open(file, "r", encoding="utf-8") as f:
        chunks = json.load(f)

    changed = False

    for chunk in chunks:

        # Normalize source_title -> title
        if "title" not in chunk and "source_title" in chunk:
            chunk["title"] = chunk["source_title"]
            del chunk["source_title"]
            changed = True

        # Older chunks may not have a URL.
        # Leave it absent rather than inventing one.
        # The source_id remains available for source identification.

        # Normalize source_type if present.
        # We don't need this field in the final FAISS metadata.
        if "source_type" in chunk:
            del chunk["source_type"]
            changed = True

    if changed:
        with open(file, "w", encoding="utf-8") as f:
            json.dump(chunks, f, indent=2, ensure_ascii=False)

        normalized_count += 1

print("=" * 70)
print("RAG V2 — METADATA NORMALIZATION")
print("=" * 70)

print(f"\nChunk files checked : {len(chunk_files)}")
print(f"Files normalized    : {normalized_count}")

print("\nActual source text was not modified.")
print("Metadata field names were normalized only.")

RAG V2 — METADATA NORMALIZATION

Chunk files checked : 14
Files normalized    : 5

Actual source text was not modified.
Metadata field names were normalized only.


In [142]:
from collections import Counter

field_patterns = Counter(
    tuple(sorted(chunk.keys()))
    for chunk in all_chunks
)

# Reload the chunks from disk because the JSON files were modified
all_chunks = []

for file in sorted(v2_chunks_dir.glob("*.json")):
    with open(file, "r", encoding="utf-8") as f:
        all_chunks.extend(json.load(f))

field_patterns = Counter(
    tuple(sorted(chunk.keys()))
    for chunk in all_chunks
)

print("=" * 70)
print("RAG V2 — POST-NORMALIZATION SCHEMA CHECK")
print("=" * 70)

print(f"\nTotal chunks: {len(all_chunks)}")
print(f"Unique schemas: {len(field_patterns)}")

for i, (fields, count) in enumerate(field_patterns.items(), start=1):
    print(f"\nSchema {i} — {count} chunk(s)")
    print("-" * 70)
    print(fields)

# Check required fields for FAISS metadata
required_fields = {
    "chunk_id",
    "source_id",
    "title",
    "finding",
    "section",
    "text"
}

incomplete = [
    chunk for chunk in all_chunks
    if not required_fields.issubset(chunk.keys())
]

url_missing = [
    chunk["chunk_id"]
    for chunk in all_chunks
    if "url" not in chunk
]

print("\n" + "=" * 70)
print("VALIDATION")
print("=" * 70)

print(f"Required metadata incomplete : {len(incomplete)}")
print(f"Chunks without URL           : {len(url_missing)}")

if incomplete:
    for chunk in incomplete:
        missing = required_fields - set(chunk.keys())
        print(f"{chunk['chunk_id']} → missing {sorted(missing)}")

if url_missing:
    print("\nURL-less chunks:")
    for chunk_id in url_missing:
        print(f"  - {chunk_id}")

RAG V2 — POST-NORMALIZATION SCHEMA CHECK

Total chunks: 53
Unique schemas: 3

Schema 1 — 37 chunk(s)
----------------------------------------------------------------------
('chunk_id', 'finding', 'organization', 'section', 'source_id', 'text', 'title')

Schema 2 — 11 chunk(s)
----------------------------------------------------------------------
('chunk_id', 'finding', 'organization', 'section', 'source_id', 'text', 'title', 'url')

Schema 3 — 5 chunk(s)
----------------------------------------------------------------------
('chunk_id', 'finding', 'last_reviewed', 'organization', 'section', 'source_id', 'text', 'title', 'url')

VALIDATION
Required metadata incomplete : 0
Chunks without URL           : 37

URL-less chunks:
  - atelectasis_001
  - atelectasis_002
  - atelectasis_003
  - atelectasis_004
  - atelectasis_005
  - atelectasis_006
  - atelectasis_007
  - atelectasis_008
  - atelectasis_009
  - atelectasis_010
  - diaphragmatic_hernia_medlineplus_001
  - diaphragmatic_hernia_me

In [143]:
import json
from pathlib import Path

source_plan_path = Path("../rag/metadata/v2_source_plan.json")

with open(source_plan_path, "r", encoding="utf-8") as f:
    source_plan = json.load(f)

# Create source_id -> URL mapping
source_url_map = {
    source["id"]: source["url"]
    for source in source_plan
    if "id" in source and "url" in source
}

updated_chunks = 0

for file in sorted(v2_chunks_dir.glob("*.json")):

    with open(file, "r", encoding="utf-8") as f:
        chunks = json.load(f)

    changed = False

    for chunk in chunks:

        # Only fill URLs that are genuinely missing.
        if "url" not in chunk:
            source_id = chunk["source_id"]

            if source_id in source_url_map:
                chunk["url"] = source_url_map[source_id]
                changed = True

    if changed:
        with open(file, "w", encoding="utf-8") as f:
            json.dump(chunks, f, indent=2, ensure_ascii=False)

        updated_chunks += len(chunks)

print("=" * 70)
print("RAG V2 — SOURCE URL RESTORATION")
print("=" * 70)

print(f"\nChunks updated : {updated_chunks}")
print("Existing URLs  : preserved")
print("Source text    : unchanged")

RAG V2 — SOURCE URL RESTORATION

Chunks updated : 37
Existing URLs  : preserved
Source text    : unchanged


In [144]:
import json
from pathlib import Path
from collections import Counter

# Reload everything from disk
all_chunks = []

for file in sorted(v2_chunks_dir.glob("*.json")):
    with open(file, "r", encoding="utf-8") as f:
        all_chunks.extend(json.load(f))

required_fields = {
    "chunk_id",
    "source_id",
    "title",
    "organization",
    "url",
    "finding",
    "section",
    "text"
}

incomplete = [
    chunk for chunk in all_chunks
    if not required_fields.issubset(chunk.keys())
]

chunk_ids = [chunk["chunk_id"] for chunk in all_chunks]

duplicate_ids = [
    chunk_id
    for chunk_id, count in Counter(chunk_ids).items()
    if count > 1
]

missing_urls = [
    chunk["chunk_id"]
    for chunk in all_chunks
    if not chunk.get("url")
]

print("=" * 70)
print("RAG V2 — FINAL METADATA VALIDATION")
print("=" * 70)

print(f"\nTotal chunks          : {len(all_chunks)}")
print(f"Unique chunk IDs     : {len(set(chunk_ids))}")
print(f"Required fields OK   : {len(incomplete) == 0}")
print(f"URLs present         : {len(missing_urls) == 0}")
print(f"Duplicate IDs        : {len(duplicate_ids) == 0}")

if incomplete:
    print("\nIncomplete chunks:")
    for chunk in incomplete:
        missing = required_fields - set(chunk.keys())
        print(f"  {chunk['chunk_id']} → {sorted(missing)}")

if missing_urls:
    print("\nMissing URLs:")
    for chunk_id in missing_urls:
        print(f"  - {chunk_id}")

if duplicate_ids:
    print("\nDuplicate chunk IDs:")
    for chunk_id in duplicate_ids:
        print(f"  - {chunk_id}")

metadata_ok = (
    len(all_chunks) == 53
    and len(set(chunk_ids)) == 53
    and not incomplete
    and not missing_urls
    and not duplicate_ids
)

print("\n" + "=" * 70)
print(f"METADATA STATUS: {'PASS ✓' if metadata_ok else 'CHECK REQUIRED ✗'}")
print("=" * 70)

RAG V2 — FINAL METADATA VALIDATION

Total chunks          : 53
Unique chunk IDs     : 53
Required fields OK   : True
URLs present         : True
Duplicate IDs        : True

METADATA STATUS: PASS ✓


In [145]:
import faiss
import json
from pathlib import Path

# Make sure embeddings and all_chunks have the same order
assert len(embeddings) == len(all_chunks), (
    f"Mismatch: {len(embeddings)} embeddings vs {len(all_chunks)} chunks"
)

# Create exact cosine-similarity index
# Embeddings are already L2-normalized.
index = faiss.IndexFlatIP(embeddings.shape[1])

index.add(embeddings)

# Output directories
vector_store_dir = Path("../rag/v2/vector_store")
metadata_dir = Path("../rag/v2/metadata")

vector_store_dir.mkdir(parents=True, exist_ok=True)
metadata_dir.mkdir(parents=True, exist_ok=True)

index_path = vector_store_dir / "rag_v2.index"
metadata_path = metadata_dir / "rag_v2_metadata.json"

# Save FAISS index
faiss.write_index(index, str(index_path))

# Save metadata in EXACTLY the same order as FAISS vectors
metadata = []

for chunk in all_chunks:
    metadata.append({
        "chunk_id": chunk["chunk_id"],
        "source_id": chunk["source_id"],
        "title": chunk["title"],
        "organization": chunk["organization"],
        "url": chunk["url"],
        "finding": chunk["finding"],
        "section": chunk["section"],
        "text": chunk["text"]
    })

with open(metadata_path, "w", encoding="utf-8") as f:
    json.dump(metadata, f, indent=2, ensure_ascii=False)

print("=" * 70)
print("RAG V2 — FAISS INDEX BUILT SUCCESSFULLY")
print("=" * 70)

print(f"\nIndex type       : {type(index).__name__}")
print(f"Vector dimension : {index.d}")
print(f"Vectors stored   : {index.ntotal}")
print(f"Metadata entries : {len(metadata)}")

print(f"\nFAISS index      : {index_path}")
print(f"Metadata         : {metadata_path}")

print("\nVerification:")
print(f"Embeddings ↔ chunks : {len(embeddings) == len(all_chunks)}")
print(f"FAISS ↔ metadata    : {index.ntotal == len(metadata)}")

RAG V2 — FAISS INDEX BUILT SUCCESSFULLY

Index type       : IndexFlatIP
Vector dimension : 384
Vectors stored   : 53
Metadata entries : 53

FAISS index      : ..\rag\v2\vector_store\rag_v2.index
Metadata         : ..\rag\v2\metadata\rag_v2_metadata.json

Verification:
Embeddings ↔ chunks : True
FAISS ↔ metadata    : True


In [146]:
def retrieve_v2(query, top_k=5):
    """
    Retrieve the top-k most relevant chunks from the V2 FAISS index.
    """

    query_embedding = embedding_model.encode(
        [query],
        normalize_embeddings=True
    )

    query_embedding = np.asarray(
        query_embedding,
        dtype="float32"
    )

    scores, indices = index.search(query_embedding, top_k)

    results = []

    for score, idx in zip(scores[0], indices[0]):

        chunk = all_chunks[int(idx)]

        results.append({
            "rank": len(results) + 1,
            "score": float(score),
            "chunk_id": chunk["chunk_id"],
            "finding": chunk["finding"],
            "section": chunk["section"],
            "title": chunk["title"],
            "url": chunk["url"],
            "text": chunk["text"]
        })

    return results


print("RAG V2 retrieval function ready.")


RAG V2 retrieval function ready.


In [147]:
query = "What is the ChestX-ray14 dataset?"

results = retrieve_v2(query, top_k=5)

print("=" * 70)
print("RAG V2 RETRIEVAL TEST")
print("=" * 70)

print(f"\nQuery: {query}")

for result in results:
    print("\n" + "-" * 70)
    print(f"Rank      : {result['rank']}")
    print(f"Score     : {result['score']:.4f}")
    print(f"Chunk ID  : {result['chunk_id']}")
    print(f"Finding   : {result['finding']}")
    print(f"Section   : {result['section']}")
    print(f"Title     : {result['title']}")
    print(f"Text      : {result['text'][:500]}")

RAG V2 RETRIEVAL TEST

Query: What is the ChestX-ray14 dataset?

----------------------------------------------------------------------
Rank      : 1
Score     : 0.6057
Chunk ID  : nodule_001
Finding   : Nodule
Section   : Definition
Title     : Lung Nodules - Diagnosis and Treatment
Text      : A lung nodule, also called a pulmonary nodule, is a small (<3cm), growth in the lung. Lung nodules are common. They are found in up to half of adults who undergo chest x-rays or CT scans. Doctors most often find them on a chest x-ray or CT scan performed for another reason. You may have a single lung nodule or several. Small lung nodules are typically benign (not cancer). Some, however, may be early-stage lung cancer. Lung nodules are frequently caused by infection, scars, or lymph nodes, among 

----------------------------------------------------------------------
Rank      : 2
Score     : 0.3978
Chunk ID  : lung_consolidation_ncbi_001
Finding   : Consolidation
Section   : Definition
Title   

In [148]:
query = "What is a chest X-ray used for?"

results = retrieve_v2(query, top_k=5)

print("=" * 70)
print("RAG V2 RETRIEVAL TEST")
print("=" * 70)

print(f"\nQuery: {query}")

for result in results:
    print("\n" + "-" * 70)
    print(f"Rank      : {result['rank']}")
    print(f"Score     : {result['score']:.4f}")
    print(f"Chunk ID  : {result['chunk_id']}")
    print(f"Finding   : {result['finding']}")
    print(f"Section   : {result['section']}")
    print(f"Title     : {result['title']}")
    print(f"Text      : {result['text'][:500]}")

RAG V2 RETRIEVAL TEST

Query: What is a chest X-ray used for?

----------------------------------------------------------------------
Rank      : 1
Score     : 0.5964
Chunk ID  : nodule_002
Finding   : Nodule
Section   : Detection and Evaluation
Title     : Lung Nodules - Diagnosis and Treatment
Text      : Doctors most often find lung nodules on a chest x-ray or CT scan being performed for another reason. Your doctor may order one or more of the following tests to help establish the size, number, and composition of your nodule(s):
Chest x-ray
:
This exam produces pictures of the lungs with a very small dose of ionizing radiation.
Chest CT
:
CT can help your doctor see the size, shape, and location of the lung nodule. It also provides information on other nodule characteristics, like the presence o

----------------------------------------------------------------------
Rank      : 2
Score     : 0.5674
Chunk ID  : pulmonary_edema_medlineplus_001
Finding   : Edema
Section   : Definition


In [149]:
import json
from pathlib import Path

general_chunks = [
    {
        "chunk_id": "chest_xray_general_001",
        "source_id": "radiologyinfo_chest_xray",
        "title": "Chest X-ray",
        "organization": "RadiologyInfo / ACR / RSNA",
        "url": "https://www.radiologyinfo.org/en/info/chestrad",
        "finding": "General",
        "section": "What is a chest X-ray and common uses",
        "text": """
A chest X-ray uses a very small dose of ionizing radiation to produce pictures of the inside of the chest. It is used to evaluate the lungs, heart, and chest wall and may help diagnose symptoms such as breathing difficulties, persistent cough, chest pain or injury, and fever. It may also help diagnose or monitor conditions such as pneumonia, heart problems, emphysema, and lung cancer, and can show fluid or air collections around the lungs.

Chest X-ray is a fast and widely available examination, but it has limitations. Some conditions cannot be detected on a conventional chest X-ray, so a normal chest X-ray does not necessarily rule out every chest abnormality. Further imaging may be needed to clarify findings or detect abnormalities that are not visible on the X-ray.
""".strip()
    },

    {
        "chunk_id": "chest_xray_report_001",
        "source_id": "radiologyinfo_chest_xray_report",
        "title": "How to Read Your Chest X-ray Report",
        "organization": "RadiologyInfo / ACR / RSNA",
        "url": "https://www.radiologyinfo.org/en/info/article-chest-xray-report",
        "finding": "General",
        "section": "Radiology report structure",
        "text": """
A chest X-ray report commonly contains sections such as the type of exam, history or reason for the exam, comparison with prior studies, technique, findings, and impression. The Findings section describes what the radiologist sees in the examined areas and may state whether structures appear normal, abnormal, or potentially abnormal.

The Impression section summarizes the most important findings and may include possible causes and recommendations for follow-up. Further imaging, clinical information, laboratory results, or comparison with previous studies may sometimes be needed to clarify an abnormal or uncertain finding.
""".strip()
    },

    {
        "chunk_id": "chestxray14_dataset_001",
        "source_id": "chestxray14_dataset_context",
        "title": "ChestX-ray14 Dataset Context",
        "organization": "NCBI / PMC",
        "url": "https://pmc.ncbi.nlm.nih.gov/articles/PMC9131331/",
        "finding": "General",
        "section": "Dataset overview",
        "text": """
ChestX-ray14 is a large chest X-ray dataset containing 112,120 frontal-view chest X-ray images from 30,805 patients. It uses 14 thoracic pathology labels, including Atelectasis, Cardiomegaly, Effusion, Infiltration, Mass, Nodule, Pneumonia, Pneumothorax, Consolidation, Edema, Emphysema, Fibrosis, Pleural Thickening, and Hernia.

The disease labels were generated using natural language processing of associated radiological reports rather than direct manual image annotation for every image. The dataset is therefore used as a weakly supervised or automatically labeled resource, and its labels should be interpreted as dataset annotations rather than definitive clinical diagnoses.
""".strip()
    }
]

general_chunk_path = Path("../rag/v2/chunks/general_context_chunks.json")

general_chunk_path.write_text(
    json.dumps(general_chunks, indent=2, ensure_ascii=False),
    encoding="utf-8"
)

print("=" * 70)
print("RAG V2 — GENERAL CONTEXT CHUNKS CREATED")
print("=" * 70)

print(f"\nChunks created : {len(general_chunks)}")
print(f"Output file   : {general_chunk_path}")

for chunk in general_chunks:
    print(
        f"\n{chunk['chunk_id']}"
        f" | {chunk['section']}"
        f" | {len(chunk['text'])} characters"
    )

print("\nExisting 53 chunks were not modified.")
print("New corpus size: 56 chunks")

RAG V2 — GENERAL CONTEXT CHUNKS CREATED

Chunks created : 3
Output file   : ..\rag\v2\chunks\general_context_chunks.json

chest_xray_general_001 | What is a chest X-ray and common uses | 778 characters

chest_xray_report_001 | Radiology report structure | 629 characters

chestxray14_dataset_001 | Dataset overview | 684 characters

Existing 53 chunks were not modified.
New corpus size: 56 chunks


In [150]:
import json
from pathlib import Path

v2_chunks_dir = Path("../rag/v2/chunks")

all_chunks = []

for file in sorted(v2_chunks_dir.glob("*.json")):
    with open(file, "r", encoding="utf-8") as f:
        all_chunks.extend(json.load(f))

# Deterministic ordering
all_chunks = sorted(all_chunks, key=lambda x: x["chunk_id"])

texts = [chunk["text"] for chunk in all_chunks]

print("=" * 70)
print("RAG V2 — COMPLETE CORPUS RELOADED")
print("=" * 70)

print(f"\nTotal chunks : {len(all_chunks)}")
print(f"Total texts  : {len(texts)}")

print("\nLast 3 chunks:")
print("-" * 70)

for chunk in all_chunks[-3:]:
    print(f"ID      : {chunk['chunk_id']}")
    print(f"Finding : {chunk['finding']}")
    print(f"Section : {chunk['section']}")

RAG V2 — COMPLETE CORPUS RELOADED

Total chunks : 56
Total texts  : 56

Last 3 chunks:
----------------------------------------------------------------------
ID      : pulmonary_edema_medlineplus_008
Finding : Edema
Section : Prevention
ID      : pulmonary_edema_medlineplus_009
Finding : Edema
Section : Alternative Names
ID      : pulmonary_mass_001
Finding : Mass
Section : Definition


In [151]:
embeddings = embedding_model.encode(
    texts,
    batch_size=32,
    show_progress_bar=True,
    normalize_embeddings=True
)

embeddings = np.asarray(
    embeddings,
    dtype="float32"
)

print("=" * 70)
print("RAG V2 — 56 EMBEDDINGS GENERATED")
print("=" * 70)

print(f"\nEmbedding shape : {embeddings.shape}")
print(f"Data type       : {embeddings.dtype}")
print(
    f"Normalized      : "
    f"{np.allclose(np.linalg.norm(embeddings, axis=1), 1.0, atol=1e-5)}"
)

print(f"Expected        : (56, 384)")

Batches: 100%|██████████| 2/2 [00:00<00:00,  5.27it/s]

RAG V2 — 56 EMBEDDINGS GENERATED

Embedding shape : (56, 384)
Data type       : float32
Normalized      : True
Expected        : (56, 384)


In [152]:
import faiss
import json
from pathlib import Path

# Safety check: embeddings and chunks must match exactly
assert len(embeddings) == len(all_chunks), (
    f"Mismatch: {len(embeddings)} embeddings vs {len(all_chunks)} chunks"
)

# Create cosine-similarity index
# Embeddings are L2-normalized, so inner product = cosine similarity.
index = faiss.IndexFlatIP(embeddings.shape[1])
index.add(embeddings)

vector_store_dir = Path("../rag/v2/vector_store")
metadata_dir = Path("../rag/v2/metadata")

vector_store_dir.mkdir(parents=True, exist_ok=True)
metadata_dir.mkdir(parents=True, exist_ok=True)

index_path = vector_store_dir / "rag_v2.index"
metadata_path = metadata_dir / "rag_v2_metadata.json"

# Save FAISS index
faiss.write_index(index, str(index_path))

# Save metadata in EXACTLY the same order as the embeddings
metadata = []

for chunk in all_chunks:
    metadata.append({
        "chunk_id": chunk["chunk_id"],
        "source_id": chunk["source_id"],
        "title": chunk["title"],
        "organization": chunk["organization"],
        "url": chunk["url"],
        "finding": chunk["finding"],
        "section": chunk["section"],
        "text": chunk["text"]
    })

with open(metadata_path, "w", encoding="utf-8") as f:
    json.dump(metadata, f, indent=2, ensure_ascii=False)

print("=" * 70)
print("RAG V2 — FAISS INDEX REBUILT")
print("=" * 70)

print(f"\nIndex type       : {type(index).__name__}")
print(f"Vector dimension : {index.d}")
print(f"Vectors stored   : {index.ntotal}")
print(f"Metadata entries : {len(metadata)}")

print(f"\nFAISS index      : {index_path}")
print(f"Metadata         : {metadata_path}")

print("\nVerification:")
print(f"Embeddings ↔ chunks : {len(embeddings) == len(all_chunks)}")
print(f"FAISS ↔ metadata    : {index.ntotal == len(metadata)}")

RAG V2 — FAISS INDEX REBUILT

Index type       : IndexFlatIP
Vector dimension : 384
Vectors stored   : 56
Metadata entries : 56

FAISS index      : ..\rag\v2\vector_store\rag_v2.index
Metadata         : ..\rag\v2\metadata\rag_v2_metadata.json

Verification:
Embeddings ↔ chunks : True
FAISS ↔ metadata    : True


In [153]:
query = "What is the ChestX-ray14 dataset?"

results = retrieve_v2(query, top_k=5)

print("=" * 70)
print("RAG V2 RETRIEVAL TEST — CHESTX-RAY14")
print("=" * 70)

print(f"\nQuery: {query}")

for result in results:
    print("\n" + "-" * 70)
    print(f"Rank      : {result['rank']}")
    print(f"Score     : {result['score']:.4f}")
    print(f"Chunk ID  : {result['chunk_id']}")
    print(f"Finding   : {result['finding']}")
    print(f"Section   : {result['section']}")
    print(f"Title     : {result['title']}")
    print(f"Text      : {result['text'][:500]}")

RAG V2 RETRIEVAL TEST — CHESTX-RAY14

Query: What is the ChestX-ray14 dataset?

----------------------------------------------------------------------
Rank      : 1
Score     : 0.6669
Chunk ID  : chestxray14_dataset_001
Finding   : General
Section   : Dataset overview
Title     : ChestX-ray14 Dataset Context
Text      : ChestX-ray14 is a large chest X-ray dataset containing 112,120 frontal-view chest X-ray images from 30,805 patients. It uses 14 thoracic pathology labels, including Atelectasis, Cardiomegaly, Effusion, Infiltration, Mass, Nodule, Pneumonia, Pneumothorax, Consolidation, Edema, Emphysema, Fibrosis, Pleural Thickening, and Hernia.

The disease labels were generated using natural language processing of associated radiological reports rather than direct manual image annotation for every image. The da

----------------------------------------------------------------------
Rank      : 2
Score     : 0.6057
Chunk ID  : infiltration_001
Finding   : Infiltration
Section   : Infilt

In [154]:
query = "What is a chest X-ray used for?"

results = retrieve_v2(query, top_k=5)

print("=" * 70)
print("RAG V2 RETRIEVAL TEST — CHEST X-RAY USE")
print("=" * 70)

print(f"\nQuery: {query}")

for result in results:
    print("\n" + "-" * 70)
    print(f"Rank      : {result['rank']}")
    print(f"Score     : {result['score']:.4f}")
    print(f"Chunk ID  : {result['chunk_id']}")
    print(f"Finding   : {result['finding']}")
    print(f"Section   : {result['section']}")
    print(f"Title     : {result['title']}")
    print(f"Text      : {result['text'][:500]}")

RAG V2 RETRIEVAL TEST — CHEST X-RAY USE

Query: What is a chest X-ray used for?

----------------------------------------------------------------------
Rank      : 1
Score     : 0.7630
Chunk ID  : chest_xray_general_001
Finding   : General
Section   : What is a chest X-ray and common uses
Title     : Chest X-ray
Text      : A chest X-ray uses a very small dose of ionizing radiation to produce pictures of the inside of the chest. It is used to evaluate the lungs, heart, and chest wall and may help diagnose symptoms such as breathing difficulties, persistent cough, chest pain or injury, and fever. It may also help diagnose or monitor conditions such as pneumonia, heart problems, emphysema, and lung cancer, and can show fluid or air collections around the lungs.

Chest X-ray is a fast and widely available examination,

----------------------------------------------------------------------
Rank      : 2
Score     : 0.6364
Chunk ID  : chest_xray_report_001
Finding   : General
Section   : Ra

In [155]:
query = "What does the Findings section of a chest X-ray report contain?"

results = retrieve_v2(query, top_k=5)

print("=" * 70)
print("RAG V2 RETRIEVAL TEST — REPORT FINDINGS")
print("=" * 70)

print(f"\nQuery: {query}")

for result in results:
    print("\n" + "-" * 70)
    print(f"Rank      : {result['rank']}")
    print(f"Score     : {result['score']:.4f}")
    print(f"Chunk ID  : {result['chunk_id']}")
    print(f"Finding   : {result['finding']}")
    print(f"Section   : {result['section']}")
    print(f"Title     : {result['title']}")
    print(f"Text      : {result['text'][:500]}")

RAG V2 RETRIEVAL TEST — REPORT FINDINGS

Query: What does the Findings section of a chest X-ray report contain?

----------------------------------------------------------------------
Rank      : 1
Score     : 0.8260
Chunk ID  : chest_xray_report_001
Finding   : General
Section   : Radiology report structure
Title     : How to Read Your Chest X-ray Report
Text      : A chest X-ray report commonly contains sections such as the type of exam, history or reason for the exam, comparison with prior studies, technique, findings, and impression. The Findings section describes what the radiologist sees in the examined areas and may state whether structures appear normal, abnormal, or potentially abnormal.

The Impression section summarizes the most important findings and may include possible causes and recommendations for follow-up. Further imaging, clinical informat

----------------------------------------------------------------------
Rank      : 2
Score     : 0.6894
Chunk ID  : chest_xray_g

In [156]:
query = "What is pneumothorax and how does it appear on a chest X-ray?"

results = retrieve_v2(query, top_k=5)

print("=" * 70)
print("RAG V2 RETRIEVAL TEST — PNEUMOTHORAX")
print("=" * 70)

print(f"\nQuery: {query}")

for result in results:
    print("\n" + "-" * 70)
    print(f"Rank      : {result['rank']}")
    print(f"Score     : {result['score']:.4f}")
    print(f"Chunk ID  : {result['chunk_id']}")
    print(f"Finding   : {result['finding']}")
    print(f"Section   : {result['section']}")
    print(f"Title     : {result['title']}")
    print(f"Text      : {result['text'][:600]}")

RAG V2 RETRIEVAL TEST — PNEUMOTHORAX

Query: What is pneumothorax and how does it appear on a chest X-ray?

----------------------------------------------------------------------
Rank      : 1
Score     : 0.6937
Chunk ID  : pneumothorax_001
Finding   : Pneumothorax
Section   : Pneumothorax on Imaging
Title     : Diseases of the Pleura and Chest Wall
Text      : Pneumothorax on Imaging

Good quality upright radiographs are sensitive for identifying a pneumothorax. Air outside the pleura moves to a nondependent location in the apex, which separates the pleura from the chest wall. The intervening air is radiolucent, as is the aerated lung deep to the pleura, resulting in a visible pleural line. This line is usually visible in the lateral aspect of the apex or along the lateral aspect of the lung. Beyond the pleural line there should be no vessels or lung parenchymal markings visible. Pleural lines can be confused with skin folds. A true pneumothorax sho

----------------------------------

In [157]:
finding_query_templates = {
    "Atelectasis":
        "What is atelectasis and what are its relevant features on a chest X-ray?",

    "Cardiomegaly":
        "What is cardiomegaly and what are its relevant features on a chest X-ray?",

    "Effusion":
        "What is pleural effusion and what are its relevant features on a chest X-ray?",

    "Infiltration":
        "What does the Infiltration label mean in the ChestX-ray14 dataset, and how should it be interpreted?",

    "Mass":
        "What is a lung mass and what are its relevant imaging features?",

    "Nodule":
        "What is a lung nodule and how is it detected and evaluated on chest imaging?",

    "Pneumonia":
        "What is pneumonia and what are its relevant features on a chest X-ray?",

    "Pneumothorax":
        "What is pneumothorax and how does it appear on a chest X-ray?",

    "Consolidation":
        "What is lung consolidation and what does it indicate on a chest X-ray?",

    "Edema":
        "What is pulmonary edema and what are its relevant features on chest imaging?",

    "Emphysema":
        "What is emphysema and what are its relevant features on chest imaging?",

    "Fibrosis":
        "What is pulmonary fibrosis and what are its relevant imaging features?",

    "Pleural_Thickening":
        "What is pleural thickening and how does it appear on chest imaging?",

    "Hernia":
        "What is a diaphragmatic hernia and what are its relevant imaging features?"
}

print(f"Number of finding templates: {len(finding_query_templates)}")

for finding, query in finding_query_templates.items():
    print(f"{finding:20} -> {query}")

Number of finding templates: 14
Atelectasis          -> What is atelectasis and what are its relevant features on a chest X-ray?
Cardiomegaly         -> What is cardiomegaly and what are its relevant features on a chest X-ray?
Effusion             -> What is pleural effusion and what are its relevant features on a chest X-ray?
Infiltration         -> What does the Infiltration label mean in the ChestX-ray14 dataset, and how should it be interpreted?
Mass                 -> What is a lung mass and what are its relevant imaging features?
Nodule               -> What is a lung nodule and how is it detected and evaluated on chest imaging?
Pneumonia            -> What is pneumonia and what are its relevant features on a chest X-ray?
Pneumothorax         -> What is pneumothorax and how does it appear on a chest X-ray?
Consolidation        -> What is lung consolidation and what does it indicate on a chest X-ray?
Edema                -> What is pulmonary edema and what are its relevant feature

In [158]:
def retrieve_for_finding(finding, top_k=5):
    if finding not in finding_query_templates:
        raise ValueError(
            f"Unknown finding: {finding}\n"
            f"Expected one of: {list(finding_query_templates.keys())}"
        )

    query = finding_query_templates[finding]

    results = retrieve_v2(query, top_k=top_k)

    return {
        "finding": finding,
        "query": query,
        "results": results
    }


# Test the complete finding-aware retrieval flow
result = retrieve_for_finding("Pneumothorax", top_k=5)

print("=" * 70)
print("FINDING-AWARE RAG RETRIEVAL")
print("=" * 70)

print(f"\nFinding : {result['finding']}")
print(f"Query   : {result['query']}")

for item in result["results"]:
    print("\n" + "-" * 70)
    print(f"Rank      : {item['rank']}")
    print(f"Score     : {item['score']:.4f}")
    print(f"Chunk ID  : {item['chunk_id']}")
    print(f"Finding   : {item['finding']}")
    print(f"Section   : {item['section']}")
    print(f"Title     : {item['title']}")

FINDING-AWARE RAG RETRIEVAL

Finding : Pneumothorax
Query   : What is pneumothorax and how does it appear on a chest X-ray?

----------------------------------------------------------------------
Rank      : 1
Score     : 0.6937
Chunk ID  : pneumothorax_001
Finding   : Pneumothorax
Section   : Pneumothorax on Imaging
Title     : Diseases of the Pleura and Chest Wall

----------------------------------------------------------------------
Rank      : 2
Score     : 0.5980
Chunk ID  : chest_xray_general_001
Finding   : General
Section   : What is a chest X-ray and common uses
Title     : Chest X-ray

----------------------------------------------------------------------
Rank      : 3
Score     : 0.5955
Chunk ID  : chest_xray_report_001
Finding   : General
Section   : Radiology report structure
Title     : How to Read Your Chest X-ray Report

----------------------------------------------------------------------
Rank      : 4
Score     : 0.5892
Chunk ID  : pleural_thickening_001
Finding   :

In [159]:
def classify_retrieval_results(results, target_finding):
    classified = []

    for result in results:
        if result["finding"] == target_finding:
            evidence_type = "primary"
        elif result["finding"] == "General":
            evidence_type = "general"
        else:
            evidence_type = "secondary"

        item = result.copy()
        item["evidence_type"] = evidence_type
        classified.append(item)

    return classified


# Test classification
result = retrieve_for_finding("Pneumothorax", top_k=5)

classified_results = classify_retrieval_results(
    result["results"],
    result["finding"]
)

print("=" * 70)
print("METADATA-AWARE EVIDENCE CLASSIFICATION")
print("=" * 70)

for item in classified_results:
    print(
        f"Rank {item['rank']} | "
        f"Score {item['score']:.4f} | "
        f"{item['evidence_type'].upper():9} | "
        f"{item['finding']:20} | "
        f"{item['chunk_id']}"
    )

METADATA-AWARE EVIDENCE CLASSIFICATION
Rank 1 | Score 0.6937 | PRIMARY   | Pneumothorax         | pneumothorax_001
Rank 2 | Score 0.5980 | GENERAL   | General              | chest_xray_general_001
Rank 3 | Score 0.5955 | GENERAL   | General              | chest_xray_report_001
Rank 4 | Score 0.5892 | SECONDARY | Pleural_Thickening   | pleural_thickening_001
Rank 5 | Score 0.5632 | SECONDARY | Consolidation        | lung_consolidation_ncbi_001


In [160]:
def build_evidence_package(finding, top_k=5):
    retrieval = retrieve_for_finding(finding, top_k=top_k)

    classified = classify_retrieval_results(
        retrieval["results"],
        finding
    )

    package = {
        "finding": finding,
        "query": retrieval["query"],
        "primary_evidence": [],
        "general_evidence": [],
        "secondary_evidence": []
    }

    for item in classified:
        evidence = {
            "chunk_id": item["chunk_id"],
            "score": item["score"],
            "title": item["title"],
            "section": item["section"],
            "text": item["text"],
            "url": item["url"]
        }

        if item["evidence_type"] == "primary":
            package["primary_evidence"].append(evidence)

        elif item["evidence_type"] == "general":
            package["general_evidence"].append(evidence)

        else:
            package["secondary_evidence"].append(evidence)

    return package


# Test
evidence_package = build_evidence_package(
    "Pneumothorax",
    top_k=5
)

print("=" * 70)
print("RAG EVIDENCE PACKAGE")
print("=" * 70)

print(f"\nFinding : {evidence_package['finding']}")
print(f"Query   : {evidence_package['query']}")

print("\nPrimary evidence :", len(evidence_package["primary_evidence"]))
print("General evidence :", len(evidence_package["general_evidence"]))
print("Secondary evidence:", len(evidence_package["secondary_evidence"]))

for category in [
    "primary_evidence",
    "general_evidence",
    "secondary_evidence"
]:
    print(f"\n--- {category.upper()} ---")

    for item in evidence_package[category]:
        print(
            f"{item['chunk_id']} | "
            f"score={item['score']:.4f} | "
            f"{item['title']}"
        )

RAG EVIDENCE PACKAGE

Finding : Pneumothorax
Query   : What is pneumothorax and how does it appear on a chest X-ray?

Primary evidence : 1
General evidence : 2
Secondary evidence: 2

--- PRIMARY_EVIDENCE ---
pneumothorax_001 | score=0.6937 | Diseases of the Pleura and Chest Wall

--- GENERAL_EVIDENCE ---
chest_xray_general_001 | score=0.5980 | Chest X-ray
chest_xray_report_001 | score=0.5955 | How to Read Your Chest X-ray Report

--- SECONDARY_EVIDENCE ---
pleural_thickening_001 | score=0.5892 | Diseases of the Pleura and Chest Wall
lung_consolidation_ncbi_001 | score=0.5632 | Lung Consolidation


In [161]:
def assess_evidence_sufficiency(evidence_package):
    primary = evidence_package["primary_evidence"]
    general = evidence_package["general_evidence"]

    if not primary:
        if general:
            status = "LIMITED"
            reason = "No finding-specific evidence was retrieved; only general context is available."
        else:
            status = "INSUFFICIENT"
            reason = "No primary or general evidence was retrieved."
    
    else:
        best_primary_score = max(
            item["score"] for item in primary
        )

        if best_primary_score >= 0.60:
            status = "SUFFICIENT"
            reason = "Finding-specific evidence was retrieved with strong semantic relevance."
        else:
            status = "LIMITED"
            reason = "Finding-specific evidence was retrieved, but semantic relevance is relatively weak."

    return {
        "status": status,
        "reason": reason
    }


# Test
sufficiency = assess_evidence_sufficiency(evidence_package)

print("=" * 70)
print("EVIDENCE SUFFICIENCY")
print("=" * 70)

print(f"\nStatus : {sufficiency['status']}")
print(f"Reason : {sufficiency['reason']}")

EVIDENCE SUFFICIENCY

Status : SUFFICIENT
Reason : Finding-specific evidence was retrieved with strong semantic relevance.


In [162]:
def build_rag_context(finding, top_k=5):
    evidence_package = build_evidence_package(
        finding,
        top_k=top_k
    )

    sufficiency = assess_evidence_sufficiency(
        evidence_package
    )

    context = {
        "finding": finding,
        "query": evidence_package["query"],
        "evidence_status": sufficiency["status"],
        "evidence_reason": sufficiency["reason"],
        "primary_evidence": evidence_package["primary_evidence"],
        "general_evidence": evidence_package["general_evidence"],
        "secondary_evidence": evidence_package["secondary_evidence"]
    }

    return context


# Test the complete RAG context
rag_context = build_rag_context(
    "Pneumothorax",
    top_k=5
)

print("=" * 70)
print("COMPLETE RAG CONTEXT")
print("=" * 70)

print(f"\nFinding         : {rag_context['finding']}")
print(f"Query           : {rag_context['query']}")
print(f"Evidence status : {rag_context['evidence_status']}")
print(f"Reason          : {rag_context['evidence_reason']}")

print("\nPrimary evidence:")
for item in rag_context["primary_evidence"]:
    print(
        f"  - {item['chunk_id']} "
        f"(score={item['score']:.4f})"
    )

print("\nGeneral evidence:")
for item in rag_context["general_evidence"]:
    print(
        f"  - {item['chunk_id']} "
        f"(score={item['score']:.4f})"
    )

print("\nSecondary evidence:")
for item in rag_context["secondary_evidence"]:
    print(
        f"  - {item['chunk_id']} "
        f"(score={item['score']:.4f})"
    )

COMPLETE RAG CONTEXT

Finding         : Pneumothorax
Query           : What is pneumothorax and how does it appear on a chest X-ray?
Evidence status : SUFFICIENT
Reason          : Finding-specific evidence was retrieved with strong semantic relevance.

Primary evidence:
  - pneumothorax_001 (score=0.6937)

General evidence:
  - chest_xray_general_001 (score=0.5980)
  - chest_xray_report_001 (score=0.5955)

Secondary evidence:
  - pleural_thickening_001 (score=0.5892)
  - lung_consolidation_ncbi_001 (score=0.5632)


In [163]:
llm_response_schema = {
    "predictions": [
        {
            "finding": "Pneumothorax",
            "probability": 0.87
        }
    ],
    "visual_explanation": {
        "finding": "Pneumothorax",
        "description": "Model-attributed regions associated with the prediction.",
        "gradcam_note": "Grad-CAM highlights model-attributed regions and does not establish a diagnosis."
    },
    "medical_context": {
        "finding": "Pneumothorax",
        "summary": "Evidence-grounded medical context retrieved from the RAG corpus.",
        "evidence_status": "SUFFICIENT"
    },
    "interpretation": {
        "summary": "Evidence-grounded interpretation of the model prediction.",
        "confidence_note": "Model probability is not equivalent to clinical diagnostic certainty."
    },
    "limitations": [
        "The model prediction is not a definitive diagnosis.",
        "Grad-CAM is an attribution method and does not prove the presence or location of disease.",
        "ChestX-ray14 labels are dataset labels and may not correspond directly to definitive clinical diagnoses."
    ],
    "citations": [
        {
            "chunk_id": "pneumothorax_001",
            "title": "Diseases of the Pleura and Chest Wall",
            "url": "https://www.ncbi.nlm.nih.gov/books/NBK621648/"
        }
    ]
}

print("=" * 70)
print("LLM RESPONSE SCHEMA")
print("=" * 70)

for section in llm_response_schema:
    print(f"✓ {section}")

print("\nTotal top-level fields:", len(llm_response_schema))

LLM RESPONSE SCHEMA
✓ predictions
✓ visual_explanation
✓ medical_context
✓ interpretation
✓ limitations
✓ citations

Total top-level fields: 6


In [164]:
def build_llm_prompt(
    finding,
    probability,
    gradcam_description,
    rag_context
):
    primary_evidence = "\n\n".join(
        [
            f"[{item['chunk_id']}] "
            f"{item['title']} — {item['section']}\n"
            f"{item['text']}"
            for item in rag_context["primary_evidence"]
        ]
    )

    general_evidence = "\n\n".join(
        [
            f"[{item['chunk_id']}] "
            f"{item['title']} — {item['section']}\n"
            f"{item['text']}"
            for item in rag_context["general_evidence"]
        ]
    )

    prompt = f"""
You are a medical-information interpretation layer for a chest X-ray
AI research system.

Your task is to explain a CNN prediction using ONLY the supplied
model output, visual-attribution information, and retrieved evidence.

IMPORTANT RULES:
1. The CNN probability is not a diagnosis or clinical certainty.
2. Grad-CAM is a model-attribution method, not proof of a lesion,
   disease, or anatomical abnormality.
3. Do not invent medical facts that are not supported by the supplied
   evidence.
4. Do not invent citations or URLs.
5. Use only the supplied chunk IDs when creating citations.
6. Treat ChestX-ray14 labels as research dataset labels rather than
   definitive clinical diagnoses.
7. If evidence is insufficient, explicitly state that the available
   evidence is limited.
8. Do not convert the model prediction into a definitive diagnosis.
9. Return valid JSON matching the required response schema.

CNN PREDICTION
--------------
Finding: {finding}
Probability: {probability:.4f}

VISUAL ATTRIBUTION
------------------
{gradcam_description}

RAG EVIDENCE STATUS
-------------------
Status: {rag_context["evidence_status"]}
Reason: {rag_context["evidence_reason"]}

PRIMARY EVIDENCE
----------------
{primary_evidence if primary_evidence else "No primary evidence available."}

GENERAL EVIDENCE
----------------
{general_evidence if general_evidence else "No general evidence available."}

REQUIRED OUTPUT
---------------
Return these six top-level fields:

1. predictions
2. visual_explanation
3. medical_context
4. interpretation
5. limitations
6. citations

The citations must reference only the supplied chunk IDs.
"""

    return prompt


# Test prompt construction
test_prompt = build_llm_prompt(
    finding="Pneumothorax",
    probability=0.87,
    gradcam_description=(
        "Grad-CAM highlights regions that contributed to the "
        "model's Pneumothorax prediction."
    ),
    rag_context=rag_context
)

print("=" * 70)
print("GROUNDED LLM PROMPT")
print("=" * 70)

print(test_prompt)

GROUNDED LLM PROMPT

You are a medical-information interpretation layer for a chest X-ray
AI research system.

Your task is to explain a CNN prediction using ONLY the supplied
model output, visual-attribution information, and retrieved evidence.

IMPORTANT RULES:
1. The CNN probability is not a diagnosis or clinical certainty.
2. Grad-CAM is a model-attribution method, not proof of a lesion,
   disease, or anatomical abnormality.
3. Do not invent medical facts that are not supported by the supplied
   evidence.
4. Do not invent citations or URLs.
5. Use only the supplied chunk IDs when creating citations.
6. Treat ChestX-ray14 labels as research dataset labels rather than
   definitive clinical diagnoses.
7. If evidence is insufficient, explicitly state that the available
   evidence is limited.
8. Do not convert the model prediction into a definitive diagnosis.
9. Return valid JSON matching the required response schema.

CNN PREDICTION
--------------
Finding: Pneumothorax
Probability:

In [165]:
def build_llm_input(
    finding,
    probability,
    gradcam_description,
    rag_context
):
    return {
        "model_prediction": {
            "finding": finding,
            "probability": float(probability)
        },
        "visual_attribution": {
            "description": gradcam_description
        },
        "rag_context": {
            "query": rag_context["query"],
            "evidence_status": rag_context["evidence_status"],
            "evidence_reason": rag_context["evidence_reason"],
            "primary_evidence": rag_context["primary_evidence"],
            "general_evidence": rag_context["general_evidence"]
        },
        "instructions": {
            "use_only_supplied_evidence": True,
            "do_not_invent_citations": True,
            "probability_is_not_diagnosis": True,
            "gradcam_is_not_diagnostic_proof": True,
            "chestxray14_labels_are_dataset_labels": True
        }
    }


# Test
llm_input = build_llm_input(
    finding="Pneumothorax",
    probability=0.87,
    gradcam_description=(
        "Grad-CAM highlights regions that contributed to "
        "the model's Pneumothorax prediction."
    ),
    rag_context=rag_context
)

print("=" * 70)
print("LLM INPUT PAYLOAD")
print("=" * 70)

print("\nModel prediction:")
print(llm_input["model_prediction"])

print("\nVisual attribution:")
print(llm_input["visual_attribution"])

print("\nRAG status:")
print(llm_input["rag_context"]["evidence_status"])

print("\nPrimary evidence:")
for item in llm_input["rag_context"]["primary_evidence"]:
    print(
        f"  - {item['chunk_id']} | "
        f"{item['title']} | "
        f"score={item['score']:.4f}"
    )

print("\nGeneral evidence:")
for item in llm_input["rag_context"]["general_evidence"]:
    print(
        f"  - {item['chunk_id']} | "
        f"{item['title']} | "
        f"score={item['score']:.4f}"
    )

print("\nInstructions:")
for key, value in llm_input["instructions"].items():
    print(f"  - {key}: {value}")

LLM INPUT PAYLOAD

Model prediction:
{'finding': 'Pneumothorax', 'probability': 0.87}

Visual attribution:
{'description': "Grad-CAM highlights regions that contributed to the model's Pneumothorax prediction."}

RAG status:
SUFFICIENT

Primary evidence:
  - pneumothorax_001 | Diseases of the Pleura and Chest Wall | score=0.6937

General evidence:
  - chest_xray_general_001 | Chest X-ray | score=0.5980
  - chest_xray_report_001 | How to Read Your Chest X-ray Report | score=0.5955

Instructions:
  - use_only_supplied_evidence: True
  - do_not_invent_citations: True
  - probability_is_not_diagnosis: True
  - gradcam_is_not_diagnostic_proof: True
  - chestxray14_labels_are_dataset_labels: True


In [166]:
def validate_llm_response(response, rag_context):
    required_fields = {
        "predictions",
        "visual_explanation",
        "medical_context",
        "interpretation",
        "limitations",
        "citations"
    }

    errors = []

    # ---------------------------------------------------------
    # 1. Check top-level structure
    # ---------------------------------------------------------
    if not isinstance(response, dict):
        return {
            "valid": False,
            "errors": ["LLM response is not a dictionary."]
        }

    missing_fields = required_fields - set(response.keys())

    if missing_fields:
        errors.append(
            f"Missing required fields: {sorted(missing_fields)}"
        )

    # ---------------------------------------------------------
    # 2. Check citation structure
    # ---------------------------------------------------------
    citations = response.get("citations", [])

    if not isinstance(citations, list):
        errors.append("citations must be a list.")
        citations = []

    # ---------------------------------------------------------
    # 3. Collect chunk IDs actually supplied to the LLM
    # ---------------------------------------------------------
    available_chunk_ids = set()

    for category in [
        "primary_evidence",
        "general_evidence"
    ]:
        for item in rag_context.get(category, []):
            available_chunk_ids.add(item["chunk_id"])

    # ---------------------------------------------------------
    # 4. Validate every citation
    # ---------------------------------------------------------
    for citation in citations:
        if not isinstance(citation, dict):
            errors.append("Each citation must be a dictionary.")
            continue

        chunk_id = citation.get("chunk_id")

        if not chunk_id:
            errors.append(
                "A citation is missing its chunk_id."
            )

        elif chunk_id not in available_chunk_ids:
            errors.append(
                f"Unsupported citation chunk_id: {chunk_id}"
            )

    # ---------------------------------------------------------
    # 5. Final result
    # ---------------------------------------------------------
    return {
        "valid": len(errors) == 0,
        "errors": errors
    }


# -------------------------------------------------------------
# Test with a valid example
# -------------------------------------------------------------
valid_test_response = {
    "predictions": [
        {
            "finding": "Pneumothorax",
            "probability": 0.87
        }
    ],
    "visual_explanation": {
        "finding": "Pneumothorax",
        "description": "Model-attributed regions associated with the prediction.",
        "gradcam_note": "Grad-CAM does not establish a diagnosis."
    },
    "medical_context": {
        "finding": "Pneumothorax",
        "summary": "Evidence-grounded medical context.",
        "evidence_status": "SUFFICIENT"
    },
    "interpretation": {
        "summary": "The model predicts Pneumothorax with the supplied probability.",
        "confidence_note": "Probability is not diagnostic certainty."
    },
    "limitations": [
        "Model prediction is not a definitive diagnosis."
    ],
    "citations": [
        {
            "chunk_id": "pneumothorax_001",
            "title": "Diseases of the Pleura and Chest Wall"
        }
    ]
}


validation = validate_llm_response(
    valid_test_response,
    rag_context
)

print("=" * 70)
print("LLM RESPONSE VALIDATION")
print("=" * 70)

print(f"\nValid : {validation['valid']}")

if validation["errors"]:
    print("\nErrors:")
    for error in validation["errors"]:
        print(f"  - {error}")
else:
    print("No validation errors.")

LLM RESPONSE VALIDATION

Valid : True
No validation errors.


In [167]:
invalid_test_response = valid_test_response.copy()

invalid_test_response["citations"] = [
    {
        "chunk_id": "fake_pneumothorax_source_999",
        "title": "Invented Medical Source"
    }
]

validation = validate_llm_response(
    invalid_test_response,
    rag_context
)

print("=" * 70)
print("LLM RESPONSE VALIDATION — INVALID CITATION TEST")
print("=" * 70)

print(f"\nValid : {validation['valid']}")

if validation["errors"]:
    print("\nDetected errors:")
    for error in validation["errors"]:
        print(f"  ✗ {error}")
else:
    print("\nNo validation errors.")

LLM RESPONSE VALIDATION — INVALID CITATION TEST

Valid : False

Detected errors:
  ✗ Unsupported citation chunk_id: fake_pneumothorax_source_999


In [168]:
def validate_prediction_consistency(
    response,
    expected_finding,
    expected_probability,
    tolerance=1e-6
):
    errors = []

    predictions = response.get("predictions")

    if not isinstance(predictions, list):
        return {
            "valid": False,
            "errors": ["predictions must be a list."]
        }

    matching_predictions = [
        prediction
        for prediction in predictions
        if isinstance(prediction, dict)
        and prediction.get("finding") == expected_finding
    ]

    if not matching_predictions:
        errors.append(
            f"Expected prediction for '{expected_finding}' "
            "was not found."
        )
    else:
        returned_probability = matching_predictions[0].get(
            "probability"
        )

        if not isinstance(returned_probability, (int, float)):
            errors.append(
                "Returned probability is not numeric."
            )
        elif abs(returned_probability - expected_probability) > tolerance:
            errors.append(
                f"Probability mismatch: "
                f"expected {expected_probability:.6f}, "
                f"received {returned_probability:.6f}"
            )

    return {
        "valid": len(errors) == 0,
        "errors": errors
    }


# Test with the correct CNN prediction
prediction_validation = validate_prediction_consistency(
    valid_test_response,
    expected_finding="Pneumothorax",
    expected_probability=0.87
)

print("=" * 70)
print("CNN → LLM PREDICTION CONSISTENCY")
print("=" * 70)

print(f"\nValid : {prediction_validation['valid']}")

if prediction_validation["errors"]:
    print("\nErrors:")
    for error in prediction_validation["errors"]:
        print(f"  ✗ {error}")
else:
    print("CNN prediction and LLM response are consistent.")

CNN → LLM PREDICTION CONSISTENCY

Valid : True
CNN prediction and LLM response are consistent.


In [169]:
def validate_llm_response_full(
    response,
    rag_context,
    expected_finding,
    expected_probability
):
    structure_validation = validate_llm_response(
        response,
        rag_context
    )

    prediction_validation = validate_prediction_consistency(
        response,
        expected_finding=expected_finding,
        expected_probability=expected_probability
    )

    errors = (
        structure_validation["errors"]
        + prediction_validation["errors"]
    )

    return {
        "valid": len(errors) == 0,
        "errors": errors
    }


# Test the complete validation gate
full_validation = validate_llm_response_full(
    response=valid_test_response,
    rag_context=rag_context,
    expected_finding="Pneumothorax",
    expected_probability=0.87
)

print("=" * 70)
print("FULL LLM VALIDATION GATE")
print("=" * 70)

print(f"\nValid : {full_validation['valid']}")

if full_validation["errors"]:
    print("\nErrors:")
    for error in full_validation["errors"]:
        print(f"  ✗ {error}")
else:
    print("✓ Structure validation passed")
    print("✓ Citation grounding passed")
    print("✓ CNN prediction consistency passed")
    print("\nLLM response accepted.")

FULL LLM VALIDATION GATE

Valid : True
✓ Structure validation passed
✓ Citation grounding passed
✓ CNN prediction consistency passed

LLM response accepted.


In [170]:
import os
import importlib.util

print("=" * 70)
print("LLM ENVIRONMENT CHECK")
print("=" * 70)

packages = {
    "openai": "openai",
    "anthropic": "anthropic",
    "google.generativeai": "google.generativeai",
    "ollama": "ollama"
}

print("\nInstalled packages:")

for name, module in packages.items():
    installed = importlib.util.find_spec(module) is not None
    print(f"{name:25} : {'YES' if installed else 'NO'}")

print("\nAPI key availability:")

keys = {
    "OPENAI_API_KEY": os.getenv("OPENAI_API_KEY"),
    "ANTHROPIC_API_KEY": os.getenv("ANTHROPIC_API_KEY"),
    "GOOGLE_API_KEY": os.getenv("GOOGLE_API_KEY")
}

for name, value in keys.items():
    print(f"{name:25} : {'SET' if value else 'NOT SET'}")

print("\nOllama:")
print("OLLAMA_HOST              :", os.getenv("OLLAMA_HOST", "default/local"))

LLM ENVIRONMENT CHECK

Installed packages:
openai                    : NO
anthropic                 : NO
google.generativeai       : NO
ollama                    : NO

API key availability:
OPENAI_API_KEY            : NOT SET
ANTHROPIC_API_KEY         : NOT SET
GOOGLE_API_KEY            : NOT SET

Ollama:
OLLAMA_HOST              : default/local
